# Formula 1 Podium Prediction — Milestone 1

**Goal.** Predict whether a driver finishes on the podium (`positionOrder <= 3`) using only information
available **after qualifying and before the race starts**.

**Unit of analysis:** one driver entering one race → one row per `(raceId, driverId)`.

| # | Section | What it contains |
|---|---------|------------------|
| 0 | Setup | Imports, data path, reproducibility, raw-file fingerprints |
| 1 | Audit & raw EDA | Sentinels, primary/foreign keys, exploratory plots **before** cleaning |
| 2 | Cleaning | Sentinel handling, semantic typing, duplicate rule, EDA **after** cleaning |
| 3 | Data-engineering questions | Q1 front-row conversion, Q2 home advantage, Q3 mechanical retirements |
| 4 | Modelling table & features | Leakage-safe features, justification, split, preprocessing-order experiments |
| 5 | Models | Baseline, statistical models, shallow NN, comparison table |
| 6 | Experiments & XAI | Feature-group ablation, permutation importance, SHAP (global + local), LIME |
| 7 | Inference | Prediction function on a complete and a partially-missing record |

**Rule used throughout:** the raw CSVs are never modified. Everything is done on in-memory copies, and
Section 7 re-checks the file fingerprints to prove it.

## 0. Setup

In [ ]:
import os, glob, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 110

SEED = 42
np.random.seed(SEED)

### 0.1 Data location
The notebook looks for the dataset in this order, so the same file runs on Kaggle and locally without edits:
1. the `F1_DATA_DIR` environment variable, if set;
2. any Kaggle input folder containing `circuits.csv`;
3. a local `data/` folder next to the notebook.

In [ ]:
def find_data_dir():
    candidates = []
    if os.environ.get("F1_DATA_DIR"):
        candidates.append(os.environ["F1_DATA_DIR"])
    candidates += sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/circuits.csv", recursive=True))
    candidates += ["data", "../data"]
    for c in candidates:
        if os.path.exists(os.path.join(c, "circuits.csv")):
            return c
    raise FileNotFoundError("Dataset not found. Set F1_DATA_DIR or put the CSVs in ./data")

DATA_DIR = find_data_dir()
TABLES = ["circuits", "constructors", "constructor_results", "constructor_standings", "drivers",
          "driver_standings", "races", "results", "qualifying", "lap_times", "pit_stops",
          "sprint_results", "seasons", "status"]
print("Data directory:", DATA_DIR)
missing = [t for t in TABLES if not os.path.exists(os.path.join(DATA_DIR, f"{t}.csv"))]
assert not missing, f"Missing tables: {missing}"
print(f"All {len(TABLES)} tables present.")

### 0.2 Raw-file fingerprints
We store an MD5 hash of every raw CSV now. At the end of the notebook we hash them again: identical hashes
prove the raw files stayed immutable and all cleaning happened on copies.

In [ ]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW_HASHES = {t: file_md5(os.path.join(DATA_DIR, f"{t}.csv")) for t in TABLES}
pd.Series(RAW_HASHES, name="md5").to_frame()

## 1. Data audit and exploratory analysis (before cleaning)

### 1.1 Loading the raw tables
**Choice:** we load every column as **text** (`dtype=str`, `keep_default_na=False`).

**Why:** pandas' default reader silently turns some strings into `NaN` and guesses types. For an audit we want
to see the data exactly as stored, so we can count placeholder values like `\N` ourselves instead of letting
the reader hide them. Typing happens deliberately in Section 2.

In [ ]:
RAW = {t: pd.read_csv(os.path.join(DATA_DIR, f"{t}.csv"), dtype=str, keep_default_na=False) for t in TABLES}

inventory = pd.DataFrame({
    "rows": {t: len(df) for t, df in RAW.items()},
    "columns": {t: df.shape[1] for t, df in RAW.items()},
    "memory_MB": {t: round(df.memory_usage(deep=True).sum() / 1e6, 2) for t, df in RAW.items()},
}).sort_values("rows", ascending=False)
inventory

### 1.2 Sentinel ("fake missing") values
A **sentinel** is a placeholder string that means "missing" but is not an actual empty value.
We scan every column of every table for a list of common sentinels rather than assuming only `\N` exists,
so the conclusion comes from the data and not from manual inspection.

In [ ]:
SENTINELS = ["\\N", "", "NULL", "null", "NaN", "nan", "None", "NA", "N/A", "-", "?"]

def sentinel_profile(tables):
    rows = []
    for t, df in tables.items():
        for c in df.columns:
            counts = df[c].dropna().astype(str).str.strip().value_counts()   # count each distinct value once
            for s in SENTINELS:
                n = int(counts.get(s, 0))
                if n:
                    rows.append({"table": t, "column": c, "sentinel": "empty string" if s == "" else s, "count": n,
                                 "pct_of_rows": round(100 * n / len(df), 2)})
    out = pd.DataFrame(rows, columns=["table", "column", "sentinel", "count", "pct_of_rows"])
    return out.sort_values(["table", "pct_of_rows"], ascending=[True, False])

sent_raw = sentinel_profile(RAW)
print("Distinct sentinel strings found:", sent_raw["sentinel"].unique().tolist())
sent_raw.reset_index(drop=True)

**Reading the table.** Two sentinels exist, not one:
- the literal string `\N` (the documented Ergast placeholder), in 30 columns;
- the **empty string** `''`, in 68 qualifying `q2`/`q3` values. These would be easy to miss because `\N` is the
  "official" placeholder.

None of the other candidates (`NULL`, `NaN`, `-`, …) occur. So Section 2 replaces exactly these two strings with
real missing values.

The amount of missingness is very uneven, which the plot below makes visible:

In [ ]:
plot_df = (sent_raw.assign(label=sent_raw["table"] + "." + sent_raw["column"])
           .pivot_table(index="label", columns="sentinel", values="pct_of_rows", aggfunc="sum", fill_value=0))
plot_df = plot_df.loc[plot_df.sum(axis=1).sort_values().index]
fig, ax = plt.subplots(figsize=(9, 0.3 * len(plot_df) + 1))
plot_df.plot.barh(stacked=True, ax=ax, width=0.8,
                  color=[{"\\N": "#c0392b", "empty string": "#f39c12"}.get(c, "#7f8c8d") for c in plot_df.columns])
for y, v in enumerate(plot_df.sum(axis=1)):
    ax.text(v + 1, y, f"{v:.1f}%", va="center", fontsize=8)
ax.set_xlabel("% of rows holding a sentinel"); ax.set_ylabel("")
ax.set_title("Where the raw data uses placeholder ('sentinel') values")
ax.set_xlim(0, 110); ax.legend(title="sentinel", loc="lower right")
plt.tight_layout(); plt.show()

**What this tells us, and why it matters for the task:**
- `results.position`, `time`, `milliseconds` and the fastest-lap columns are `\N` for drivers who did not
  finish or were not timed. These are **post-race** fields, so they will never be model features anyway.
  The target uses `positionOrder`, which has **no** sentinels, so every result row has a usable label.
- `qualifying.q2` / `q3` are mostly `\N` because of the knockout format: only the faster drivers take part
  in Q2 and Q3. That missingness is **informative** (it means "eliminated earlier"), not random.
- `races` session dates/times (`fp1_date`, `quali_date`, …) exist only for recent seasons; we do not need them.
- `drivers.number` / `code` are missing for older drivers; they are identifiers, not features.

### 1.3 Semantic type check
Which columns *should* be numbers or dates? For each column we measure the share of non-sentinel values that
parse as a number, a date, or a lap-time string (`m:ss.sss`). This decides the typing rules in Section 2.

In [ ]:
def parse_share(s, kind):
    s = s[(s != "\\N") & (s != "")]
    if len(s) == 0:
        return np.nan
    if kind == "numeric":
        return pd.to_numeric(s, errors="coerce").notna().mean()
    if kind == "date":
        return pd.to_datetime(s, format="%Y-%m-%d", errors="coerce").notna().mean()
    if kind == "laptime":
        return s.str.fullmatch(r"(\d+:)?\d{1,2}:\d{2}(\.\d+)?|\d+\.\d+").mean()

type_rows = []
for t, df in RAW.items():
    for c in df.columns:
        num, dat, lap = (parse_share(df[c], k) for k in ("numeric", "date", "laptime"))
        type_rows.append({"table": t, "column": c, "numeric": num, "date": dat, "lap_time_text": lap})
types = pd.DataFrame(type_rows)

def suggested(r):
    if r["date"] == 1: return "date"
    if r["numeric"] == 1: return "numeric"
    if r["lap_time_text"] and r["lap_time_text"] > 0.95: return "duration → seconds"
    if r["numeric"] and r["numeric"] > 0.5: return "mostly numeric (inspect)"
    return "text / id"
types["suggested_type"] = types.apply(suggested, axis=1)
types[types["suggested_type"] != "text / id"].round(3).reset_index(drop=True)

In [ ]:
# Columns that look numeric but are not fully parseable: show what the odd values are.
odd = types[types["suggested_type"] == "mostly numeric (inspect)"]
for _, r in odd.iterrows():
    s = RAW[r["table"]][r["column"]]
    bad = s[pd.to_numeric(s, errors="coerce").isna() & (s != "\\N")]
    print(f"{r['table']}.{r['column']}: non-numeric values -> {bad.value_counts().head(8).to_dict()}")

**Typing decisions this supports (applied in Section 2):**
- IDs, positions, grid, laps, points → numeric.
- `races.date`, `drivers.dob` → real dates.
- `qualifying.q1/q2/q3`, `results.fastestLapTime`, `lap_times.time` → seconds (float).
- `positionText` holds codes such as `R` (retired), `D` (disqualified), `W` (withdrew), `F` (failed to qualify).
  It is a **post-race** field, kept only for auditing.
- `results.time` mixes absolute race time for the winner with gaps (`+5.2`) for others, so it is not a
  clean duration. It is post-race and is not used.

### 1.4 Primary-key checks
A primary key must be **unique** and **non-null**. We test the declared key of every table, plus the
*natural* key `(raceId, driverId)` of the per-driver tables, because that is our modelling grain.

In [ ]:
PRIMARY_KEYS = {
    "circuits": ["circuitId"], "constructors": ["constructorId"], "drivers": ["driverId"],
    "races": ["raceId"], "seasons": ["year"], "status": ["statusId"],
    "results": ["resultId"], "qualifying": ["qualifyId"], "sprint_results": ["resultId"],
    "driver_standings": ["driverStandingsId"], "constructor_standings": ["constructorStandingsId"],
    "constructor_results": ["constructorResultsId"],
    "lap_times": ["raceId", "driverId", "lap"], "pit_stops": ["raceId", "driverId", "stop"],
}
NATURAL_KEYS = {
    "results": ["raceId", "driverId"], "qualifying": ["raceId", "driverId"],
    "sprint_results": ["raceId", "driverId"], "driver_standings": ["raceId", "driverId"],
    "constructor_standings": ["raceId", "constructorId"], "races": ["year", "round"],
}

def key_report(tables, keys, kind):
    out = []
    for t, k in keys.items():
        df = tables[t]
        # counts real NaN *and* text sentinels, so it works on raw (text) and clean (typed) tables alike
        nulls = int((df[k].isna() | df[k].isin(["\\N", ""])).any(axis=1).sum())
        dups = int(df.duplicated(k, keep=False).sum())
        out.append({"table": t, "key": ", ".join(k), "kind": kind, "rows": len(df),
                    "null_keys": nulls, "rows_in_duplicate_keys": dups,
                    "valid": nulls == 0 and dups == 0})
    return pd.DataFrame(out)

pk_raw = pd.concat([key_report(RAW, PRIMARY_KEYS, "primary"),
                    key_report(RAW, NATURAL_KEYS, "natural")], ignore_index=True)
pk_raw

**Result.** Every declared primary key is unique and non-null. The only failure is the natural key
`results(raceId, driverId)`: some drivers appear **more than once in the same race**. That breaks our
"one row = one prediction" rule, so we look at those rows closely.

In [ ]:
res_raw = RAW["results"].merge(RAW["races"][["raceId", "year", "name"]], on="raceId")
dup_mask = res_raw.duplicated(["raceId", "driverId"], keep=False)
dups = res_raw[dup_mask].merge(RAW["status"], on="statusId")
print(f"{dups[['raceId','driverId']].drop_duplicates().shape[0]} driver-race pairs, "
      f"{dup_mask.sum()} rows involved, across {dups['raceId'].nunique()} races")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
dups.drop_duplicates(["raceId", "driverId"])["year"].astype(int).value_counts().sort_index() \
    .plot.bar(ax=ax[0], color="#c0392b")
ax[0].set_title("Duplicated (raceId, driverId) pairs per season")
ax[0].set_xlabel("Season"); ax[0].set_ylabel("Pairs")
dups["status"].value_counts().head(10).sort_values().plot.barh(ax=ax[1], color="#7f8c8d")
ax[1].set_title("Status of the duplicated rows"); ax[1].set_xlabel("Rows")
plt.tight_layout(); plt.show()

dups.sort_values(["raceId", "driverId", "resultId"])[
    ["year", "name", "driverId", "constructorId", "grid", "positionOrder", "points", "laps", "status"]
].head(12)

**Interpretation.** 85 driver-race pairs (176 rows) are duplicated, almost all between 1950 and 1958, with a few
up to 1964. They are **shared drives**: a driver handed his car over or took over a teammate's car mid-race,
so he gets one result row per car. The single 1978 case is a driver listed both as *Did not prequalify* and
*Did not qualify*. **None of them fall in 2003–2024**, but the rule still has to be defined and applied,
because the Q1–Q3 analyses and the cleaned tables cover all years. The rule is chosen in Section 2.

### 1.5 Foreign-key reconciliation
Every foreign key value must exist in the table it points to. We count **orphans**: child rows whose key
has no match in the parent table.

In [ ]:
FOREIGN_KEYS = [
    ("races", "circuitId", "circuits", "circuitId"), ("races", "year", "seasons", "year"),
    ("results", "raceId", "races", "raceId"), ("results", "driverId", "drivers", "driverId"),
    ("results", "constructorId", "constructors", "constructorId"), ("results", "statusId", "status", "statusId"),
    ("qualifying", "raceId", "races", "raceId"), ("qualifying", "driverId", "drivers", "driverId"),
    ("qualifying", "constructorId", "constructors", "constructorId"),
    ("driver_standings", "raceId", "races", "raceId"), ("driver_standings", "driverId", "drivers", "driverId"),
    ("constructor_standings", "raceId", "races", "raceId"),
    ("constructor_standings", "constructorId", "constructors", "constructorId"),
    ("constructor_results", "raceId", "races", "raceId"),
    ("constructor_results", "constructorId", "constructors", "constructorId"),
    ("lap_times", "raceId", "races", "raceId"), ("lap_times", "driverId", "drivers", "driverId"),
    ("pit_stops", "raceId", "races", "raceId"), ("pit_stops", "driverId", "drivers", "driverId"),
    ("sprint_results", "raceId", "races", "raceId"), ("sprint_results", "driverId", "drivers", "driverId"),
    ("sprint_results", "constructorId", "constructors", "constructorId"),
    ("sprint_results", "statusId", "status", "statusId"),
]

def fk_report(tables):
    out = []
    for child, ccol, parent, pcol in FOREIGN_KEYS:
        child_vals, parent_vals = tables[child][ccol], tables[parent][pcol]
        numeric = pd.api.types.is_numeric_dtype
        if not (numeric(child_vals) and numeric(parent_vals)):   # raw text: compare as strings
            child_vals, parent_vals = child_vals.astype(str), parent_vals.astype(str)
        orphans = ~child_vals.isin(set(parent_vals))
        out.append({"child": f"{child}.{ccol}", "parent": f"{parent}.{pcol}", "child_rows": len(child_vals),
                    "orphans": int(orphans.sum()), "reconciles": not orphans.any()})
    return pd.DataFrame(out)

fk_raw = fk_report(RAW)
print("All foreign keys reconcile:", fk_raw["reconciles"].all())
fk_raw

A second, softer consistency check: a driver's constructor in **qualifying** should normally match his
constructor in **results** for the same race. Mismatches would mean the joins could attach the wrong team.

In [ ]:
qr = RAW["qualifying"][["raceId", "driverId", "constructorId"]].merge(
    RAW["results"][["raceId", "driverId", "constructorId"]].drop_duplicates(["raceId", "driverId"]),
    on=["raceId", "driverId"], how="outer", suffixes=("_quali", "_result"), indicator=True)
print(qr["_merge"].value_counts().to_dict())
both = qr[qr["_merge"] == "both"]
mismatch = both[both["constructorId_quali"] != both["constructorId_result"]]
print("Constructor mismatches between qualifying and results:", len(mismatch))
mismatch.merge(RAW["races"][["raceId", "year", "name"]], on="raceId")[
    ["year", "name", "driverId", "constructorId_quali", "constructorId_result"]]

- Every qualifying row has a matching result row (`left_only = 0`). The ~16k results without qualifying are the
  pre-1994 and 1996–2002 entries discussed in Section 1.6.
- All 10 constructor mismatches are the same case: in late 2015 the team appears as *Marussia* (id 206) in
  qualifying and *Manor Marussia* (id 209) in results. It is one team under two IDs, not a data error that
  would mislead the model. **Rule:** whenever we need the team for a race, we take `constructorId` from
  `results`, the official race entry, and use qualifying only for qualifying positions and times.

### 1.6 Exploratory plots of the raw data
These plots are about **coverage and shape**: what the data contains over time, and what that implies
for the modelling window. They are redrawn after cleaning in Section 2.8.

In [ ]:
races_raw = RAW["races"].assign(year=lambda d: d["year"].astype(int))
res_y = RAW["results"].merge(races_raw[["raceId", "year"]], on="raceId")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
races_raw.groupby("year").size().plot(ax=ax[0], color="#2c3e50")
ax[0].set_title("Races per season"); ax[0].set_xlabel("Season"); ax[0].set_ylabel("Races")
res_y.groupby(["year", "raceId"]).size().groupby("year").mean().plot(ax=ax[1], color="#2c3e50")
ax[1].set_title("Average drivers entered per race"); ax[1].set_xlabel("Season"); ax[1].set_ylabel("Drivers")
plt.tight_layout(); plt.show()

The field size changes a lot (over 30 entries in the 1950s–80s, 20 today). Since every race has exactly
3 podium places, the **podium base rate per row changes with the era**. That matters for the target:

In [ ]:
res_y["podium"] = res_y["positionOrder"].astype(int) <= 3
base = res_y.groupby("year")["podium"].mean()
fig, ax = plt.subplots(figsize=(13, 3.5))
base.plot(ax=ax, color="#c0392b")
ax.set_title("Share of entries that finish on the podium (target base rate) by season")
ax.set_ylabel("Podium rate"); ax.set_xlabel("Season")
ax.axhline(3 / 20, ls="--", color="grey"); ax.text(1951, 3 / 20 + 0.012, "3 of 20 drivers = 0.15", color="grey")
plt.tight_layout(); plt.show()
print(f"Overall podium rate: {res_y['podium'].mean():.3f}")

**The key coverage plot.** Our prediction cutoff is *after qualifying*, so qualifying data is central.
For each season we measure what share of race entries have a matching qualifying row:

In [ ]:
q_keys = RAW["qualifying"][["raceId", "driverId"]].assign(has_quali=True)
cov = res_y.merge(q_keys, on=["raceId", "driverId"], how="left")
cov["has_quali"] = cov["has_quali"].fillna(False).astype(bool)
q_cov = cov.groupby("year")["has_quali"].mean()

fig, ax = plt.subplots(figsize=(13, 3.8))
ax.bar(q_cov.index, q_cov.values, color=np.where(q_cov.index >= 2003, "#27ae60", "#bdc3c7"))
ax.set_title("Share of race entries with a qualifying record, by season")
ax.set_ylabel("Coverage"); ax.set_xlabel("Season"); ax.set_ylim(0, 1.05)
ax.axvline(2002.5, ls="--", color="black")
ax.text(2003, 1.01, "complete from 2003", fontsize=9)
ax.set_xlim(1984.5, 2025)
ax.text(1985.2, 0.5, "no qualifying table\nbefore 1994 (0%)", fontsize=9, color="grey")
plt.tight_layout(); plt.show()
print(q_cov.loc[1994:2004].round(3).to_string())

**Finding that drives a design decision.** Qualifying data starts in 1994, and coverage collapses between
1996 and 2002 (as low as ~6%), then is **complete every season from 2003**. If we trained on all years, the
model would mostly learn from rows where qualifying is missing for reasons unrelated to the driver, which
is coverage drift, not signal. We therefore restrict the **modelling table to 2003–2024**. The
data-engineering questions in Section 3 use whatever span each question requires.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
modern = res_y[res_y["year"] >= 2003].copy()
modern["grid"] = modern["grid"].astype(int)
modern["positionOrder"] = modern["positionOrder"].astype(int)
ax[0].hist(modern.loc[modern["grid"] > 0, "grid"], bins=np.arange(0.5, 27.5), color="#2980b9", edgecolor="white")
ax[0].set_title("Grid position, 2003–2024 (raw)"); ax[0].set_xlabel("Grid slot"); ax[0].set_ylabel("Entries")
ax[0].bar([0], [(modern["grid"] == 0).sum()], width=0.95, color="#c0392b", label="grid = 0 (pit-lane start / did not start)")
ax[0].legend(loc="lower left", fontsize=8)
pod_by_grid = modern[modern["grid"].between(1, 20)].groupby("grid")["podium"].mean()
ax[1].bar(pod_by_grid.index, pod_by_grid.values, color="#8e44ad")
ax[1].set_title("Podium rate by starting grid slot, 2003–2024")
ax[1].set_xlabel("Grid slot"); ax[1].set_ylabel("Podium rate"); ax[1].set_xticks(range(1, 21))
plt.tight_layout(); plt.show()
print(f"grid = 0 entries in 2003–2024: {(modern['grid'] == 0).sum()} of {len(modern)}")

Two takeaways:
1. **Grid position is a very strong signal.** Podium rate falls steeply from P1 to about P6 and is near zero
   beyond P10. Any model must beat a simple "starts in the top 3" rule to be worth its complexity. In
   Section 5 that rule is kept as a reference point next to the official baseline model (logistic regression;
   the brief requires the baseline to be a statistical ML model or a shallow neural network).
2. **`grid = 0` is a sentinel in disguise.** Grid slot 0 does not exist; it marks a pit-lane start or a car
   that never took the start. It must be handled explicitly in cleaning rather than read as "better than P1".

In [ ]:
status_counts = res_y.merge(RAW["status"], on="statusId")
status_counts = status_counts[status_counts["year"] >= 2003]["status"].value_counts()
fig, ax = plt.subplots(figsize=(10, 5))
status_counts.head(20).sort_values().plot.barh(ax=ax, color="#16a085"); ax.set_ylabel("")
ax.set_title("Most common race outcomes (status), 2003–2024"); ax.set_xlabel("Entries")
plt.tight_layout(); plt.show()
print(f"{RAW['status'].shape[0]} distinct status codes in total; "
      f"{(status_counts > 0).sum()} occur since 2003.")

The `status` table mixes finishers (`Finished`, `+1 Lap`…), crashes (`Accident`, `Collision`),
technical failures (`Engine`, `Gearbox`, `Hydraulics`…) and regulatory outcomes (`Disqualified`,
`Did not qualify`). It is **post-race** information, so it is never a feature for the current race. It is
needed for Q3 (mechanical retirements) and for a *past-races* reliability feature in Section 4.

### 1.7 Audit summary
| Check | Result | Action in Section 2 |
|---|---|---|
| Sentinels | `\N` in 30 columns, empty string in qualifying `q2`/`q3` | Replace both with real missing values |
| Qualifying vs results team | 10 constructor mismatches | Take team from `results` |
| Types | Everything loaded as text; dates and `m:ss.sss` durations present | Cast numerics, parse dates, convert durations to seconds |
| Primary keys | All unique and non-null | — |
| Natural key `(raceId, driverId)` | Fails in `results` (shared drives) | Apply a documented keep rule |
| Foreign keys | All reconcile | Re-check after cleaning |
| `grid = 0` | Hidden sentinel | Treat as pit-lane / unknown start |
| Qualifying coverage | Complete only from 2003 | Model on 2003–2024 |

## 2. Data cleaning

Every step below works on `CLEAN`, a set of **copies** of the raw tables. `RAW` is never modified.
Each step ends with a check or a plot that shows exactly what it changed.

### 2.1–2.2 Working copies and sentinel replacement
**Rule:** a cell whose *entire* value is `\N` or an empty string becomes a real missing value (`NaN`).
Text that merely *contains* those characters is untouched.

**Why NaN and not 0 or a default value?** Missing is not zero: a missing `q3` means "did not reach Q3", not
"0-second lap". Filling in values is a **modelling** decision, made in Section 4 inside the training pipeline
with training data only. Filling now, with statistics from all years, would leak test-period information
into training.

In [ ]:
CLEAN = {t: df.copy().replace({"\\N": np.nan, "": np.nan}) for t, df in RAW.items()}

# Proof: for every column, the new missing count must equal the sentinel count found in Section 1.2.
expected = sent_raw.groupby(["table", "column"])["count"].sum()
observed = pd.Series({(t, c): int(CLEAN[t][c].isna().sum()) for t, c in expected.index})
check = pd.DataFrame({"sentinels_in_raw": expected, "NaN_after_replace": observed})
check["match"] = check["sentinels_in_raw"] == check["NaN_after_replace"]
assert check["match"].all(), "Sentinel replacement changed a different number of cells than expected"
untouched = sum(CLEAN[t][c].isna().sum() for t in TABLES for c in CLEAN[t].columns
                if (t, c) not in expected.index)
assert untouched == 0, "Columns without sentinels gained missing values"
print(f"{check['sentinels_in_raw'].sum():,} sentinel cells replaced in {len(check)} columns; "
      f"every count matches Section 1.2; no other column changed.")
print("Sentinel scan after replacement found:", len(sentinel_profile(CLEAN)), "sentinels")

### 2.3 Semantic typing
The type rules come from the parse-rate table built in Section 1.3, not from a hand-written list:
- a column that parsed as a number in **100%** of its non-missing values → numeric;
- a column that parsed as a `YYYY-MM-DD` date in 100% of values → date;
- lap-time style text (`1:26.572`, `16:44.718`, `26.898`) → a new `<column>_sec` column in seconds.
  The original text column is kept for traceability.

Everything else (names, references, URLs, `positionText` codes, the mixed `results.time` field) stays text.

In [ ]:
def to_seconds(s):
    """'1:26.572' -> 86.572, '1:02:03.5' -> 3723.5, '26.898' -> 26.898, missing stays missing."""
    txt = s.astype(object).where(s.notna(), None).astype(str)
    n_colons = txt.str.count(":")
    parts = txt.str.split(":", expand=True).apply(pd.to_numeric, errors="coerce")
    for k in range(3):
        if k not in parts:
            parts[k] = np.nan
    out = pd.Series(np.nan, index=s.index)
    out[n_colons == 0] = parts[0]
    out[n_colons == 1] = parts[0] * 60 + parts[1]
    out[n_colons == 2] = parts[0] * 3600 + parts[1] * 60 + parts[2]
    return out.where(s.notna())

DURATION_COLS = {"qualifying": ["q1", "q2", "q3"], "results": ["fastestLapTime"],
                 "sprint_results": ["fastestLapTime"], "lap_times": ["time"], "pit_stops": ["duration"]}
NUMERIC_COLS = types.loc[types["suggested_type"] == "numeric"].groupby("table")["column"].apply(list).to_dict()
DATE_COLS = types.loc[types["suggested_type"] == "date"].groupby("table")["column"].apply(list).to_dict()

dtype_before = pd.Series({t: CLEAN[t].dtypes.astype(str).value_counts().to_dict() for t in TABLES})
typing_log = []
for t in TABLES:
    df = CLEAN[t]
    for c in NUMERIC_COLS.get(t, []):
        before_na = df[c].isna().sum()
        df[c] = pd.to_numeric(df[c])                        # raises if anything fails: nothing silent
        df[c] = df[c].astype("Int64") if (df[c].dropna() % 1 == 0).all() else df[c]
        typing_log.append({"table": t, "column": c, "to": str(df[c].dtype),
                           "new_missing": int(df[c].isna().sum() - before_na)})
    for c in DATE_COLS.get(t, []):
        before_na = df[c].isna().sum()
        df[c] = pd.to_datetime(df[c], format="%Y-%m-%d")
        typing_log.append({"table": t, "column": c, "to": "datetime",
                           "new_missing": int(df[c].isna().sum() - before_na)})
    for c in DURATION_COLS.get(t, []):
        df[c + "_sec"] = to_seconds(df[c])
        typing_log.append({"table": t, "column": c + "_sec", "to": "float seconds",
                           "new_missing": int(df[c + "_sec"].isna().sum() - df[c].isna().sum())})

typing_log = pd.DataFrame(typing_log)
assert typing_log["new_missing"].sum() == 0, "Some values failed to convert"
print(f"{len(typing_log)} columns typed; values lost in conversion: {typing_log['new_missing'].sum()}")
typing_log.groupby("to").size().rename("columns").to_frame()

In [ ]:
# Spot-check: the conversions on a few real values
pd.DataFrame({
    "q1 (text)": CLEAN["qualifying"]["q1"].head(3).values,
    "q1_sec": CLEAN["qualifying"]["q1_sec"].head(3).values,
    "pit duration (text)": CLEAN["pit_stops"]["duration"].sort_values(key=lambda s: s.str.len()).tail(3).values,
    "duration_sec": CLEAN["pit_stops"].loc[CLEAN["pit_stops"]["duration"].sort_values(
        key=lambda s: s.str.len()).tail(3).index, "duration_sec"].values,
})

In [ ]:
type_counts = pd.DataFrame({
    "before (raw)": pd.concat([RAW[t].dtypes for t in TABLES]).astype(str).value_counts(),
    "after (clean)": pd.concat([CLEAN[t].dtypes for t in TABLES]).astype(str)
                       .replace({"datetime64[ns]": "datetime", "datetime64[us]": "datetime"}).value_counts(),
}).fillna(0).astype(int)
ax = type_counts.T.plot.barh(stacked=True, figsize=(10, 2.8), colormap="tab20c")
ax.set_xlabel("Number of columns across all 14 tables")
ax.set_title("Column types before and after semantic typing")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()
type_counts

Before typing, every column was text (`object`). After typing, IDs, positions and counts are integers,
points and coordinates are floats, dates are real dates, and durations exist as seconds. The assertion
above guarantees **no value was lost** during conversion.

### 2.4 Qualifying-time plausibility
A lap time can be valid as text and still be impossible. We compare each driver's Q1 time with the fastest
qualifying time **of the same race**. Comparing within a race keeps the check fair across circuits of very
different lengths.

In [ ]:
qc = CLEAN["qualifying"].merge(CLEAN["races"][["raceId", "year"]], on="raceId")
race_best = qc.groupby("raceId")[["q1_sec", "q2_sec", "q3_sec"]].transform("min").min(axis=1)
qc["q1_ratio"] = qc["q1_sec"] / race_best

RATIO_LIMIT = 3.0
outliers = qc[qc["q1_ratio"] > RATIO_LIMIT]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].hist(qc["q1_ratio"].dropna().clip(upper=1.6), bins=60, color="#2980b9")
ax[0].set_title("Q1 time ÷ fastest time in the same race\n(values above 1.6 shown at 1.6)")
ax[0].set_xlabel("Ratio to race-best time"); ax[0].set_ylabel("Drivers")
ax[1].scatter(qc["year"], qc["q1_ratio"], s=6, alpha=0.4, color="#7f8c8d")
ax[1].scatter(outliers["year"], outliers["q1_ratio"], s=40, color="#c0392b", label=f"> {RATIO_LIMIT}× race best")
ax[1].axhline(RATIO_LIMIT, ls="--", color="#c0392b")
ax[1].set_yscale("log"); ax[1].set_title("Every Q1 time vs race best (log scale)")
ax[1].set_xlabel("Season"); ax[1].set_ylabel("Ratio"); ax[1].legend()
plt.tight_layout(); plt.show()
print(qc["q1_ratio"].describe(percentiles=[0.5, 0.99, 0.999]).round(3).to_string())
outliers[["year", "raceId", "driverId", "q1", "q1_sec", "q1_ratio"]]

99.9% of Q1 times are within 1.3× of the race-best time. The next slowest after the outlier is ~1.46×, which
is plausible (wet session, a car problem). Exactly **one** time is impossible: `16:42.640` (1995), ten times
the pole time, almost certainly a typing error. Our threshold of 3× sits in the empty gap between the
plausible values and that error, so its exact value does not change the result. We set that time to missing.

In [ ]:
bad_idx = CLEAN["qualifying"].index[CLEAN["qualifying"]["qualifyId"].isin(outliers["qualifyId"])]
CLEAN["qualifying"].loc[bad_idx, ["q1", "q1_sec"]] = np.nan
print(f"Implausible Q1 times set to missing: {len(bad_idx)}")

**Are 2003–2005 times comparable with later years?** Qualifying formats changed: single-lap qualifying
(2003–2005) and the Q1/Q2/Q3 knockout from 2006. The plot shows two things:

In [ ]:
qc = CLEAN["qualifying"].merge(CLEAN["races"][["raceId", "year"]], on="raceId")
qc = qc[qc["year"] >= 2003]
qc["race_best"] = qc.groupby("raceId")[["q1_sec", "q2_sec", "q3_sec"]].transform("min").min(axis=1)
qc["q1_gap_pct"] = 100 * (qc["q1_sec"] / qc["race_best"] - 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
qc.groupby("year")["q1_gap_pct"].median().plot.bar(ax=ax[0], color="#16a085")
ax[0].set_title("Median Q1 gap to race-best time (%)"); ax[0].set_xlabel("Season"); ax[0].set_ylabel("%")
qc.groupby("year")[["q2_sec", "q3_sec"]].apply(lambda d: d.notna().mean()).plot(ax=ax[1], marker="o")
ax[1].set_title("Share of drivers with a Q2 / Q3 time"); ax[1].set_xlabel("Season"); ax[1].set_ylabel("Share")
plt.tight_layout(); plt.show()

- **Relative pace is comparable across the whole window.** The median gap to the race-best time moves between
  about 1.3% and 3.5% from year to year, depending on how close the field was that season, and 2003–2005
  do not stand out from later years. So a *relative* measure (gap to pole in %) is a format-robust feature.
  Raw seconds are not comparable across circuits anyway.
- **Q2/Q3 do not exist before 2006** (2005's few Q2 values are the second session of that year's aggregate
  format). A feature such as "reached Q3" would be meaningless for 2003–2005. Section 4 therefore uses the
  qualifying **position** and the **gap to pole**, which exist in every season.

### 2.5 `grid = 0`
Grid slot 0 does not exist. It marks a driver with **no grid slot**: in 2003–2024 that is a pit-lane start
for most of them, and a driver who never started for a few. We replace it with missing and add a flag.

**We do not delete the non-starters.** The only way to tell them apart is the race status, which is post-race
information. Choosing rows with it would use the future to shape the dataset.

In [ ]:
res = CLEAN["results"].merge(CLEAN["races"][["raceId", "year"]], on="raceId", how="left")
res = res.merge(CLEAN["status"], on="statusId", how="left")
res["no_grid_slot"] = (res["grid"] == 0).astype(int)
res.loc[res["grid"] == 0, "grid"] = pd.NA

g0 = res[(res["no_grid_slot"] == 1) & (res["year"] >= 2003)]
print(f"2003–2024: {len(g0)} entries with no grid slot; {int((g0['laps'] > 0).sum())} of them raced "
      f"(pit-lane starts), {int((g0['laps'] == 0).sum())} did not; podiums among them: "
      f"{int((g0['positionOrder'] <= 3).sum())}")
fig, ax = plt.subplots(figsize=(13, 3.3))
(res[res["year"] >= 2003].groupby("year")["no_grid_slot"].sum()).plot.bar(ax=ax, color="#c0392b")
ax.set_title("Entries with no grid slot (grid = 0 in raw data), per season 2003–2024")
ax.set_xlabel("Season"); ax.set_ylabel("Entries")
plt.tight_layout(); plt.show()

- The flag is legal at prediction time: whether a car starts from the pit lane is decided before the race.
- Missing grid slots are **much more frequent since 2019** (5–18 per season, vs 0–4 before). Those seasons
  fall in our validation and test periods, so handling this properly matters more there than in training.

### 2.6 Duplicate driver entries (shared drives)
**The rule:**
1. **Features come from the car the driver started in.** That is the entry that existed before the race,
   which matches our pre-race prediction setting. Concretely: the row with a real grid slot. If several rows
   have one, or none does, we keep the first entry (lowest `resultId`).
2. **The target is a podium in *any* of his rows.** The question is "did this *driver* finish on the podium",
   and in a shared drive he can earn it in the second car.

**Rejected alternatives:** *"keep the best-finishing row"* picks the row by looking at the race result, which
is post-race information. *"Delete the duplicated drivers"* throws away real results instead of resolving them.

**Example: 1960 Argentine Grand Prix.** Stirling Moss started from pole in car #36, which broke after 40
laps. He then took over his teammate's car #38 and finished 3rd:

In [ ]:
names = CLEAN["drivers"][["driverId", "forename", "surname"]]
moss = names[(names["forename"] == "Stirling") & (names["surname"] == "Moss")]["driverId"].iloc[0]
races_c = CLEAN["races"]
argentina_1960 = races_c.loc[(races_c["year"] == 1960) & (races_c["name"] == "Argentine Grand Prix"), "raceId"].iloc[0]
show_cols = ["resultId", "driverId", "constructorId", "number", "grid", "laps", "positionOrder", "status"]
res[(res["raceId"] == argentina_1960) & (res["driverId"] == moss)][show_cols]

In [ ]:
n_before = len(res)
res["podium_row"] = (res["positionOrder"] <= 3).astype(int)
res["n_rows_for_driver"] = res.groupby(["raceId", "driverId"])["resultId"].transform("size")
res["podium"] = res.groupby(["raceId", "driverId"])["podium_row"].transform("max")
res["best_positionOrder"] = res.groupby(["raceId", "driverId"])["positionOrder"].transform("min")

# Rows with a grid slot first, then lowest resultId; keep the first row per (raceId, driverId)
res = (res.assign(_no_slot=res["grid"].isna())
          .sort_values(["raceId", "driverId", "_no_slot", "resultId"])
          .drop_duplicates(["raceId", "driverId"], keep="first")
          .drop(columns=["_no_slot", "podium_row"]))
res["shared_drive"] = (res["n_rows_for_driver"] > 1).astype(int)

assert not res.duplicated(["raceId", "driverId"]).any()
print(f"Rows: {n_before:,} → {len(res):,} (removed {n_before - len(res)}); "
      f"(raceId, driverId) is now unique.")
res[(res["raceId"] == argentina_1960) & (res["driverId"] == moss)][
    ["resultId", "grid", "laps", "positionOrder", "best_positionOrder", "podium", "shared_drive", "status"]]

Moss keeps one row: the pole-position car he started in (features), with `podium = 1` because he did finish
3rd (target). `best_positionOrder` keeps his best classified position, and `shared_drive` marks the merged
rows so they stay traceable.

**From here on, the target is the column `podium`**, defined once, here. None of the 85 merged pairs are in
the 2003–2024 modelling window, but the rule is applied to all years because Section 3 uses all years.

In [ ]:
CLEAN["results"] = res.drop(columns=["year", "status"])
print("Clean results table:", CLEAN["results"].shape)

### 2.7 Post-cleaning audit
Cleaning changed values, types and rows, so we **re-run the Section 1 checks** on the clean tables instead of
assuming our own code is correct. Typing in particular can silently break joins: an ID stored as text `"1"`
does not match the number `1`.

In [ ]:
pk_clean = pd.concat([key_report(CLEAN, PRIMARY_KEYS, "primary"),
                      key_report(CLEAN, NATURAL_KEYS, "natural")], ignore_index=True)
fk_clean = fk_report(CLEAN)
audit_compare = pd.DataFrame({
    "check": ["sentinel cells", "invalid primary keys", "invalid natural keys", "foreign keys with orphans"],
    "raw": [int(sent_raw["count"].sum()), int((~pk_raw.loc[lambda d: d["kind"] == "primary"]["valid"]).sum()),
            int((~pk_raw.loc[lambda d: d["kind"] == "natural"]["valid"]).sum()), int((~fk_raw["reconciles"]).sum())],
    "clean": [len(sentinel_profile(CLEAN)), int((~pk_clean.loc[lambda d: d["kind"] == "primary"]["valid"]).sum()),
              int((~pk_clean.loc[lambda d: d["kind"] == "natural"]["valid"]).sum()), int((~fk_clean["reconciles"]).sum())],
})
assert audit_compare["clean"].sum() == 0, "Post-cleaning audit failed"
print("All post-cleaning checks pass.")
audit_compare

### 2.8 Exploratory analysis after cleaning
The plots below show what cleaning changed and what it made possible.

**Missingness, before vs after.** Before cleaning, missing values were invisible to pandas (they were text).
After cleaning, the same cells are real `NaN` that pandas counts, plots and handles correctly:

In [ ]:
cols_of_interest = ["qualifying.q1", "qualifying.q2", "qualifying.q3", "results.position", "results.grid",
                    "results.fastestLapTime", "results.milliseconds", "drivers.code"]
miss = pd.DataFrame({
    "pandas sees as missing (raw)": [RAW[t][c].isna().mean() * 100 for t, c in (x.split(".") for x in cols_of_interest)],
    "pandas sees as missing (clean)": [CLEAN[t][c].isna().mean() * 100 for t, c in (x.split(".") for x in cols_of_interest)],
}, index=cols_of_interest)
ax = miss.plot.barh(figsize=(10, 4), color=["#bdc3c7", "#c0392b"])
ax.set_xlabel("% of rows"); ax.set_title("Missing values visible to pandas: raw (all 0%) vs clean")
plt.tight_layout(); plt.show()
miss.round(1)

`results.grid` now counts as missing for the "no grid slot" entries (Section 2.5), and `qualifying.q1` lost the
one impossible time (Section 2.4). Every other change is exactly the sentinel replacement.

**Qualifying times are now numbers**, so their distributions can be analysed. Raw seconds vary with circuit
length, which is why the gap to pole is the useful form:

In [ ]:
q_mod = CLEAN["qualifying"].merge(CLEAN["races"][["raceId", "year", "circuitId"]], on="raceId")
q_mod = q_mod[q_mod["year"] >= 2003].copy()
q_mod["pole_sec"] = q_mod.groupby("raceId")[["q1_sec", "q2_sec", "q3_sec"]].transform("min").min(axis=1)
q_mod["best_sec"] = q_mod[["q1_sec", "q2_sec", "q3_sec"]].min(axis=1)
q_mod["gap_to_pole_pct"] = 100 * (q_mod["best_sec"] / q_mod["pole_sec"] - 1)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].hist(q_mod["q1_sec"].dropna(), bins=60, color="#2980b9")
ax[0].set_title("Q1 lap time in seconds, 2003–2024"); ax[0].set_xlabel("Seconds"); ax[0].set_ylabel("Drivers")
ax[1].hist(q_mod["gap_to_pole_pct"].dropna().clip(upper=8), bins=60, color="#16a085")
ax[1].set_title("Best qualifying time: gap to pole (%)\n(values above 8% shown at 8%)")
ax[1].set_xlabel("% slower than pole"); ax[1].set_ylabel("Drivers")
plt.tight_layout(); plt.show()

Raw seconds spread from under a minute (short circuits such as Austria) to over 1:50 (long ones such as Spa),
so a 90-second lap means very different things at different tracks. The gap to pole puts every race on one
scale: most drivers are within 0–4% of pole.

**Grid vs qualifying position.** The brief asks us to distinguish the two. After cleaning we can measure
how often they differ:

In [ ]:
gq = CLEAN["results"].merge(CLEAN["qualifying"][["raceId", "driverId", "position"]]
                            .rename(columns={"position": "quali_pos"}), on=["raceId", "driverId"])
gq = gq.merge(CLEAN["races"][["raceId", "year"]], on="raceId")
gq = gq[gq["year"] >= 2003].copy()
gq["differs"] = (gq["grid"] != gq["quali_pos"]) | gq["grid"].isna()
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
both_known = gq.dropna(subset=["grid"])
ct = pd.crosstab(both_known["quali_pos"].clip(upper=20), both_known["grid"].clip(upper=20))
sns.heatmap(np.log1p(ct), ax=ax[0], cmap="mako_r", cbar_kws={"label": "log(1 + entries)"})
ax[0].set_title("Qualifying position vs grid slot, 2003–2024"); ax[0].set_xlabel("Grid slot")
ax[0].set_ylabel("Qualifying position")
gq.groupby("year")["differs"].mean().plot(ax=ax[1], marker="o", color="#8e44ad")
ax[1].set_title("Share of drivers whose grid ≠ qualifying position"); ax[1].set_ylabel("Share")
ax[1].set_xlabel("Season")
plt.tight_layout(); plt.show()
print(f"2003–2024: grid differs from qualifying position for {gq['differs'].mean():.1%} of entries")

Most entries sit on the diagonal (grid = qualifying position), but a sizeable share do not, mainly because
of grid penalties (engine or gearbox changes) that push drivers back, and pit-lane starts. Since about
2008 this affects 18–48% of entries per season.

**Data limitation:** in 2003 the share is ~0% and in 2007 ~6%. Real grid penalties existed then, so this
most likely reflects how the source recorded those seasons rather than real racing. We report it as a
known limitation and do not try to "fix" it.
Both values are
known before the race, so **both are kept** as separate candidate features: qualifying measures pace, and
grid measures where the race actually starts.

### 2.9 Cleaning summary
Every number below is computed by the code above, not typed by hand.

In [ ]:
cleaning_summary = pd.DataFrame([
    ["2.2 Sentinels → NaN", f"{int(sent_raw['count'].sum()):,} cells", "`\\N` and empty strings mean 'missing'; NaN keeps them visible without inventing values"],
    ["2.3 Semantic typing", f"{len(typing_log)} columns", "Numbers, dates and durations become usable; zero values lost"],
    ["2.4 Implausible Q1 time", f"{len(bad_idx)} value", f"Over {RATIO_LIMIT:.0f}× the race-best time; a data-entry error"],
    ["2.5 grid = 0 → NaN + flag", f"{int(res['no_grid_slot'].sum()):,} rows", "Slot 0 does not exist; flag keeps the pit-lane information"],
    ["2.6 Shared-drive duplicates", f"{n_before - len(res)} rows merged", "One row per driver per race; features from the starting car, podium from any car"],
    ["2.7 Post-cleaning audit", "all checks pass", "Keys and joins still valid after cleaning"],
], columns=["step", "affected", "reason"])
cleaning_summary

## 3. Data-engineering questions

Each question needs information spread over several tables, so we first build one joined table and then
answer each question from it. **Every answer comes with:** the result, a plot, and a note on **grain** (what one
row is), **missingness** (what is missing and how it was handled) and, where relevant, the **pre-race cutoff**.

### 3.0 The joined table (multi-hop joins)

| Table | Joined on | What it contributes |
|---|---|---|
| `results` (clean) | base table | one row per (raceId, driverId): grid, `podium` target, status, laps, constructor |
| `races` | `raceId` | season, round, circuit |
| `circuits` | `races.circuitId` (2 hops) | circuit name and **country** (Q1, Q2) |
| `qualifying` | `raceId`, `driverId` | **qualifying position**, distinct from grid (Q1) |
| `drivers` | `driverId` | driver name and **nationality** (Q2) |
| `constructors` | `constructorId` | team name (Q3) |
| `status` | `statusId` | why the race ended (Q3) |

All joins are *left* joins onto `results`, so no race entry is dropped or duplicated; an assertion checks that
(raceId, driverId) is still unique afterwards.

The same cell defines the shared helpers: `wilson_ci` (95% interval for a rate), `rate_table` (counts, rate and Wilson interval per group) and `savefig` (stores every figure in `figures/` for the report). It also records a fingerprint of `CLEAN`, so the gate in 3.4 can prove that Phase 3 never changed the cleaned tables.

In [ ]:
from scipy.stats import spearmanr, chi2_contingency
from statsmodels.stats.proportion import proportions_ztest

os.makedirs("figures", exist_ok=True)
# Phase 3 only reads CLEAN. This fingerprint lets the gate (3.4) prove it.
CLEAN_FINGERPRINT = {t: int(pd.util.hash_pandas_object(df).sum()) for t, df in CLEAN.items()}

races_c = CLEAN["races"].rename(columns={"name": "race_name"})
circuits_c = CLEAN["circuits"].rename(columns={"name": "circuit_name", "country": "circuit_country"})
drivers_c = CLEAN["drivers"].assign(driver=lambda d: d["forename"] + " " + d["surname"])
constructors_c = CLEAN["constructors"].rename(columns={"name": "constructor"})
quali_c = CLEAN["qualifying"][["raceId", "driverId", "position"]].rename(columns={"position": "quali_pos"})

entries = (CLEAN["results"]
           .merge(races_c[["raceId", "year", "round", "circuitId", "race_name"]], on="raceId", how="left")
           .merge(circuits_c[["circuitId", "circuit_name", "circuit_country"]], on="circuitId", how="left")
           .merge(quali_c, on=["raceId", "driverId"], how="left")
           .merge(drivers_c[["driverId", "driver", "nationality"]], on="driverId", how="left")
           .merge(constructors_c[["constructorId", "constructor"]], on="constructorId", how="left")
           .merge(CLEAN["status"], on="statusId", how="left"))

assert len(entries) == len(CLEAN["results"]), "a join added or removed rows"
assert not entries.duplicated(["raceId", "driverId"]).any()
assert entries[["year", "circuit_name", "driver", "constructor", "status"]].notna().all().all()
print(f"Joined table: {len(entries):,} rows (one per driver per race), {entries.shape[1]} columns, "
      f"{entries['year'].min()}–{entries['year'].max()}")


def wilson_ci(successes, n, z=1.96):
    """95% Wilson confidence interval for a proportion. It stays sensible for small samples and for
    rates near 0% or 100%, where the simple 'p ± 1.96·SE' interval breaks down."""
    successes, n = np.asarray(successes, float), np.asarray(n, float)
    p = np.where(n > 0, successes / np.maximum(n, 1), np.nan)
    denom = 1 + z ** 2 / n
    centre = (p + z ** 2 / (2 * n)) / denom
    half = z * np.sqrt(p * (1 - p) / n + z ** 2 / (4 * n ** 2)) / denom
    return centre - half, centre + half


def rate_table(df, by, flag, min_n=0):
    """Per group: rows (n), hits (sum of a 0/1 column), rate = hits / n, and its 95% Wilson interval."""
    out = df.groupby(by, observed=True)[flag].agg(n="size", hits="sum").reset_index()
    out["rate"] = out["hits"] / out["n"]
    out["ci_low"], out["ci_high"] = wilson_ci(out["hits"], out["n"])
    return out[out["n"] >= min_n].reset_index(drop=True)


def savefig(name):
    plt.savefig(f"figures/{name}.png", dpi=150, bbox_inches="tight")

**Result:** 26,668 rows, exactly as many as `CLEAN["results"]`, and every (raceId, driverId) pair still appears once.
So the six joins attached information without adding or losing a single race entry.

### 3.1 Q1: Which circuits convert a front-row start into a podium most often, and does that ranking change after 2014?

**Definitions (our choices):**
- **Front row** = starting position 1 or 2.
- **Conversion rate** of a circuit = front-row starters who finished on the podium ÷ all front-row starters.
- Measured **twice**, as the brief requires: by **grid slot** (where the car actually started) and by
  **qualifying position** (pace in qualifying). They differ when a front-row qualifier gets a grid penalty.
- **Periods:** 2003–2013 (pre-hybrid) vs 2014–2024 (hybrid era). Two equal 11-season windows, both with complete
  qualifying data, which the qualifying-vs-grid comparison needs (Section 1.6).
- **Minimum sample:** a circuit is ranked only with **≥ 10 front-row starts in a period** (5 races). With fewer, a rate
  of 100% or 0% is mostly luck. Every rate is shown with a 95% Wilson interval.

In [ ]:
MIN_FRONT_ROW_STARTS = 10
q1 = entries[entries["year"].between(2003, 2024)].copy()
q1["period"] = np.where(q1["year"] >= 2014, "2014–2024 (hybrid)", "2003–2013")
q1["front_row_grid"] = q1["grid"].isin([1, 2])
q1["front_row_quali"] = q1["quali_pos"].isin([1, 2])


def conversion(df, flag):
    g = (df[df[flag]].groupby(["circuit_name", "period"])
         .agg(starts=("podium", "size"), podiums=("podium", "sum")).reset_index())
    g["rate"] = g["podiums"] / g["starts"]
    g["ci_low"], g["ci_high"] = wilson_ci(g["podiums"], g["starts"])
    return g


conv_grid, conv_quali = conversion(q1, "front_row_grid"), conversion(q1, "front_row_quali")
eligible = conv_grid[conv_grid["starts"] >= MIN_FRONT_ROW_STARTS]
both_periods = eligible.groupby("circuit_name").filter(lambda d: d["period"].nunique() == 2)
wide = both_periods.pivot(index="circuit_name", columns="period", values="rate")
wide.columns = ["pre_2014", "hybrid"]
wide = wide.sort_values("hybrid")
print(f"Circuits with ≥ {MIN_FRONT_ROW_STARTS} front-row starts: "
      f"{eligible[eligible['period'] == '2003–2013']['circuit_name'].nunique()} in 2003–2013, "
      f"{eligible[eligible['period'] != '2003–2013']['circuit_name'].nunique()} in 2014–2024, "
      f"{len(wide)} in both")

**What this cell does:** it keeps 2003–2024, marks each entry as a front-row start by grid slot (`front_row_grid`) and
by qualifying position (`front_row_quali`), and counts starts and podiums per circuit and period with `conversion()`.
`eligible` keeps only the circuit-periods with at least 10 front-row starts.

**Output:** 21 circuits qualify in 2003–2013, 19 in 2014–2024, and 14 in both. Only those 14 can be compared across
the two periods.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.36 * len(wide) + 1.5))
y = np.arange(len(wide))
ax.hlines(y, wide["pre_2014"], wide["hybrid"], color="#bdc3c7", lw=2)
ax.scatter(wide["pre_2014"], y, color="#7f8c8d", s=45, label="2003–2013", zorder=3)
ax.scatter(wide["hybrid"], y, color="#c0392b", s=45, label="2014–2024 (hybrid)", zorder=3)
ax.set_yticks(y); ax.set_yticklabels(wide.index)
ax.set_xlabel("Share of front-row starters (grid 1–2) who finished on the podium")
ax.set_title("Q1: front-row → podium conversion by circuit, before vs during the hybrid era\n"
             f"(circuits with ≥ {MIN_FRONT_ROW_STARTS} front-row starts in both periods)")
ax.set_xlim(0, 1.05); ax.legend(loc="lower right")
plt.tight_layout(); savefig("q1_conversion_by_circuit"); plt.show()

**How to read the chart:** one row per circuit, sorted by its 2014–2024 rate (highest at the top). The grey dot is
2003–2013, the red dot is 2014–2024, and the line between them is the change. A red dot to the right of the grey one
means conversion rose at that circuit; where only one dot shows (Bahrain), both periods have the same rate. Most red
dots sit to the right: conversion rose at 11 of the 14 circuits. The chart has no intervals, so small gaps between
neighbouring circuits mean little (the tables below give the Wilson intervals).

In [ ]:
def ranked(period):
    d = eligible[eligible["period"] == period].sort_values(["rate", "starts"], ascending=False)
    d = d.assign(rank=np.arange(1, len(d) + 1))
    return d[["rank", "circuit_name", "starts", "podiums", "rate", "ci_low", "ci_high"]].round(2)

print("Top 5, 2003–2013:"); display_pre = ranked("2003–2013").head(5); print(display_pre.to_string(index=False))
print("\nTop 5, 2014–2024 (hybrid):"); display_post = ranked("2014–2024 (hybrid)").head(5)
print(display_post.to_string(index=False))

from scipy.stats import spearmanr
rho, p_rho = spearmanr(wide["pre_2014"], wide["hybrid"])
overall = (q1[q1["front_row_grid"]].groupby("period")["podium"].agg(["size", "sum", "mean"])
           .rename(columns={"size": "front-row starts", "sum": "podiums", "mean": "conversion"}))
print(f"\nRank correlation between the two periods (Spearman, {len(wide)} circuits): "
      f"rho = {rho:.2f} (p = {p_rho:.2f})")
print(f"Circuits whose conversion rose / fell / stayed equal: {int((wide['hybrid'] > wide['pre_2014']).sum())} / "
      f"{int((wide['hybrid'] < wide['pre_2014']).sum())} / {int((wide['hybrid'] == wide['pre_2014']).sum())}")

for per, d in eligible.groupby("period"):
    chi2, p_het, dof, _ = chi2_contingency(np.c_[d["podiums"], d["starts"] - d["podiums"]])
    print(f"{per}: could all circuits share one conversion rate? chi-square = {chi2:.1f}, df = {dof}, p = {p_het:.2f}")

# The chi-square p-value uses an approximation that needs expected counts of about 5 or more. Some of ours are smaller,
# so we also compute the p-value by simulation: assume one common rate, redraw every circuit's podiums 20,000 times
# with its real number of starts, and count how often chance alone gives a chi-square as large as the real one.
rng_mc = np.random.default_rng(SEED)
for per, d in eligible.groupby("period"):
    n = d["starts"].to_numpy(); k_obs = d["podiums"].to_numpy()
    _, _, _, expected = chi2_contingency(np.c_[k_obs, n - k_obs])
    k = rng_mc.binomial(n, k_obs.sum() / n.sum(), size=(20_000, len(n)))
    p_hat = k.sum(axis=1, keepdims=True) / n.sum()
    chi_sim = (((k - n * p_hat) ** 2) / (n * p_hat) + (((n - k) - n * (1 - p_hat)) ** 2) / (n * (1 - p_hat))).sum(axis=1)
    chi_obs = chi2_contingency(np.c_[k_obs, n - k_obs])[0]
    print(f"{per}: {int((expected < 5).sum())} of {expected.size} expected counts below 5; "
          f"simulation p = {(chi_sim >= chi_obs).mean():.2f}")

from scipy.stats import norm
k1, n1 = q1.loc[q1["front_row_grid"] & (q1["year"] < 2014), "podium"].agg(["sum", "size"])
k2, n2 = q1.loc[q1["front_row_grid"] & (q1["year"] >= 2014), "podium"].agg(["sum", "size"])
p_pool = (k1 + k2) / (n1 + n2)
z = (k2 / n2 - k1 / n1) / np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
print(f"Overall rise {k1/n1:.3f} → {k2/n2:.3f}: two-proportion z = {z:.2f}, p = {2 * (1 - norm.cdf(abs(z))):.3f}")
overall.round(3)

**How to read the tables:** `rate` = podiums ÷ starts, and `ci_low` / `ci_high` is the 95% Wilson interval, the range where the true rate plausibly lies. Magny-Cours's 92% comes with an interval of 65–99%, because it rests on only 12 starts.

**Reading the rank correlation:** 1 would mean identical rankings in both periods, 0 means no relation between them.

**Reading the chi-square test:** it asks whether all circuits could share one true conversion rate. A large p-value means the differences between circuits are no bigger than chance produces with so few starts per circuit.

**Is the chi-square test valid with so few starts?** Its p-value relies on an approximation that needs expected counts of about 5 or more, and 18 of our 80 expected counts are smaller. So we also computed the p-value by simulation, which needs no approximation: it gives the same answer (0.95 with both methods for 2003–2013; 0.75 vs 0.74 for 2014–2024). The conclusion does not depend on the approximation.

**Grid vs qualifying.** The same conversion, defined by qualifying position instead of grid slot:

In [ ]:
compare = (pd.DataFrame({
    "by grid slot": q1[q1["front_row_grid"]].groupby("period")["podium"].mean(),
    "by qualifying position": q1[q1["front_row_quali"]].groupby("period")["podium"].mean()}))
moved = q1[q1["front_row_quali"] & ~q1["front_row_grid"]]
moved_in = q1[q1["front_row_grid"] & ~q1["front_row_quali"]]

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
compare.plot.bar(ax=ax[0], color=["#c0392b", "#2980b9"], rot=0)
ax[0].set_ylabel("Podium rate"); ax[0].set_ylim(0, 1); ax[0].set_xlabel("")
ax[0].set_title("Front-row conversion: grid slot vs qualifying position")
for c in ax[0].containers:
    ax[0].bar_label(c, fmt="%.2f", fontsize=8)
qg = conv_grid.merge(conv_quali, on=["circuit_name", "period"], suffixes=("_grid", "_quali"))
qg = qg[(qg["starts_grid"] >= MIN_FRONT_ROW_STARTS) & (qg["starts_quali"] >= MIN_FRONT_ROW_STARTS)]
for per, col in [("2003–2013", "#7f8c8d"), ("2014–2024 (hybrid)", "#c0392b")]:
    d = qg[qg["period"] == per]
    ax[1].scatter(d["rate_quali"], d["rate_grid"], s=30, color=col, label=per, alpha=0.8)
ax[1].plot([0, 1], [0, 1], ls="--", color="grey")
ax[1].set_xlabel("Conversion by qualifying position"); ax[1].set_ylabel("Conversion by grid slot")
ax[1].set_title("Per circuit: the two definitions"); ax[1].legend(fontsize=8)
plt.tight_layout(); savefig("q1_grid_vs_qualifying"); plt.show()
print(f"Front-row qualifiers who did NOT start from the front row: {len(moved)} "
      f"(podiums: {int(moved['podium'].sum())}); drivers promoted onto the front row: {len(moved_in)}")
for per in qg["period"].unique():
    d = qg[qg["period"] == per]
    print(f"{per}: rank correlation between grid-based and qualifying-based circuit rates = "
          f"{spearmanr(d['rate_grid'], d['rate_quali']).statistic:.2f} ({len(d)} circuits)")
compare.round(3)

**How to read the charts:** *left:* the overall conversion per period, by grid slot (red) and by qualifying position
(blue). The qualifying bars are slightly lower in both periods, because some front-row qualifiers started further
back after a penalty. *Right:* one dot per circuit and period; the x-axis is its rate by qualifying position, the
y-axis its rate by grid slot. A dot on the dashed diagonal means both definitions agree. Most dots sit on or very near
it, which is why the circuit ranking barely depends on the definition (rank correlation 0.94 and 0.88).

**Answer to Q1.**
- **Highest conversion, 2003–2013:** Magny-Cours (92%, but only 12 starts), Istanbul Park (79%), Monza and Monaco
  (77%), Silverstone (73%).
- **Highest conversion, 2014–2024:** Suzuka (89%), Yas Marina (86%), Monaco (85%), Barcelona and Spa (82%).
- **Does the ranking change after 2014? The data cannot show that it does.** Among the 14 circuits with enough data in
  both periods the rank correlation is only 0.30 (p = 0.29), so the order looks reshuffled, and Monaco is the only
  circuit in the top 5 of both periods. But the chi-square test finds no real difference between circuits inside
  either period (p = 0.95 for 2003–2013, p = 0.74 for 2014–2024). With 10–26 front-row starts per circuit, chance
  alone produces a reshuffle this large. Magny-Cours and Istanbul drop out of the comparison because they were not
  raced, or barely raced, in the hybrid era.
- **The clear change is the level, not the order:** front-row conversion rose from 68% to 76% overall (z = 2.6,
  p ≈ 0.01), and it rose at 11 of the 14 circuits. In the hybrid era a front-row start became a more reliable route
  to the podium, possibly because fewer cars could fight for wins.
- **Grid vs qualifying:** defining the front row by qualifying position gives slightly lower conversion
  (67% / 74% vs 68% / 76%), because 38 front-row qualifiers started further back after penalties and only 11 of
  them (29%) still reached the podium. The circuit ranking itself barely depends on the definition (rank
  correlation 0.94 before 2014, 0.88 after).
- **Caution:** with 10–26 front-row starts per circuit, each rate has a wide interval (often ±15–20 points), so
  neighbouring positions in the ranking are not reliably different. The overall rise is the finding that holds.

**Grain:** one row per driver per race (2003–2024), filtered to front-row starters, then aggregated to circuit ×
period. Each race contributes two front-row starters. **Missingness:** drivers with no grid slot (pit-lane starts)
or no qualifying record cannot be front-row starters under that definition, so they drop out naturally rather than
being filled in. The target `podium` is never missing. **Cutoff:** grid and qualifying position are both known
before the race, and the podium is the outcome being explained. A circuit's conversion rate could only become a
model feature if it were computed from earlier races, never from the seasons being predicted.

### 3.2 Q2: Do drivers podium more often at home, after controlling for starting position?

**Step 1: who is "at home"?** A driver is at home when their nationality matches the country of the circuit.
The two are written differently ("British" vs "UK"), so we use an explicit mapping, shown in full below.
Two clean-up steps come first: one nationality has a trailing space (`"Argentinian "`), and the USA appears as both
`"USA"` and `"United States"` in `circuits`.

In [ ]:
NATIONALITY_TO_COUNTRY = {
    "American": "USA", "Argentine": "Argentina", "Argentinian": "Argentina", "Australian": "Australia",
    "Austrian": "Austria", "Belgian": "Belgium", "Brazilian": "Brazil", "British": "UK", "Canadian": "Canada",
    "Chinese": "China", "Dutch": "Netherlands", "French": "France", "German": "Germany", "Hungarian": "Hungary",
    "Indian": "India", "Italian": "Italy", "Japanese": "Japan", "Malaysian": "Malaysia", "Mexican": "Mexico",
    "Monegasque": "Monaco", "Portuguese": "Portugal", "Russian": "Russia", "South African": "South Africa",
    "Spanish": "Spain", "Swedish": "Sweden", "Swiss": "Switzerland",
    # dual nationalities: at home in either country
    "American-Italian": ["USA", "Italy"], "Argentine-Italian": ["Argentina", "Italy"],
}
NO_HOME_GP = {"Chilean", "Colombian", "Czech", "Danish", "East German", "Finnish", "Indonesian", "Irish",
              "Liechtensteiner", "New Zealander", "Polish", "Rhodesian", "Thai", "Uruguayan", "Venezuelan"}
entries["nationality"] = entries["nationality"].str.strip()
entries["circuit_country"] = entries["circuit_country"].replace({"United States": "USA"})


def home_flag(nationality, country):
    """True when the circuit lies in (one of) the driver's home countries. Phase 4 reuses it for home_race."""
    homes = nationality.str.strip().map(NATIONALITY_TO_COUNTRY)
    country = country.replace({"United States": "USA"})
    return pd.Series([c in (h if isinstance(h, list) else [h]) for h, c in zip(homes, country)],
                     index=nationality.index)


entries["home"] = home_flag(entries["nationality"], entries["circuit_country"])
assert not set(NATIONALITY_TO_COUNTRY) & NO_HOME_GP, "a nationality is both mapped and in NO_HOME_GP"
unhandled = set(entries["nationality"]) - set(NATIONALITY_TO_COUNTRY) - NO_HOME_GP
assert not unhandled, f"nationality not handled on purpose: {unhandled}"
used = entries["nationality"].value_counts().sort_index().sort_values(ascending=False, kind="stable")  # ties by name
mapping_table = pd.DataFrame({
    "entries": used,
    "home country": [NATIONALITY_TO_COUNTRY.get(n, "none (no Grand Prix in this country)") for n in used.index],
    "home entries": entries.groupby("nationality")["home"].sum().reindex(used.index),
})
unmapped = mapping_table[mapping_table["home entries"] == 0]
mapped_countries = {c for v in NATIONALITY_TO_COUNTRY.values() for c in (v if isinstance(v, list) else [v])}
assert mapped_countries <= set(entries["circuit_country"]), "mapping points to a country with no circuit"
print(f"{len(mapping_table)} nationalities; {len(mapping_table) - len(unmapped)} have at least one home race.")
print("Circuit countries that no driver nationality maps to (no driver from there in the data):",
      sorted(set(entries["circuit_country"]) - mapped_countries))
mapping_table

Nationalities with zero home entries (e.g. Finnish, New Zealander) belong to countries that never hosted a
Grand Prix in this dataset. Their drivers are always "away", which is correct, not missing data. East German and
Rhodesian drivers are deliberately left unmapped: those states never hosted a race. Every such nationality is listed in `NO_HOME_GP`, and an assert stops the run if a nationality is neither mapped nor listed, so a misspelling cannot silently turn a home race into an away race.

**Step 2: control for starting position.** Home drivers might simply start further forward, which alone would
give them more podiums. So we compare home and away **within the same grid band**: P1–3, P4–6, P7–10, P11+.
The overall effect is the average of the within-band differences, weighted by how many home entries fall in each
band. That answers: *"for home drivers, how much higher is the podium rate than for away drivers who started
from the same band?"*

**Uncertainty:** a 95% interval from a bootstrap. We resample whole races 2,000 times and recompute the effect;
the middle 95% of those results is the interval. Resampling races, not rows, respects that drivers in the same
race are not independent (only 3 can podium).

In [ ]:
GRID_BANDS = [0, 3, 6, 10, np.inf]
BAND_LABELS = ["P1–3", "P4–6", "P7–10", "P11+"]


def band_table(df):
    t = (df.groupby(["band", "home"], observed=True)["podium"].agg(["size", "mean"]).unstack("home"))
    t.columns = [f"{'home' if h else 'away'}_{s}" for s, h in t.columns]
    return t


def race_counts(df):
    """Per race: number of entries and podiums in each (band, home) cell. Shape: races × 8 cells."""
    cells = pd.MultiIndex.from_product([BAND_LABELS, [False, True]], names=["band", "home"])
    g = df.groupby(["raceId", "band", "home"], observed=True)["podium"].agg(["size", "sum"])
    n = g["size"].unstack(["band", "home"]).reindex(columns=cells, fill_value=0).fillna(0).to_numpy()
    p = g["sum"].unstack(["band", "home"]).reindex(columns=cells, fill_value=0).fillna(0).to_numpy()
    return n, p


def effect_from_totals(N, P):
    """N, P: (..., 8) totals per cell, ordered band × [away, home]. Returns the home-weighted within-band difference."""
    N, P = N.reshape(*N.shape[:-1], 4, 2), P.reshape(*P.shape[:-1], 4, 2)
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = P / N
    diff = rate[..., 1] - rate[..., 0]
    w = np.where(np.isnan(diff), 0, N[..., 1])
    return np.nansum(np.nan_to_num(diff) * w, axis=-1) / w.sum(axis=-1)


def adjusted_effect(df):
    n, p = race_counts(df)
    return float(effect_from_totals(n.sum(0), p.sum(0)))


def bootstrap_ci(df, n_boot=2000, seed=SEED):
    """Resample whole races with replacement: each bootstrap sample = how many times each race is drawn."""
    rng = np.random.default_rng(seed)
    n, p = race_counts(df)
    draws = rng.multinomial(len(n), np.full(len(n), 1 / len(n)), size=n_boot)
    return np.percentile(effect_from_totals(draws @ n, draws @ p), [2.5, 97.5])


def q2_sample(first_year, last_year=2024):
    """Entries with a grid slot, without the Indianapolis 500 (1950–1960; almost no F1 regulars entered it)."""
    keep = (entries["year"].between(first_year, last_year) & entries["grid"].notna()
            & (entries["race_name"] != "Indianapolis 500"))
    d = entries[keep].copy()
    d["band"] = pd.cut(d["grid"].astype(float), GRID_BANDS, labels=BAND_LABELS)
    return d


q2 = q2_sample(2003)
bt = band_table(q2)
bt["difference"] = bt["home_mean"] - bt["away_mean"]
bt["home_ci_low"], bt["home_ci_high"] = wilson_ci(bt["home_mean"] * bt["home_size"], bt["home_size"])
bt["away_ci_low"], bt["away_ci_high"] = wilson_ci(bt["away_mean"] * bt["away_size"], bt["away_size"])
raw_effect = q2.loc[q2["home"], "podium"].mean() - q2.loc[~q2["home"], "podium"].mean()
adj_effect = adjusted_effect(q2)
adj_ci = bootstrap_ci(q2)
print(f"2003–2024: {int(q2['home'].sum())} home entries, {int((~q2['home']).sum()):,} away entries "
      f"({int(entries['year'].between(2003, 2024).sum() - len(q2))} entries without a grid slot excluded)")
print(f"Raw difference (home − away podium rate):      {raw_effect:+.3f}")
print(f"Adjusted for grid band:                        {adj_effect:+.3f}  "
      f"(95% interval {adj_ci[0]:+.3f} to {adj_ci[1]:+.3f})")
p13 = q2[q2["band"] == "P1–3"]
print(f"Inside P1–3, share starting from pole: home {(p13.loc[p13['home'], 'grid'] == 1).mean():.0%}, "
      f"away {(p13.loc[~p13['home'], 'grid'] == 1).mean():.0%}")
bt[["home_size", "home_mean", "away_size", "away_mean", "difference"]].round(3)

**What this cell does:** it keeps the 2003–2024 entries that have a grid slot, puts each into a grid band, and compares
home and away podium rates inside each band (`bt`). `adjusted_effect` combines the four band differences, weighting
each band by its number of home entries, and `bootstrap_ci` gives the 95% interval by resampling whole races 2,000
times.

**Output:** the raw difference is −0.2 points and the grid-adjusted difference −1.5 points (interval −4.5 to +1.2).
In the table, `difference` is home minus away inside each band: home drivers are behind in P1–3 (−4.7 points) and
P7–10 (−3.0), and level in P4–6 and P11+.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4), gridspec_kw={"width_ratios": [3, 2]})
x = np.arange(len(bt)); w = 0.38
for off, side, col in [(-w / 2, "away", "#7f8c8d"), (w / 2, "home", "#c0392b")]:
    m = bt[f"{side}_mean"]
    err = [m - bt[f"{side}_ci_low"], bt[f"{side}_ci_high"] - m]
    ax[0].bar(x + off, m, w, yerr=err, capsize=3, color=col, label=side)
ax[0].set_xticks(x); ax[0].set_xticklabels([f"{b}\n(home n={int(n)})" for b, n in zip(bt.index, bt["home_size"])])
ax[0].set_ylabel("Podium rate"); ax[0].set_title("Q2: podium rate, home vs away, within the same grid band (2003–2024)")
ax[0].legend()
ax[1].bar(["raw\n(no control)", "adjusted for\ngrid band"], [raw_effect, adj_effect], color=["#bdc3c7", "#c0392b"],
          yerr=[[0, adj_effect - adj_ci[0]], [0, adj_ci[1] - adj_effect]], capsize=4)
ax[1].axhline(0, color="black", lw=0.8)
ax[1].set_ylabel("Home − away podium rate"); ax[1].set_title("Overall home effect")
plt.tight_layout(); savefig("q2_home_vs_away"); plt.show()

**How to read the charts:** *left:* for each grid band, the grey bar is the away podium rate and the red bar the home
rate, with 95% intervals. Taller red bars would mean a home advantage; instead the bars are about equal, and every red
interval overlaps its grey bar. The home intervals are wide because there are few home entries (the n under each
band). *Right:* the overall home minus away difference, raw (grey) and adjusted for grid band (red). The red error
bar crosses zero, so the data show no home effect.

**A note on the P1–3 band:** inside it, home drivers started from pole more often than away drivers (46% vs 33%, printed above), and pole converts better than P2–3. So this band, if anything, flatters the home side. The adjusted difference is still slightly negative, so the conclusion does not depend on the band edges.

**Stability checks.** A small number of home entries can be dominated by a few drivers, so we test whether the
result survives (a) removing the three drivers with the most **home races** and (b) using all seasons since 1950,
which gives many more home races but mixes eras with different field sizes and rules. Both samples leave out the Indianapolis 500 (1950–1960), an American oval race that almost no F1 regulars entered and where nearly every entrant was "at home".

Why "most home races" and not "most home podiums"? Removing the drivers with the most home podiums would select
drivers *by the outcome we are measuring*, which pushes the home rate down automatically and biases the check.

In [ ]:
home_races = q2[q2["home"]].groupby("driver").size().sort_values(ascending=False)
top3 = home_races.head(3)
q2_wo = q2[~q2["driver"].isin(top3.index)]
q2_all = q2_sample(1950)
stability = pd.DataFrame([
    ["Main: 2003–2024", int(q2["home"].sum()), adj_effect, *adj_ci],
    [f"Without {', '.join(top3.index)}", int(q2_wo["home"].sum()), adjusted_effect(q2_wo), *bootstrap_ci(q2_wo)],
    ["All seasons 1950–2024", int(q2_all["home"].sum()), adjusted_effect(q2_all), *bootstrap_ci(q2_all)],
], columns=["sample", "home entries", "adjusted effect", "ci_low", "ci_high"])
print("Drivers with the most home races (2003–2024):", top3.to_dict())
for drv in top3.index:
    d = q2[q2["driver"] == drv]
    print(f"  {drv}: podium rate at home {d.loc[d['home'], 'podium'].mean():.0%} (n = {int(d['home'].sum())}), "
          f"away {d.loc[~d['home'], 'podium'].mean():.0%}")

fig, ax = plt.subplots(figsize=(11, 2.8))
yy = np.arange(len(stability))[::-1]
ax.errorbar(stability["adjusted effect"], yy,
            xerr=[stability["adjusted effect"] - stability["ci_low"], stability["ci_high"] - stability["adjusted effect"]],
            fmt="o", color="#c0392b", capsize=4)
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(yy); ax.set_yticklabels(stability["sample"])
ax.set_xlabel("Home − away podium rate, grid-adjusted (95% interval)")
ax.set_title("Q2: does the home effect hold up?")
plt.tight_layout(); savefig("q2_stability_checks"); plt.show()
stability.round(3)

**How to read the chart:** each dot is the adjusted home effect for one sample, with its 95% interval, and the black
line is zero (no effect). An interval that crosses zero means no clear effect. The main sample and the all-seasons
sample both cross zero. The sample without Alonso, Hamilton and Button sits entirely left of zero: without them, home
drivers do worse. The lines printed above the chart show why: Alonso and Hamilton podiumed more often at home, Button
less often.

**Answer to Q2. No: once starting position is controlled, drivers do not podium more often at home.**
- Raw comparison (2003–2024): home and away podium rates are almost identical (difference −0.2 points).
- Within the same grid band, home drivers are never clearly better: the combined, grid-adjusted difference is
  **−1.5 percentage points** (95% interval −4.5 to +1.2), so zero lies inside the interval.
- **The overall "no effect" hides drivers who go both ways.** Alonso (36% podium rate at home vs 27% away) and
  Hamilton (79% vs 56%) did better at home, while Button never reached a home podium (0 of 14). Without these three,
  the remaining home drivers do significantly **worse** than away drivers from the same grid band: −3.3 points
  (interval −6.0 to −0.7, so zero is outside). This is one subgroup among several checks, so we report it as a
  pattern worth noting, not as a proven home disadvantage.
- **All seasons since 1950** (1,939 home entries, Indianapolis 500 excluded) give −0.4 points (interval −1.6 to +0.8), essentially zero.
- **Interpretation:** any extra motivation or crowd support at home does not show up in podium results beyond
  what the grid position already explains. Home entries are few (364 in 2003–2024, only 61 from the top 3 grid
  slots), so a small effect could still be hidden; the data rule out a large home advantage.
- **Limitation:** nationality is an imperfect proxy for "home" (it ignores where a driver lives or which team's
  home race it is).

**Grain:** one row per driver per race; the comparison is made inside grid bands, then combined. **Missingness:**
entries without a grid slot are excluded from the bands (they cannot be placed in one), and the Indianapolis 500 is
left out; nationalities whose country never hosted a race are "away" by definition, not missing. **Cutoff:** both
nationality and circuit are known long before the race, so a `home` flag would be a legal pre-race feature.

### 3.3 Q3: Which constructors have the highest mechanical-retirement rate in 2014–2021 vs 2022–2024?

The brief defines it: *a race start that ended early because a component of the car failed; it excludes crashes, driver causes and regulation decisions.* That gives us two things to build:
1. **The cause**, from `status`. We map every status to exactly one group. Statuses that name no cause ("Retired") go to an **ambiguous** group. We report a **strict** rate (a component is named) and a **broad** rate (strict + ambiguous), so the reader sees how much our judgment matters.
2. **The start**, for the denominator. A car that never started cannot retire, and counting it would make rates look lower than they are.

In [ ]:
q3_window = (CLEAN["results"][["raceId", "driverId", "constructorId", "laps", "positionText", "statusId"]]
             .merge(CLEAN["races"][["raceId", "year"]], on="raceId")
             .merge(CLEAN["status"], on="statusId")
             .merge(CLEAN["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor"}),
                    on="constructorId"))
q3_window = q3_window[q3_window["year"].between(2014, 2024)].copy()
q3_window["era"] = np.where(q3_window["year"] <= 2021, "2014–2021", "2022–2024")
print(f"{len(q3_window):,} entries, {q3_window['raceId'].nunique()} races, "
      f"{q3_window['status'].nunique()} distinct statuses, {q3_window['constructor'].nunique()} constructors")
q3_window.groupby("era").agg(seasons=("year", "nunique"), races=("raceId", "nunique"), entries=("driverId", "size"),
                             constructors=("constructor", "nunique"))

The window holds 4,626 entries from 228 races: 160 races in 2014–2021 (8 seasons) and 68 in 2022–2024 (3 seasons), with 66 distinct status values and 20 constructor names.

**The status map.** We list the statuses group by group and let the asserts catch anything we missed. The table under the code shows every status that occurs in 2014–2024, with its group and count (the report reuses it).

In [ ]:
STATUS_LISTS = {
    "finished": ["Finished", "Not classified"],   # plus every "+N Lap(s)"; "Not classified" = still running, too few laps
    "mechanical": [
        "Engine", "Engine fire", "Engine misfire", "Power Unit", "Power loss", "Turbo", "Supercharger", "ERS",
        "Battery", "Alternator", "Electrical", "Electronics", "Ignition", "Magneto", "Distributor", "Spark plugs",
        "Launch control", "Throttle", "Injection", "Fuel system", "Fuel pressure", "Fuel pump", "Fuel leak",
        "Fuel pipe", "Oil leak", "Oil pressure", "Oil pump", "Oil line", "Oil pipe", "Water leak", "Water pressure",
        "Water pump", "Water pipe", "Overheating", "Cooling system", "Radiator", "Heat shield fire", "Exhaust",
        "Gearbox", "Transmission", "Clutch", "Drivetrain", "Driveshaft", "Halfshaft", "CV joint", "Differential",
        "Axle", "Crankshaft", "Hydraulics", "Pneumatics", "Brakes", "Brake duct", "Suspension", "Steering",
        "Track rod", "Wheel bearing", "Chassis", "Undertray", "Mechanical", "Technical"],
    "crash / driver": ["Accident", "Fatal accident", "Collision", "Collision damage", "Damage", "Spun off", "Debris",
                       "Illness", "Driver unwell", "Physical", "Injury", "Injured", "Eye injury"],
    "regulatory": ["Disqualified", "Excluded", "Underweight", "Safety concerns"],
    "non-start": ["Did not qualify", "Did not prequalify", "107% Rule"],
    "ambiguous": ["Retired", "Withdrew", "Not restarted", "Stalled", "Wheel", "Wheel nut", "Wheel rim", "Puncture",
                  "Tyre", "Tyre puncture", "Front wing", "Rear wing", "Broken wing", "Out of fuel", "Fuel",
                  "Refuelling", "Fuel rig", "Fire", "Vibrations", "Handling", "Seat", "Driver Seat", "Safety",
                  "Safety belt"],
}
all_status = CLEAN["status"]["status"]
listed = [s for group in STATUS_LISTS.values() for s in group]
assert len(listed) == len(set(listed)), "A status is listed in two groups"
assert set(listed) <= set(all_status), f"Typo, not a real status: {set(listed) - set(all_status)}"
STATUS_GROUP = {s: g for g, group in STATUS_LISTS.items() for s in group}
STATUS_GROUP |= {s: "finished" for s in all_status[all_status.str.fullmatch(r"\+\d+ Laps?")]}
assert set(all_status) <= set(STATUS_GROUP), f"Unclassified: {set(all_status) - set(STATUS_GROUP)}"
MECHANICAL = {s for s, g in STATUS_GROUP.items() if g == "mechanical"}
print(f"All {len(all_status)} statuses in the status table classified; {len(MECHANICAL)} count as mechanical.")

counts = q3_window["status"].value_counts()
status_map = (pd.DataFrame({"status": list(STATUS_GROUP), "group": list(STATUS_GROUP.values())})
              .assign(entries=lambda d: d["status"].map(counts).fillna(0).astype(int))
              .loc[lambda d: d["entries"] > 0]
              .assign(group=lambda d: pd.Categorical(d["group"], categories=list(STATUS_LISTS)))
              .sort_values(["group", "entries"], ascending=[True, False]))
status_map["label"] = status_map["status"] + " (" + status_map["entries"].astype(str) + ")"
with pd.option_context("display.max_colwidth", None):
    display(status_map.groupby("group", observed=True)
            .agg(statuses=("status", "size"), entries=("entries", "sum"), statuses_with_entries=("label", ", ".join)))

- The map covers **all 139 statuses** in the `status` table, not only the 66 seen in 2014–2024. Phase 4 reuses `MECHANICAL` for a past-reliability feature over 2003–2024, so older statuses such as "Alternator" and "Launch control" need a group too.
- **Three asserts:** no status sits in two groups; every listed name exists in the table (this catches typos such as "Power unit"); every status in the table has a group.
- In 2014–2024, 379 entries carry one of 37 mechanical statuses, led by Engine (81), Brakes (49), Power Unit (42) and Gearbox (42).
- **"Not classified" sits with "finished" on purpose:** it marks a car that was still running at the end but had covered too few laps to be classified. Nothing failed, so it must not count as a mechanical retirement. The group name means "did not stop because of a failure", not "classified finisher".
- **Ambiguous on purpose:** "Retired" (50 entries) names no cause. "Wheel", "Wheel nut" and "Puncture" usually come from a pit-stop error or debris, not from a failed component. "Front wing" and "Rear wing" can be crash damage, and "Vibrations" often come from a flat-spotted tyre. "Withdrew" is explained in the next cell.

**Who started?** The status alone cannot say. "Withdrew" usually marks a car that never took the start, but some "Withdrew" entries completed laps. We check `positionText` against laps.

In [ ]:
code_label = q3_window["positionText"].where(~q3_window["positionText"].str.fullmatch(r"\d+"), "classified (number)")
print(pd.crosstab(code_label, q3_window["laps"].eq(0).map({True: "0 laps", False: "1+ laps"}), margins=True), "\n")

q3_window["started"] = ~q3_window["positionText"].isin(["W", "F"])
q3_window["group"] = np.where(q3_window["started"], q3_window["status"].map(STATUS_GROUP), "did not start")
q3_window["mech_strict"] = (q3_window["group"] == "mechanical").astype(int)
q3_window["mech_broad"] = q3_window["group"].isin(["mechanical", "ambiguous"]).astype(int)

dns = q3_window[~q3_window["started"]]
print(f"Entries that never took the start: {len(dns)}; by status group: "
      f"{dns['status'].map(STATUS_GROUP).value_counts().to_dict()}")
r0 = q3_window[(q3_window["positionText"] == "R") & q3_window["laps"].eq(0)]
print(f"Retired (R) with 0 laps, cars that started: {len(r0)}; by group: {r0['group'].value_counts().to_dict()}")
print("'Withdrew' entries that did start (positionText R):",
      q3_window.loc[(q3_window["status"] == "Withdrew") & q3_window["started"], ["year", "laps"]].values.tolist())
leclerc = CLEAN["drivers"].loc[CLEAN["drivers"]["surname"] == "Leclerc", "driverId"].iloc[0]
monaco21 = CLEAN["races"].loc[(CLEAN["races"]["year"] == 2021) & (CLEAN["races"]["name"] == "Monaco Grand Prix"), "raceId"].iloc[0]
print("Example, Leclerc at Monaco 2021:")
q3_window[(q3_window["raceId"] == monaco21) & (q3_window["driverId"] == leclerc)][
    ["year", "constructor", "laps", "positionText", "status", "started", "group", "mech_strict"]]

- **`positionText = "W"` marks a non-starter.** All 28 `W` entries have 0 laps. The 111 `R` (retired) entries with 0 laps did start: 101 of them are first-lap crashes or collisions. So `started = positionText not in {"W", "F"}` (`F` = failed to qualify; none in this window). Three "Withdrew" entries completed 47–55 laps and carry `R`, so they count as starts with an unrecorded cause (ambiguous).
- **This changes the numerator too.** 15 of the 28 non-starters have a mechanical status. Charles Leclerc qualified on pole at Monaco 2021, and the Ferrari's driveshaft failed on the way to the grid. The car failed, but it never made a race start, so by the brief's definition this is not a mechanical *retirement*. It leaves both numerator and denominator.

**Rates per constructor and era**, over starts only. We keep a constructor-era pair with **at least 40 starts** (about one full season with two cars); teams with a handful of races would show extreme rates from tiny samples.

In [ ]:
MIN_STARTS_Q3 = 40   # about one full season with two cars
starts = q3_window[q3_window["started"]]
strict = rate_table(starts, ["constructor", "era"], "mech_strict")
broad = rate_table(starts, ["constructor", "era"], "mech_broad")
q3_all = (strict.rename(columns={"n": "starts", "hits": "mech_strict", "rate": "strict_rate"})
          .merge(broad[["constructor", "era", "hits", "rate"]].rename(columns={"hits": "mech_broad", "rate": "broad_rate"}),
                 on=["constructor", "era"]))
q3_small = q3_all[q3_all["starts"] < MIN_STARTS_Q3]
q3_rates = q3_all[q3_all["starts"] >= MIN_STARTS_Q3].copy()
q3_rates["rank"] = q3_rates.groupby("era")["strict_rate"].rank(ascending=False, method="min").astype(int)
q3_rates = q3_rates.sort_values(["era", "rank"]).reset_index(drop=True)
print(f"Constructor-era pairs kept: {len(q3_rates)}; dropped for fewer than {MIN_STARTS_Q3} starts: "
      f"{sorted(f'{c} ({e}, {n})' for c, e, n in q3_small[['constructor', 'era', 'starts']].itertuples(index=False))}")

era_mech = rate_table(starts, "era", "mech_strict").merge(
    rate_table(starts, "era", "mech_broad")[["era", "hits", "rate"]], on="era", suffixes=("_strict", "_broad"))
z, p_era = proportions_ztest(era_mech["hits_strict"], era_mech["n"])
print(era_mech.round(4).to_string(index=False))
print(f"Two-proportion z-test, strict rate 2014–2021 vs 2022–2024: z = {z:.2f}, p = {p_era:.2g}")
q3_rates.round(3)

- `MIN_STARTS_Q3 = 40` removes Caterham (33 starts) and Marussia (31).
- **Reliability improved between the eras:** the strict rate fell from **8.8%** (287 of 3,246 starts) to **5.7%** (77 of 1,352), z = 3.60, p = 0.0003. The broad rate fell the same way, from 11.0% to 6.9%.

In [ ]:
n_max = q3_rates.groupby("era").size().max()
fig, axes = plt.subplots(1, 2, figsize=(15, 0.36 * n_max + 1.6))
for ax, (era, d) in zip(axes, q3_rates.groupby("era")):
    d = d.sort_values("strict_rate")
    ax.barh(d["constructor"] + " (" + d["starts"].astype(str) + " starts)", d["strict_rate"],
            xerr=[d["strict_rate"] - d["ci_low"], d["ci_high"] - d["strict_rate"]],
            color="#c0392b", ecolor="#7f8c8d", capsize=2)
    overall = era_mech.set_index("era").at[era, "rate_strict"]
    ax.axvline(overall, ls="--", color="#2c3e50", label=f"all constructors: {overall:.1%}")
    ax.set_xlim(0, max(0.3, q3_rates["ci_high"].max() + 0.01)); ax.legend(loc="lower right", fontsize=8)
    ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
    ax.set_title(f"{era}: mechanical retirements per race start (strict)")
    ax.set_xlabel("Mechanical-retirement rate (95% Wilson CI)")
plt.tight_layout(); savefig("q3_mechanical_by_constructor"); plt.show()

both = q3_rates.pivot(index="constructor", columns="era", values="strict_rate").dropna().sort_values("2014–2021")
fig, ax = plt.subplots(figsize=(9, 0.45 * len(both) + 1.5))
for y, (c, r) in enumerate(both.iterrows()):
    ax.plot([r["2014–2021"], r["2022–2024"]], [y, y], color="#bdc3c7", lw=3, zorder=1)
ax.scatter(both["2014–2021"], range(len(both)), color="#2c3e50", s=60, label="2014–2021", zorder=2)
ax.scatter(both["2022–2024"], range(len(both)), color="#e67e22", s=60, label="2022–2024", zorder=2)
ax.set_yticks(range(len(both))); ax.set_yticklabels(both.index)
ax.xaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
ax.set_xlabel("Mechanical-retirement rate per start (strict)"); ax.legend()
ax.set_title("Constructors present in both eras: strict mechanical-retirement rate")
plt.tight_layout(); savefig("q3_dumbbell"); plt.show()

season = rate_table(starts, "year", "mech_strict").merge(rate_table(starts, "year", "mech_broad")[["year", "rate"]],
                                                       on="year", suffixes=("_strict", "_broad"))
fig, ax = plt.subplots(figsize=(11, 3.4))
ax.fill_between(season["year"], season["ci_low"], season["ci_high"], color="#c0392b", alpha=0.15)
ax.plot(season["year"], season["rate_strict"], marker="o", color="#c0392b", label="strict (component named)")
ax.plot(season["year"], season["rate_broad"], marker="o", ls="--", color="#7f8c8d", label="broad (+ ambiguous)")
ax.axvline(2021.5, color="black", ls=":"); ax.text(2021.6, ax.get_ylim()[1] * 0.92, "2022 rules", fontsize=9)
ax.yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1)); ax.set_xticks(season["year"])
ax.set_xlabel("Season"); ax.set_ylabel("Rate per start"); ax.legend()
ax.set_title("Mechanical-retirement rate per race start, all constructors")
plt.tight_layout(); savefig("q3_by_season"); plt.show()
print("Strict rate per season:", ", ".join(f"{y}: {r:.1%}" for y, r in zip(season["year"], season["rate_strict"])))

**How to read the three charts:**
- *Bars per era:* each bar is a team's strict rate, the thin line its 95% interval, and the dashed line the all-teams
  average. A team is clearly worse than average only if its whole interval sits to the right of the dashed line.
- *Dumbbell:* one line per team present in both eras; the dark dot is 2014–2021 and the orange dot 2022–2024. A long
  line means a big change, and the direction shows whether the team got better (moved left) or worse (moved right).
- *Season line:* the strict rate per season (red, with its interval as a shaded band) and the broad rate (grey
  dashes). The dotted line marks the 2022 rule change.

- **2014–2021:** Lotus F1 (17.3% of 75 starts), Toro Rosso (16.1% of 242) and Renault (13.1% of 199) failed most often. Among the teams that raced the whole period, Mercedes (3.8%) and Ferrari (4.7%) failed least.
- **2022–2024:** Alfa Romeo (12.5% of 88), Alpine (10.4% of 135) and AlphaTauri (8.0% of 88) lead. The intervals are wide, but the lower bounds of Alfa Romeo (7.1%) and Alpine (6.3%) still sit above the field's 5.7%.
- **Dumbbell (11 constructors in both eras):** McLaren improved most (12.7% → 3.7%), followed by Sauber (10.6% → 2.1%, on only 48 starts) and Red Bull (10.7% → 3.7%). Alfa Romeo (3.3% → 12.5%) and Alpine (6.8% → 10.4%) got worse.
- **Season trend:** the drop is not a clean step at 2022. The strict rate was already falling (13.5% in 2017, 4.3% in 2021), jumped to 8.9% in 2022, the first season of the new cars, then fell to 2.9% in 2024. The era gap mixes a long decline with a one-season spike after the rule change.

**Sensitivity:** does the ranking survive the broad definition, and where do the high rates come from?

In [ ]:
top = []
for era, d in q3_rates.groupby("era"):
    s = d.nlargest(5, "strict_rate").reset_index(drop=True)
    b = d.nlargest(5, "broad_rate").reset_index(drop=True)
    top.append(pd.DataFrame({"era": era, "place": range(1, 6),
                             "strict": s["constructor"] + " " + (100 * s["strict_rate"]).round(1).astype(str) + "%",
                             "broad": b["constructor"] + " " + (100 * b["broad_rate"]).round(1).astype(str) + "%"}))
print(pd.concat(top).to_string(index=False), "\n")
amb = starts[starts["group"] == "ambiguous"]
print("Ambiguous statuses among starters, per era:")
print(pd.crosstab(amb["status"], amb["era"], margins=True).sort_values("All", ascending=False).to_string(), "\n")

# Where do the worst rates come from? Strict failures / starts per season for each era's top 4.
worst = q3_rates[q3_rates["rank"] <= 4]
per_season = (starts.merge(worst[["constructor", "era"]], on=["constructor", "era"])
              .groupby(["era", "constructor", "year"])["mech_strict"].agg(["sum", "size"]))
per_season["failures/starts"] = per_season["sum"].astype(str) + "/" + per_season["size"].astype(str)
per_season["failures/starts"].unstack("year").fillna("")

- **Strict vs broad:** in 2014–2021 the top 4 is identical (Lotus F1, Toro Rosso, Renault, McLaren), and Sauber (13.1% broad) replaces Red Bull in 5th. In 2022–2024 Alfa Romeo stays first, and RB F1 Team jumps from 4.2% to 10.4% on 3 extra ambiguous entries out of 48 starts. "Retired" (49 of the 87 ambiguous entries among starters) is the status that moves rates most.
- **High era rates come from a few bad seasons.** Toro Rosso had 18 failures in 76 starts in 2014–2015, Renault 10 in 40 in 2017, and Lotus F1 8 in 37 in 2014. McLaren had 25 of its 40 failures in 2015–2017, its Honda-engine seasons. One era rate per constructor hides these swings.

> **Answer (Q3).** In 2014–2021 the highest strict mechanical-retirement rates belonged to Lotus F1 (17.3% of 75 starts), Toro Rosso (16.1% of 242) and Renault (13.1% of 199). In 2022–2024 they belonged to Alfa Romeo (12.5% of 88), Alpine (10.4% of 135) and AlphaTauri (8.0% of 88). Across all teams, mechanical retirements fell from 8.8% to 5.7% of starts (p = 0.0003), although the decline began before 2022 and the first season of the new rules spiked to 8.9%. The broad definition keeps the same leader in each era (Lotus F1, Alfa Romeo); only RB F1 Team moves up, on 3 ambiguous entries.
>
> **Joins:** `results` → `races` on `raceId` (season), → `status` on `statusId` (cause), → `constructors` on `constructorId` (team name). **Grain:** one driver in one race; after aggregation, one constructor in one era, counted over race starts only.
>
> **Missingness:** status is never missing. The start flag comes from `positionText` (`W` = did not start), which removes 28 non-starters from numerator and denominator. Two teams below 40 starts are excluded.
>
> **Cutoff:** `status` is post-race information. That is fine here, because Q3 describes past races; status never becomes a feature for the race it describes. Phase 4 uses `MECHANICAL` only on *earlier* races, for a past-reliability feature.
>
> **Caveats:** teams are counted per `constructorId` as recorded, so renames split one team's history: Toro Rosso → AlphaTauri → RB F1 Team, Force India → Racing Point → Aston Martin, Sauber → Alfa Romeo → Sauber, Lotus F1 → Renault → Alpine. The status-to-group map is our judgment; the strict/broad pair shows its effect. The 2022–2024 era has only 3 seasons, so its intervals are wider.

### 3.4 Phase 3 gate
The run stops here if any check fails: a join that changed the row count, an unhandled nationality or status, more
than two grid front-row starters in a race, a rate outside [0, 1], a group below its sample threshold, a non-starter
with laps recorded, or any change to `CLEAN`.

In [ ]:
assert len(entries) == len(CLEAN["results"]) and not entries.duplicated(["raceId", "driverId"]).any()
assert set(CLEAN["drivers"]["nationality"].str.strip()) <= set(NATIONALITY_TO_COUNTRY) | NO_HOME_GP
assert q3_window["status"].map(STATUS_GROUP).notna().all()                  # every status in the window classified
assert (q1[q1["front_row_grid"]].groupby("raceId").size() <= 2).all()       # at most 2 grid front-row starters per race
assert (eligible["starts"] >= MIN_FRONT_ROW_STARTS).all() and (q3_rates["starts"] >= MIN_STARTS_Q3).all()
for t in (eligible, conv_quali, bt, q3_rates):
    assert t.filter(regex="rate|mean").apply(lambda s: s.between(0, 1)).all().all()
assert q3_window.loc[~q3_window["started"], "laps"].eq(0).all()             # every non-starter has 0 laps
assert {t: int(pd.util.hash_pandas_object(df).sum()) for t, df in CLEAN.items()} == CLEAN_FINGERPRINT, "CLEAN changed"
print("Phase 3 gate passed: joins, nationalities and statuses complete, all rates valid, CLEAN unchanged.")

**Outputs for the next phases.** Phase 4 reuses `home_flag()` (with `NATIONALITY_TO_COUNTRY` and `NO_HOME_GP`) for
the `home_race` feature, and `MECHANICAL` for the past mechanical-DNF rate. The joined table `entries` holds one row
per driver per race with names, countries and status. The answer tables are `eligible` (Q1), `bt` and `stability` (Q2)
and `q3_rates` (Q3), and every figure is saved in `figures/` for the report.

## 4. Modelling table, features, split and preprocessing

**What the brief asks:** one row per driver per race, with leakage-safe grid, qualifying, form and standings features;
evidence behind every feature; a justified train / validation / test split; the effect of preprocessing shown on the
data; and at least two preprocessing-order experiments.

**The cutoff, in one sentence:** a feature may only use information that exists *after qualifying and before the
start* of the race it describes. Results of *earlier* races are allowed (that is what "form" means); the result of
the *current* race never is.

| Step | What happens |
|---|---|
| 4.1 | Why rows must be ordered by (year, round), never by `raceId` |
| 4.2 | The 18 features, and one function `build_features()` that computes all of them |
| 4.3 | Leakage audit: every column labelled pre-race or post-race, and a blocklist |
| 4.4 | Leakage tests: a perturbation test and two hand checks |
| 4.5 | Evidence for each feature, from the training seasons only |
| 4.6 | Time-based split: train 2003–2019, validation 2020–2021, test 2022–2024 |
| 4.7 | Preprocessing: which gaps are filled where, and what each step does to the data |
| 4.8 | Experiments E1 and E2 on the order of preprocessing steps, and E3, what real leakage does to the score |
| 4.9 | Phase 4 gate |

### 4.1 Order rows by date, never by `raceId`
Every form feature looks at "the previous races". That only works if we know which race came first. The obvious key,
`raceId`, is a trap:

In [ ]:
r = CLEAN["races"][["raceId", "year", "round", "date", "name"]]
by_id = r.sort_values("raceId")
print("Sorted by raceId, the first races are:", by_id.head(3)[["raceId", "year", "name"]].values.tolist())
print(f"The 1950 season has raceIds {r.loc[r['year'] == 1950, 'raceId'].min()}–{r.loc[r['year'] == 1950, 'raceId'].max()}")
print("Is raceId in date order?", by_id["date"].is_monotonic_increasing)
gapless = r.groupby("year")["round"].agg(lambda s: sorted(s) == list(range(1, len(s) + 1)))
assert gapless.all(), "a season skips or repeats a round number"
print(f"All {len(gapless)} seasons number their rounds 1, 2, 3, ... without gaps, so 'the previous race' is round − 1.")

**Result:** `raceId` 1 is the **2009** Australian Grand Prix, and the 1950 races have ids 833–839, so `raceId` is not
in date order. Sorting by it would put 2009 races before 1950 and mix future races into "past form". We always sort by
**(year, round)**. Every season numbers its rounds without gaps, so "the previous race of the season" is simply round − 1.

### 4.2 The features and the function that builds them
Six groups. The groups are also the units of the feature-group ablation in Phase 6.

| Group | Feature | What it is | Why it is known before the race | If missing |
|---|---|---|---|---|
| Grid | `grid` | starting slot | fixed when the grid forms | no slot → field size (back of the grid) |
| | `no_grid_slot` | 1 = no grid slot (pit-lane start or no start, see 2.5) | decided before the start | never missing |
| | `field_size` | number of cars entered | entry list is published before the weekend | never missing |
| Qualifying | `quali_pos` | qualifying position | qualifying is over | no qualifying record → field size |
| | `gap_to_pole_pct` | best qualifying lap, % slower than the fastest lap of that qualifying | qualifying is over | no lap time → slowest gap of that race |
| | `grid_minus_quali` | grid − qualifying position (> 0 = grid penalty) | both are known | computed from the filled values |
| Driver form | `drv_avg_finish_last5` | mean finishing position over the driver's previous 5 races | earlier races only | training median + indicator |
| | `drv_podium_rate_last5` | share of podiums over the previous 5 races | earlier races only | training median + indicator |
| | `drv_prior_races` | number of earlier races in the data | earlier races only | 0 for a debut |
| Constructor form | `con_avg_finish_last5` | team's mean finish over its previous 5 races | earlier races only | training median + indicator |
| | `con_podium_rate_last5` | team podiums ÷ team cars, previous 5 races | earlier races only | training median + indicator |
| | `con_mech_dnf_rate_last10` | team mechanical failures ÷ team cars, previous 10 races | earlier races only | training median + indicator |
| Standings | `drv_pos_before`, `con_pos_before` | championship position after the previous round | published after the previous race | training median + indicator |
| | `drv_pts_share_before`, `con_pts_share_before` | points ÷ leader's points after the previous round | published after the previous race | 0 (round 1, or new this season) |
| Context | `home_race` | race in the driver's home country (Phase 3 `home_flag`) | calendar + nationality | never missing |
| | `round` | round number in the season | calendar | never missing |

**Choices worth defending:**
- **Points share, not raw points.** A win was worth 10 points until 2009 and 25 from 2010, so raw points mean
  different things in different seasons. "Share of the leader's points" means the same thing in every season.
- **Position and gap for qualifying, not "reached Q3".** Q2 and Q3 do not exist before 2006 (Section 2.4).
- **The mechanical rate reuses `MECHANICAL` from Phase 3** and counts every car that stopped with a mechanical status,
  including the few that broke before the start. Q3 left those out because the brief's question is about retirements
  from a start; here the question is simply "how reliable have this team's cars been lately?".
- **No sprint results** (see 4.4).

In [ ]:
FEATURES = ["grid", "no_grid_slot", "field_size",
            "quali_pos", "gap_to_pole_pct", "grid_minus_quali",
            "drv_avg_finish_last5", "drv_podium_rate_last5", "drv_prior_races",
            "con_avg_finish_last5", "con_podium_rate_last5", "con_mech_dnf_rate_last10",
            "drv_pos_before", "drv_pts_share_before", "con_pos_before", "con_pts_share_before",
            "home_race", "round"]
GROUPS = {"grid": ["grid", "no_grid_slot", "field_size"],
          "qualifying": ["quali_pos", "gap_to_pole_pct", "grid_minus_quali"],
          "driver_form": ["drv_avg_finish_last5", "drv_podium_rate_last5", "drv_prior_races"],
          "constructor_form": ["con_avg_finish_last5", "con_podium_rate_last5", "con_mech_dnf_rate_last10"],
          "standings": ["drv_pos_before", "drv_pts_share_before", "con_pos_before", "con_pts_share_before"],
          "context": ["home_race", "round"]}
assert sum(GROUPS.values(), []) == FEATURES
STATUS_TEXT = CLEAN["status"].set_index("statusId")["status"]


def past_mean(s, k):
    """Mean of the k values BEFORE the current row: shift(1) removes the current race first."""
    return s.shift(1).rolling(k, min_periods=1).mean()


def past_sum(s, k):
    """Sum of the k values before the current row."""
    return s.shift(1).rolling(k, min_periods=1).sum()


def build_features(results, driver_st, constructor_st, fill=True):
    """One row per (raceId, driverId) for 2003–2024. Every feature uses only information known before that race."""
    races = CLEAN["races"][["raceId", "year", "round", "circuitId"]]
    hist = (results[["raceId", "driverId", "constructorId", "grid", "no_grid_slot", "positionOrder", "podium", "statusId"]]
            .merge(races, on="raceId"))
    hist["positionOrder"] = hist["positionOrder"].astype(float)
    hist["field_size"] = hist.groupby("raceId")["driverId"].transform("size")        # the entry list, known before the weekend

    # Driver form: the driver's previous races only, in date order (year, round), never by raceId
    hist = hist.sort_values(["driverId", "year", "round"])
    g = hist.groupby("driverId")
    hist["drv_avg_finish_last5"] = g["positionOrder"].transform(lambda s: past_mean(s, 5))
    hist["drv_podium_rate_last5"] = g["podium"].transform(lambda s: past_mean(s.astype(float), 5))
    hist["drv_prior_races"] = g.cumcount()

    # Constructor form: first one row per constructor per race (it has two cars), then roll over its previous races
    hist["mech_dnf"] = hist["statusId"].map(STATUS_TEXT).isin(MECHANICAL).astype(int)
    cr = (hist.groupby(["constructorId", "raceId", "year", "round"], as_index=False)
              .agg(fin=("positionOrder", "mean"), pods=("podium", "sum"), cars=("driverId", "size"), mech=("mech_dnf", "sum"))
              .sort_values(["constructorId", "year", "round"]))
    gc = cr.groupby("constructorId")
    cr["con_avg_finish_last5"] = gc["fin"].transform(lambda s: past_mean(s, 5))
    cr["con_podium_rate_last5"] = gc["pods"].transform(lambda s: past_sum(s, 5)) / gc["cars"].transform(lambda s: past_sum(s, 5))
    cr["con_mech_dnf_rate_last10"] = (gc["mech"].transform(lambda s: past_sum(s, 10))
                                      / gc["cars"].transform(lambda s: past_sum(s, 10)))
    hist = hist.merge(cr[["constructorId", "raceId", "con_avg_finish_last5", "con_podium_rate_last5",
                          "con_mech_dnf_rate_last10"]], on=["constructorId", "raceId"], how="left")

    # Standings BEFORE the race = standings after the previous round of the same season (round 1 has none)
    prev = (races[["raceId", "year", "round"]].assign(round=lambda d: d["round"] + 1)
            .rename(columns={"raceId": "prev_raceId"}))
    hist = hist.merge(prev, on=["year", "round"], how="left")
    for table, key, p in [(driver_st, "driverId", "drv"), (constructor_st, "constructorId", "con")]:
        s = (table[["raceId", key, "points", "position"]]
             .rename(columns={"raceId": "prev_raceId", "position": f"{p}_pos_before"}))
        leader = s.groupby("prev_raceId")["points"].transform("max").replace(0, np.nan)
        s[f"{p}_pts_share_before"] = s["points"] / leader
        hist = hist.merge(s.drop(columns="points"), on=["prev_raceId", key], how="left")

    # Qualifying: position, and the best lap as % slower than the fastest lap of that qualifying session
    q = (CLEAN["qualifying"][["raceId", "driverId", "position", "q1_sec", "q2_sec", "q3_sec"]]
         .rename(columns={"position": "quali_pos"}))
    q["best_sec"] = q[["q1_sec", "q2_sec", "q3_sec"]].min(axis=1)
    q["gap_to_pole_pct"] = 100 * (q["best_sec"] / q.groupby("raceId")["best_sec"].transform("min") - 1)
    hist = hist.merge(q[["raceId", "driverId", "quali_pos", "gap_to_pole_pct"]], on=["raceId", "driverId"], how="left")

    # Context: home race (Phase 3 helper) and the round number
    hist = (hist.merge(CLEAN["drivers"][["driverId", "nationality"]], on="driverId")
                .merge(CLEAN["circuits"][["circuitId", "country"]], on="circuitId"))
    hist["home_race"] = home_flag(hist["nationality"], hist["country"]).astype(int)

    for c in ["grid", "quali_pos", "drv_pos_before", "con_pos_before"]:
        hist[c] = hist[c].astype(float)
    if fill:   # domain fills: each uses only information from the same race, so it is safe before the split
        hist["grid"] = hist["grid"].fillna(hist["field_size"])                       # no grid slot → back of the grid
        hist["quali_pos"] = hist["quali_pos"].fillna(hist["field_size"])             # no qualifying record → last
        slowest = hist.groupby("raceId")["gap_to_pole_pct"].transform("max")
        hist["gap_to_pole_pct"] = hist["gap_to_pole_pct"].fillna(slowest)            # no lap time → slowest in that race
        shares = ["drv_pts_share_before", "con_pts_share_before"]
        hist[shares] = hist[shares].fillna(0)                                         # round 1 or new this season → 0 points
    hist["grid_minus_quali"] = hist["grid"] - hist["quali_pos"]                       # > 0: started behind its qualifying slot

    out = hist[hist["year"].between(2003, 2024)].sort_values(["year", "round", "driverId"])
    return out[["raceId", "driverId", "constructorId", "year"] + FEATURES + ["podium"]].reset_index(drop=True)

**How `build_features` works, step by step:**
1. Join `results` with `races`, so every row has its year and round.
2. **Driver form.** Sort by driver, year and round. `past_mean` first shifts the column down by one race
   (`shift(1)`) and only then averages the last 5 values. Because of that shift, race r only ever sees races before r.
   This one call is what makes the form features legal.
3. **Constructor form.** A team has two cars per race, so we first collapse to one row per team per race, then roll.
   Rolling over driver rows would count every race twice.
4. **Standings.** The standings table for race r already includes race r's points. So each race is joined to the
   standings after round − 1 of the same season. Round 1 has no previous round, so its standings features are empty.
5. **Qualifying gap** and **home race** (with `home_flag` from Phase 3).
6. **Domain fills** (only when `fill=True`). Each uses information from the same race only, such as its field size.
7. Everything is computed on the **full history (1950–2024)** and only then cut to 2003–2024, so the first races of
   2003 still have five earlier races of form.

The function takes `results` and the two standings tables as inputs, so the leakage test in 4.4 can call it again
with deliberately damaged copies.

In [ ]:
model_df = build_features(CLEAN["results"], CLEAN["driver_standings"], CLEAN["constructor_standings"])
assert not model_df.duplicated(["raceId", "driverId"]).any()
print(f"model_df: {len(model_df):,} rows (one per driver per race), {model_df['raceId'].nunique()} races, "
      f"{model_df['year'].min()}–{model_df['year'].max()}, {len(FEATURES)} features, podium rate {model_df['podium'].mean():.3f}")

surname = CLEAN["drivers"].set_index("driverId")["surname"]
example = model_df[(model_df["year"] == 2019) & (model_df["round"] == 10)].nsmallest(3, "grid")
print("Example: the front three of the 2019 British Grand Prix (round 10), one column per driver")
example.assign(driver=example["driverId"].map(surname)).set_index("driver")[FEATURES + ["podium"]].T.round(3)

**Result:** 8,970 rows, one per driver per race, across 428 races. 14.3% of them are podiums, close to 3 out of 20.

**How to read the example:** each column is one prediction row, the front three of the 2019 British Grand Prix.
- Bottas started from pole; Hamilton was 0.007% slower in qualifying, and `home_race = 1` because Hamilton is British.
- Leclerc had only 30 earlier races (`drv_prior_races`) and averaged 6.6 over the last five.
- Before this race Hamilton led the championship (`drv_pts_share_before = 1.0`). Leclerc was 5th with 53% of
  Hamilton's points, and Ferrari had 63% of Mercedes' points.
- All three finished on the podium (`podium = 1`). Every value came from qualifying or from earlier races.

### 4.3 Leakage audit
We label **every column** of the joined table `entries` from Phase 3 as an id, pre-race or post-race. The post-race
labels become the blocklist `POST_RACE`. Two asserts guard it: no column may be left unlabelled, and no feature may
be on the blocklist.

In [ ]:
LABELS = {
    "id / join key": ["resultId", "raceId", "driverId", "constructorId", "circuitId"],
    "pre-race": ["number", "grid", "no_grid_slot", "year", "round", "race_name", "circuit_name", "circuit_country",
                 "quali_pos", "driver", "nationality", "constructor", "home"],
    "post-race (result of this race)": ["position", "positionText", "positionOrder", "points", "laps", "time",
                                        "milliseconds", "fastestLap", "rank", "fastestLapTime", "fastestLapSpeed",
                                        "fastestLapTime_sec", "statusId", "status", "podium"],
    "post-race (created in cleaning, 2.6)": ["best_positionOrder", "n_rows_for_driver", "shared_drive"],
}
labelled = sum(LABELS.values(), [])
assert sorted(labelled) == sorted(entries.columns), f"unlabelled: {set(entries.columns) - set(labelled)}"
POST_RACE = LABELS["post-race (result of this race)"] + LABELS["post-race (created in cleaning, 2.6)"]
assert not set(FEATURES) & set(POST_RACE), "A post-race column is in the feature list"
print(f"All {len(labelled)} columns of the joined table are labelled; {len(POST_RACE)} are post-race and blocked.")
print("Tables never used for the current race: lap_times, pit_stops, constructor_results (post-race), "
      "current-race driver/constructor standings (post-race), sprint_results (see 4.4).")
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame([(lab, len(cols), ", ".join(cols)) for lab, cols in LABELS.items()],
                         columns=["label", "columns", "names"]).set_index("label"))

**Result:** all 36 columns are labelled: 5 ids, 13 pre-race and 18 post-race. The blocklist includes the three columns
created during cleaning (`best_positionOrder`, `n_rows_for_driver`, `shared_drive`), because they describe the result
or a mid-race car swap.

**Why `grid` is allowed but `laps` is not:** the grid is fixed before the lights go out, while laps are counted during
the race. Post-race columns of *earlier* races are fine, and the form features are built from them. The rule is only
that no feature uses the result of the race it predicts.

### 4.4 Leakage tests
**Test 1: perturbation.** A test that can never fail proves nothing, so this one is built to be able to fail. We take
one race (the 2015 Hungarian Grand Prix), scramble its finishing order and statuses, wipe the standings published
after it, and rebuild every feature from the damaged data. Two things must hold:
1. that race's own features do **not** change (it cannot see its own result);
2. the **next** race's features **do** change (otherwise the test could not detect anything).

In [ ]:
races_tbl = CLEAN["races"]
probe = races_tbl[(races_tbl["year"] == 2015) & (races_tbl["round"] == 10)].iloc[0]
nxt = races_tbl[(races_tbl["year"] == 2015) & (races_tbl["round"] == 11)].iloc[0]
print(f"Probe race: {probe['year']} {probe['name']} (round 10); next race: {nxt['name']} (round 11)")

R, DS, CS = CLEAN["results"].copy(), CLEAN["driver_standings"].copy(), CLEAN["constructor_standings"].copy()
rng = np.random.default_rng(SEED)
m = R["raceId"] == probe["raceId"]
R.loc[m, "positionOrder"] = rng.permutation(R.loc[m, "positionOrder"].to_numpy())     # scramble the finishing order
R.loc[m, "podium"] = (R.loc[m, "positionOrder"] <= 3).astype(int)
R.loc[m, "statusId"] = rng.permutation(R.loc[m, "statusId"].to_numpy())               # scramble why cars stopped
for table in (DS, CS):                                                                 # wipe the standings after it
    mt = table["raceId"] == probe["raceId"]
    table.loc[mt, "points"] = 0.0
    table.loc[mt, "position"] = rng.permutation(table.loc[mt, "position"].to_numpy())
tampered = build_features(R, DS, CS)


def race_features(df, race_id):
    return df[df["raceId"] == race_id].set_index(["raceId", "driverId"])[FEATURES].sort_index()


pd.testing.assert_frame_equal(race_features(model_df, probe["raceId"]), race_features(tampered, probe["raceId"]))
print("1. The probe race's own features are identical after scrambling its result: it cannot see its own outcome.")
before, after = race_features(model_df, nxt["raceId"]), race_features(tampered, nxt["raceId"])
changed = ((before != after) & ~(before.isna() & after.isna())).sum()
assert changed.sum() > 0, "the test cannot detect a change, so it proves nothing"
print("2. The next race's features did change, so the test is able to detect leakage. Rows changed per feature:")
changed[changed > 0]

**Result:** the Hungarian Grand Prix's features are identical before and after the damage. The next race (Belgium) changed
in exactly the 9 form and standings features, in 4–20 driver rows each. Those are the features that should depend on
the previous race. Grid, qualifying and context did not move. So the form and standings features look backwards only,
and the test is sensitive enough to catch it if they did not.

**Test 2: hand checks.** We recompute two features for one row by hand and compare.

In [ ]:
drv = CLEAN["drivers"].loc[CLEAN["drivers"]["surname"] == "Leclerc", "driverId"].iloc[0]
row = model_df[(model_df["driverId"] == drv) & (model_df["year"] == 2019) & (model_df["round"] == 10)].iloc[0]

history = (CLEAN["results"].merge(CLEAN["races"][["raceId", "year", "round", "name"]], on="raceId")
           .loc[lambda d: d["driverId"] == drv].sort_values(["year", "round"]))
last5 = history[(history["year"] < 2019) | ((history["year"] == 2019) & (history["round"] < 10))].tail(5)
print("Leclerc's five races before the 2019 British GP:")
print(last5[["year", "round", "name", "positionOrder"]].to_string(index=False))
by_hand = last5["positionOrder"].astype(float).mean()
print(f"Mean finish by hand: {by_hand:.2f} | drv_avg_finish_last5 in model_df: {row['drv_avg_finish_last5']:.2f}")
assert np.isclose(by_hand, row["drv_avg_finish_last5"])

prev_id = races_tbl.loc[(races_tbl["year"] == 2019) & (races_tbl["round"] == 9), "raceId"].iloc[0]
standing = CLEAN["driver_standings"][CLEAN["driver_standings"]["raceId"] == prev_id]
share = standing.loc[standing["driverId"] == drv, "points"].iloc[0] / standing["points"].max()
print(f"Points share after round 9 by hand: {share:.3f} | drv_pts_share_before: {row['drv_pts_share_before']:.3f}")
assert np.isclose(share, row["drv_pts_share_before"])

sprint_years = CLEAN["sprint_results"].merge(CLEAN["races"][["raceId", "year"]], on="raceId")["year"].value_counts().sort_index()
print("Sprint result rows per season:", {int(k): int(v) for k, v in sprint_years.items()})
LEAKAGE_TESTS_PASSED = True

**Result:** Leclerc's five races before the 2019 British Grand Prix ended 5th, 20th (the Monaco retirement), 3rd, 3rd
and 2nd. Their mean, 6.60, equals `drv_avg_finish_last5`. Leclerc's points share after round 9, 0.533, equals
`drv_pts_share_before`.

**Why sprint results are left out:** sprints happen before the Sunday race, so they would be legal. But sprint rows
exist only from 2021 (60–120 per season) and there are **none** in the training seasons 2003–2019, so a model could
never learn from them. In 2021–2022 the sprint result also set the Sunday grid, so that information is already
inside `grid`.

### 4.5 Evidence for every feature (training seasons only)
All evidence below uses the **training seasons, 2003–2019**. Choosing features while looking at validation or test
seasons would leak them into our design. There are four views: podium rate by feature value, a one-number score per
feature (univariate AUC), the correlation between features, and missing values.

In [ ]:
train_view = model_df[model_df["year"] <= 2019]          # evidence from the training seasons only
base_rate = train_view["podium"].mean()
EXACT = {"grid": (None, 20), "quali_pos": (None, 20), "drv_pos_before": (None, 20), "grid_minus_quali": (-5, 5),
         "con_pos_before": None, "round": None, "field_size": None, "no_grid_slot": None, "home_race": None}

fig, axes = plt.subplots(5, 4, figsize=(16, 17))
for ax, f in zip(axes.ravel(), FEATURES):
    x = train_view[f]
    if f in EXACT:                       # small integer features: one bar per value (extremes grouped)
        lo, hi = EXACT[f] or (None, None)
        rate = train_view.groupby(x.clip(lo, hi))["podium"].mean()
        labels = [(f"≤{v:g}" if v == lo else f"{v:g}+" if v == hi else f"{v:g}") for v in rate.index]
        xlabel = "value" + (" (extremes grouped)" if EXACT[f] else "")
    else:                                # continuous features: up to 10 groups of similar size
        rate = train_view.groupby(pd.qcut(x, 10, duplicates="drop"), observed=True)["podium"].mean()
        labels = [str(i + 1) for i in range(len(rate))]
        xlabel = "value group (1 = lowest values)"
    ax.bar(range(len(rate)), rate.values, color=next(c for g, c in zip(GROUPS, sns.color_palette("tab10")) if f in GROUPS[g]))
    if x.isna().any():
        ax.bar(len(rate), train_view.loc[x.isna(), "podium"].mean(), color="#7f8c8d")
        labels.append("NaN")
    ax.axhline(base_rate, ls="--", color="black", lw=0.8)
    step = max(1, len(labels) // 10)
    ax.set_xticks(range(0, len(labels), step)); ax.set_xticklabels(labels[::step], fontsize=7)
    ax.set_title(f, fontsize=10); ax.set_xlabel(xlabel, fontsize=8); ax.set_ylim(0, 1)
for ax in axes.ravel()[len(FEATURES):]:
    ax.axis("off")
axes[0, 0].set_ylabel("Podium rate")
fig.suptitle(f"Podium rate by feature value, training seasons 2003–2019 (dashed line = base rate {base_rate:.1%}; "
             "grey bar = rows where the feature is missing)", y=1.0)
plt.tight_layout(); savefig("p4_podium_rate_by_feature"); plt.show()

**How to read the chart:** one panel per feature. Each bar is the podium rate of the training rows with that value
(or in that value group), and the dashed line is the average podium rate (14.1%). A feature carries signal when the
bars climb or fall steeply away from the dashed line. The grey bar shows rows where the feature is missing.

**What it shows:**
- **Grid and qualifying are the steepest.** From grid P1, 80% reach the podium; from P2 63%, P3 51%, P7 10%, and
  beyond P10 only 1%. Drivers within 0.3% of the fastest qualifying lap reach the podium 66% of the time.
- **Form and standings fall or rise in a clear line.** For example, a team that took podiums with more than half its
  cars in the last five races reaches the podium 58% of the time, against 3% for a team at 10% or less.
- **Reliability matters:** teams with at most 5% mechanical failures in their last 10 races reach the podium 20% of
  the time; above 25% failures, only 3%.
- **Experience is not a straight line:** `drv_prior_races` climbs to 35% (166–216 earlier races) and then drops to 15%
  for the longest careers, which include veterans past their best. Tree models can follow a bend like that.
- **Missing is information.** A debut (no driver form) reaches the podium only 2.7% of the time, so the
  missing-indicator columns in 4.7 keep this signal instead of hiding it.
- **Weak alone:** `round` is flat; `home_race` is 12.5% vs 14.2% (matching the Q2 result); larger fields podium a
  little less (16.7% with 18 cars, 12.5% with 24, since three podium places are shared by more cars). None of the
  29 training rows without a grid slot reached the podium. `grid_minus_quali` shows that drivers who start where they
  qualified reach the podium 17% of the time, while drivers moved by a penalty or a promotion do much worse.

In [ ]:
from sklearn.metrics import roc_auc_score

rows = []
for f in FEATURES:
    ok = train_view[f].notna()
    auc = roc_auc_score(train_view.loc[ok, "podium"], train_view.loc[ok, f])
    rows.append({"feature": f, "group": next(g for g, fs in GROUPS.items() if f in fs), "AUC": max(auc, 1 - auc),
                 "direction": "higher value → more podiums" if auc >= 0.5 else "lower value → more podiums",
                 "rows with a value": f"{ok.mean():.0%}"})
uni_auc = pd.DataFrame(rows).sort_values("AUC", ascending=False).reset_index(drop=True)

palette = dict(zip(GROUPS, sns.color_palette("tab10")))
fig, ax = plt.subplots(figsize=(10, 6))
d = uni_auc.iloc[::-1]
ax.barh(d["feature"], d["AUC"], color=[palette[g] for g in d["group"]])
ax.axvline(0.5, ls="--", color="black", lw=0.8)
ax.set_xlim(0.45, 1); ax.set_xlabel("Univariate ROC-AUC on the training seasons (0.5 = no signal)")
ax.set_title("How well does each feature alone separate podium finishers?")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in palette.values()], labels=list(palette), fontsize=8)
plt.tight_layout(); savefig("p4_univariate_auc"); plt.show()
uni_auc.round(3)

**How to read the chart:** for each feature on its own, ROC-AUC measures how well it separates podium rows from the rest.
0.5 means no better than a coin flip (the dashed line), 1.0 means perfect separation. We report max(AUC, 1 − AUC)
and write the direction in a separate column, so "lower grid = more podiums" also scores high.

**What it shows:** qualifying position (0.910), grid (0.909), gap to pole (0.893) and championship position (0.893)
lead. The form and points features follow at 0.86–0.88. Experience (0.690) and team reliability (0.644) carry some
signal. `grid_minus_quali` (0.554), `field_size` (0.523), `home_race` (0.503), `no_grid_slot` (0.502) and `round`
(0.501) carry almost none on their own.

**Decision: keep all 18 features.** The strong ones are obvious. The weak ones are kept on purpose, because a feature
with no signal alone can still help in combination. `no_grid_slot` tells the model that a grid value of 20 is a
filled value, not a real slot. `grid_minus_quali` marks a fast car starting out of position. They are cheap and
cannot leak. The Phase 6 ablation then measures whether the context group adds anything, and we report it honestly
if it does not.

In [ ]:
corr = train_view[FEATURES].corr(method="spearman")
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, cmap="RdBu_r", center=0, vmin=-1, vmax=1, annot=True, fmt=".1f", annot_kws={"size": 7}, ax=ax,
            cbar_kws={"label": "Spearman correlation"})
ax.set_title("Spearman correlation between features, training seasons 2003–2019")
plt.tight_layout(); savefig("p4_feature_correlation"); plt.show()

pairs = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1)).stack().rename("rho").reset_index()
         .rename(columns={"level_0": "feature A", "level_1": "feature B"}))
pairs.loc[pairs["rho"].abs() >= 0.7].sort_values("rho", key=abs, ascending=False).round(2).reset_index(drop=True)

**How to read the heatmap:** each square is the Spearman correlation between two features. Red means they rise
together, blue means one rises as the other falls. A value near ±1 means the two carry almost the same information.
The table lists every pair with |ρ| ≥ 0.7.

**What it shows:** grid and qualifying position are nearly the same thing (ρ = 0.97), and both are closely tied to
the gap to pole (0.86–0.88). Championship position and points share are mirror images (−0.92 to −0.95). Driver
form and team form agree strongly (0.87–0.90), because teammates share a car.

**Prediction, written down now and tested in Phase 6:** grid, `quali_pos` and `gap_to_pole_pct` overlap so much that
removing only the grid group or only the qualifying group should cost little, while removing both should cost a lot.
Overlap like this does not break tree models. For logistic regression, the L2 penalty spreads the weight across the
overlapping features.

In [ ]:
miss = train_view[FEATURES].isna().mean()
miss = miss[miss > 0].sort_values()
fig, ax = plt.subplots(figsize=(9, 3))
ax.barh(miss.index, miss.values, color="#7f8c8d")
for y, v in enumerate(miss.values):
    ax.text(v + 0.002, y, f"{v:.1%}", va="center", fontsize=8)
ax.set_xlabel("Share of training rows where the feature is missing")
ax.set_title("Features still missing after the domain fills (training seasons)")
plt.tight_layout(); savefig("p4_missingness"); plt.show()
print("Features with no missing values in training:", [f for f in FEATURES if f not in miss.index])

**What it shows:** after the domain fills, seven features still have gaps, all in the training seasons' early rounds and
debuts:
- `drv_pos_before` (7.2%) and `con_pos_before` (5.8%): round 1 of every season (362 training rows) plus drivers and
  teams new in mid-season;
- driver form (1.1%): debuts;
- team form (0.5%): new teams, and teams whose ID changed after a rename.

These seven get a **statistical** fill inside the training pipeline (4.7). Every other feature is complete.

### 4.6 Train / validation / test split
| Split | Seasons | Used for |
|---|---|---|
| Train | 2003–2019 | fitting the models |
| Validation | 2020–2021 | choosing the final model and its decision threshold |
| Test | 2022–2024 | reporting once, at the very end |

**Why split by time:** in real use we predict next weekend's race from past races. A random split would put races from
the same season into both train and test, so the model would "know" a team's 2023 form while predicting 2023.

**Why these boundaries:** the three most recent seasons form the test set. 2022 brought new car rules, so the test set
also checks how the model copes with a rule change. Validation takes the two seasons just before it, so the final
model is chosen on the seasons closest to the test period.

**A known weakness, and how Phase 5 handles it:** validation is small (39 races, 117 podiums), and 2020 was a shortened
season. One validation score can move by a few points of PR-AUC by chance. So Phase 5 does **not** tune hyperparameters
on it. Hyperparameters are tuned with **season-based cross-validation inside the training seasons** (each fold trains
on earlier seasons and scores the next one), and 2020–2021 is used only for the final model choice and the threshold.

In [ ]:
SPLITS = {"train": (2003, 2019), "validation": (2020, 2021), "test": (2022, 2024)}
model_df["split"] = pd.cut(model_df["year"], [2002, 2019, 2021, 2024], labels=list(SPLITS))
X_train, X_val, X_test = (model_df.loc[model_df["split"] == s, FEATURES] for s in SPLITS)
y_train, y_val, y_test = (model_df.loc[model_df["split"] == s, "podium"] for s in SPLITS)

split_table = (model_df.groupby("split", observed=True)
               .agg(first_season=("year", "min"), last_season=("year", "max"), races=("raceId", "nunique"),
                    rows=("podium", "size"), podiums=("podium", "sum"), podium_rate=("podium", "mean")))
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
colors = ["#2980b9", "#e67e22", "#c0392b"]
ax[0].bar(split_table.index.astype(str), split_table["rows"], color=colors)
for i, v in enumerate(split_table["rows"]):
    ax[0].text(i, v + 80, f"{v:,}", ha="center", fontsize=9)
ax[0].set_title("Rows per split"); ax[0].set_ylabel("Driver-race rows"); ax[0].set_ylim(0, split_table["rows"].max() * 1.15)
ax[1].bar(split_table.index.astype(str), split_table["podium_rate"], color=colors)
ax[1].axhline(3 / 20, ls="--", color="black", lw=0.8); ax[1].text(2.45, 3 / 20 + 0.004, "3 of 20", fontsize=8, ha="right")
ax[1].set_title("Podium rate per split"); ax[1].set_ylabel("Share of rows with a podium"); ax[1].set_ylim(0, 0.2)
plt.tight_layout(); savefig("p4_split"); plt.show()
split_table.round(3)

**Result:** train 6,831 rows (321 races), validation 780 rows (39 races), test 1,359 rows (68 races).

**How to read the chart:** left, the number of rows per split; right, the podium rate per split, with the dashed line
at 3 of 20 (15%). The rates are 14.1%, 15.0% and 15.0%. Training is a little lower because several training seasons had 22 or 24
cars sharing the same 3 podium places. The splits are disjoint, and together they hold every row of `model_df` (checked in 4.9).

### 4.7 Preprocessing and its visible effect
Two kinds of filling, done in two different places:

| Kind | Examples | Where | Why there |
|---|---|---|---|
| **Domain fills** | no grid slot → back of the grid; no lap time → slowest gap of that race; round 1 → 0 points | inside `build_features`, before the split | They use only information from the same race, which exists at prediction time. No statistic from other rows is involved, so nothing can leak across splits. |
| **Statistical fills** | missing form or standings → training median, plus a missing-indicator column | inside an sklearn `Pipeline`, fitted on train only | A median is a statistic of many rows. Computed on all seasons, it would carry validation and test information into training. |

First, the domain fills. `build_features(..., fill=False)` gives the table before they are applied:

In [ ]:
raw_df = build_features(CLEAN["results"], CLEAN["driver_standings"], CLEAN["constructor_standings"], fill=False)
filled = ["grid", "quali_pos", "gap_to_pole_pct", "grid_minus_quali", "drv_pts_share_before", "con_pts_share_before"]
fill_effect = pd.DataFrame({"missing before domain fill": raw_df[filled].isna().sum(),
                            "missing after": model_df[filled].isna().sum()})

fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
fill_effect.plot.barh(ax=ax[0], color=["#7f8c8d", "#27ae60"])
ax[0].set_xlabel("Rows (2003–2024)"); ax[0].set_title("Missing values before and after the domain fills")
bins = np.arange(0.5, 25.5)
ax[1].hist([raw_df["grid"].dropna(), model_df.loc[model_df["no_grid_slot"] == 1, "grid"]], bins=bins, stacked=True,
           color=["#7f8c8d", "#c0392b"], label=["real grid slots", "filled: no grid slot → field size"])
ax[1].set_xlabel("Grid slot"); ax[1].set_ylabel("Rows"); ax[1].legend(fontsize=8)
ax[1].set_title("grid: where the filled rows go")
plt.tight_layout(); savefig("p4_domain_fills"); plt.show()
fill_effect

**What it shows:** the domain fills close every gap they are meant to close: `grid` 86 → 0, `quali_pos` 24 → 0,
`gap_to_pole_pct` 148 → 0, and the points shares 606 and 494 → 0 (round 1 and new entries now have 0 points, which is
the true value). `grid_minus_quali` (107) has no gaps left because it is computed from the filled grid and qualifying
values.

**How to read the right chart:** grey bars are real grid slots; red pieces are the rows that had no slot and were moved
to the back of the grid. Most fields have 20 cars, so most of them land on 20. The flag `no_grid_slot` keeps them
recognisable.

Next, the **statistical fill**, which lives in two pipeline templates that Phase 5 fits on the training data:
- `prep_scaled` = median imputer with missing indicators, then `StandardScaler` (for logistic regression and the MLP);
- `prep_tree` = median imputer with missing indicators only (for random forest and XGBoost).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone

prep_scaled = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True)),
                        ("scale", StandardScaler())])                                     # for logistic regression and the MLP
prep_tree = Pipeline([("impute", SimpleImputer(strategy="median", add_indicator=True))])  # for random forest and XGBoost

imputer = clone(prep_tree).fit(X_train)                     # fit on the training seasons only
out_names = imputer.get_feature_names_out(FEATURES)
Xtr_imputed = pd.DataFrame(imputer.transform(X_train), columns=out_names, index=X_train.index)
medians = pd.Series(imputer[0].statistics_, index=FEATURES)
print(f"Columns after imputation: {len(out_names)} = {len(FEATURES)} features + "
      f"{len(out_names) - len(FEATURES)} missing-indicator columns:")
print([n.replace("missingindicator_", "") for n in out_names if n.startswith("missingindicator_")])

f = "drv_pos_before"                       # the feature with the most missing values (round 1, new drivers)
bins = np.arange(0.5, X_train[f].max() + 1.5)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4), sharey=True)
ax[0].hist(X_train[f].dropna(), bins=bins, color="#2980b9")
ax[0].set_title(f"{f}, before: {X_train[f].isna().sum()} rows missing")
ax[1].hist(Xtr_imputed[f], bins=bins, color="#2980b9")
ax[1].axvline(medians[f], color="#c0392b", ls="--", label=f"training median = {medians[f]:.0f}")
ax[1].set_title("after median imputation: a spike at the median"); ax[1].legend(fontsize=8)
for a in ax:
    a.set_xlabel("Championship position after the previous round")
ax[0].set_ylabel("Training rows")
plt.tight_layout(); savefig("p4_imputation"); plt.show()
medians[X_train.isna().any()].round(3).rename("training median used for missing values").to_frame()

**What it shows:** the imputer adds **7 missing-indicator columns**, one per feature that has gaps in training, so the
model sees 25 columns. The medians come from the training seasons only (for example, 11th place for `drv_pos_before`).

**How to read the chart:** left, championship positions before imputation, with 494 rows missing; right, after
imputation, the same rows all sit at the training median, 11, which creates the tall spike. That spike is exactly why
the indicator column matters. Without it, the model would think 494 drivers were 11th in the championship. With it,
the model knows that "no standings yet" (mostly round 1) is a different situation.

**Why scale for logistic regression and the MLP but not for trees:** logistic regression's penalty treats all
coefficients alike, so features must share a scale. Gradient descent in the MLP trains badly when one input runs
0–325 and another 0–1. Trees split on the *order* of values, so scaling changes nothing for them (shown in 4.8).

In [ ]:
scaler_fit = clone(prep_scaled).fit(X_train)
Xtr_scaled = pd.DataFrame(scaler_fit.transform(X_train), columns=scaler_fit.get_feature_names_out(FEATURES),
                          index=X_train.index)
scale_effect = pd.DataFrame({"mean before": Xtr_imputed[FEATURES].mean(), "std before": Xtr_imputed[FEATURES].std(),
                             "min before": Xtr_imputed[FEATURES].min(), "max before": Xtr_imputed[FEATURES].max(),
                             "mean after": Xtr_scaled[FEATURES].mean(), "std after": Xtr_scaled[FEATURES].std()})

show = ["grid", "gap_to_pole_pct", "drv_prior_races", "con_pts_share_before"]
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for a, data, title, xlabel in [(ax[0], Xtr_imputed, "Before scaling: very different ranges", "Raw value"),
                               (ax[1], Xtr_scaled, "After StandardScaler: mean 0, standard deviation 1", "Standardised value")]:
    long = data[show].melt(var_name="feature", value_name="value")
    sns.boxplot(data=long, x="value", y="feature", ax=a, color="#5dade2", fliersize=2)
    a.set_title(title); a.set_xlabel(xlabel); a.set_ylabel("")
plt.tight_layout(); savefig("p4_scaling"); plt.show()
scale_effect.round(2)

**What it shows:** before scaling, the features live on very different ranges: `drv_prior_races` runs 0–325, the
points shares 0–1, the gap to pole 0–38%. After `StandardScaler`, every feature has mean 0 and standard deviation 1.

**How to read the boxplots:** each box covers the middle half of the training values and the whiskers cover the rest.
After scaling, the boxes sit on one common scale, but their shapes are unchanged: the long right tail of
`gap_to_pole_pct` (wet or disrupted qualifying sessions) is still a long tail. Scaling moves and stretches the
values; it does not reshape them.

### 4.8 Preprocessing-order experiments
Two questions, each answered by trying both options:
- **E1: fit the imputer and scaler before or after the split?** *Leaky:* fit `prep_scaled` on all seasons
  2003–2024, then train on train. *Correct:* fit it on train only.
- **E2: impute then scale, or scale then impute?** *A:* imputer → scaler (our choice). *B:* scaler → imputer
  (the scaler ignores missing values when fitting, and the imputer then fills in scaled units).
- **E3 (a leakage demonstration, not an ordering question):** what happens to the score when features *do* see the
  race they predict.

**Test model:** logistic regression with C = 1 and balanced class weights. It is deterministic, so any difference comes
from the preprocessing and not from a random start, and it is sensitive to scaling, which makes it the right model to
test this on.

**Scored on validation only.** The test seasons stay untouched until the final model in Phase 5, so no choice in this
notebook is ever made by looking at test results. F1 uses the threshold 0.5, which is a fair default with balanced
class weights.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score


def lr_val_scores(Xtr, Xva):
    """The experiment model: logistic regression, C = 1, balanced class weights. Fit on train, score on validation."""
    lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000).fit(Xtr, y_train)
    p = lr.predict_proba(Xva)[:, 1]
    return {"val PR-AUC": average_precision_score(y_val, p), "val ROC-AUC": roc_auc_score(y_val, p),
            "val F1 (threshold 0.5)": f1_score(y_val, (p >= 0.5).astype(int))}


experiments = {}

The helper above fits the test model on a given training matrix and scores the validation matrix. Each variant now
runs in its own cell.

**E1, leaky version:**

In [ ]:
X_all = pd.concat([X_train, X_val, X_test])               # feature values of every season (no labels)
leaky = clone(prep_scaled).fit(X_all)                      # WRONG on purpose: statistics see validation and test seasons
experiments["E1 leaky: fit on 2003–2024"] = lr_val_scores(leaky.transform(X_train), leaky.transform(X_val))
pd.Series(experiments["E1 leaky: fit on 2003–2024"]).round(4).to_frame("E1 leaky")

**E1, correct version** (the imputer and scaler only ever see the training seasons). This is also variant A of E2:

In [ ]:
correct = clone(prep_scaled).fit(X_train)                  # the version we use: statistics from the training seasons only
experiments["E1 correct / E2 A: impute → scale, fit on train"] = lr_val_scores(correct.transform(X_train),
                                                                            correct.transform(X_val))
stat_compare = pd.DataFrame({"median, train only": correct[0].statistics_, "median, all seasons": leaky[0].statistics_,
                             "mean, train only": correct[1].mean_[:len(FEATURES)],
                             "mean, all seasons": leaky[1].mean_[:len(FEATURES)]}, index=FEATURES)
print("How much the fitted statistics move when validation and test seasons are included:")
stat_compare.loc[X_train.isna().any() | stat_compare.index.isin(["grid", "gap_to_pole_pct"])].round(3)

**How to read the table:** it compares the statistics each version learns. Most medians are identical. Only the
gap to pole (1.847 vs 1.752) and team form (11.5 vs 11.4) move, and the means shift by at most about 0.16. That
already tells us the E1 difference in scores will be small: the statistics hardly change between seasons.

**E2, variant B:** scale first, then impute.

In [ ]:
scale_first = Pipeline([("scale", StandardScaler()),
                        ("impute", SimpleImputer(strategy="median", add_indicator=True))]).fit(X_train)
experiments["E2 B: scale → impute, fit on train"] = lr_val_scores(scale_first.transform(X_train),
                                                                  scale_first.transform(X_val))

# Trees split on the ORDER of values. Both orders apply one increasing straight-line map per column, so a tree
# grown with the same seed makes exactly the same predictions either way.
from sklearn.tree import DecisionTreeClassifier
tree_a = DecisionTreeClassifier(max_depth=6, random_state=SEED).fit(correct.transform(X_train), y_train)
tree_b = DecisionTreeClassifier(max_depth=6, random_state=SEED).fit(scale_first.transform(X_train), y_train)
same = np.array_equal(tree_a.predict_proba(correct.transform(X_val)), tree_b.predict_proba(scale_first.transform(X_val)))
print("Decision tree: identical validation predictions under both orders?", same)
pd.Series(experiments["E2 B: scale → impute, fit on train"]).round(4).to_frame("E2 B")

**Trees do not care about the order:** the decision tree gives identical validation predictions under both orders
(`True`). Both orders apply one increasing straight-line transformation per column, and trees only use the order
of values, so random forest and XGBoost are unaffected by E2.

In [ ]:
exp_table = pd.DataFrame(experiments).T
ref = exp_table.loc["E1 correct / E2 A: impute → scale, fit on train"]
exp_table["Δ val PR-AUC vs correct"] = exp_table["val PR-AUC"] - ref["val PR-AUC"]

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.barh(exp_table.index[::-1], exp_table["val PR-AUC"][::-1], color=["#7f8c8d", "#27ae60", "#c0392b"])
for y, v in enumerate(exp_table["val PR-AUC"][::-1]):
    ax.text(v + 0.002, y, f"{v:.4f}", va="center", fontsize=9)
lo = exp_table["val PR-AUC"].min()
ax.set_xlim(lo - 0.02, exp_table["val PR-AUC"].max() + 0.02)
ax.set_xlabel("Validation PR-AUC (logistic regression, C = 1, balanced weights)")
ax.set_title("Preprocessing-order experiments")
plt.tight_layout(); savefig("p4_preprocessing_experiments"); plt.show()
exp_table.round(4)

**How to read the chart:** one bar per variant, showing validation PR-AUC. The axis is zoomed in, and the three bars
are still almost the same length.

**Results:**
- **E1:** the leaky version scored 0.7103, the correct version 0.7100, a difference of **+0.0004**. The difference is
  tiny because the medians and means barely move between seasons (see the table above).
- **E2:** scale → impute scored 0.7092, impute → scale 0.7100, a difference of **−0.0008**.

**Decisions:**
- **Fit preprocessing on train only (E1).** It is the only valid version: the leaky one lets validation and test
  seasons shape training. Notice that the leaky version scored marginally *higher*. That is exactly why leakage is
  tempting, because it can look better while being wrong. With stronger drift between seasons, such as the 2022 rule
  change, the gap could grow.
- **Impute, then scale (E2, variant A).** Then the scaler describes exactly the data the model is trained on, including
  the filled values. Variant B's 0.0008 lower score is far too small to matter, and for tree models the two orders are
  identical.

**E3: what real leakage looks like.** E1 shows that fitting the imputer on all seasons barely matters. That does
*not* mean leakage is harmless. The dangerous kind is a **feature** that sees the result it is predicting. We build two
deliberately broken versions of the table and train the same model on each:
- **E3a, form without the shift:** `past_mean` / `past_sum` without `shift(1)`, so "the last 5 races" includes the race
  being predicted.
- **E3b, standings after the race:** each race gets the standings published *after* it instead of after the previous
  round, so its own points are inside the feature.

Both use the correct preprocessing (fitted on train only) and are scored on validation, like E1 and E2.

In [ ]:
shifted_mean, shifted_sum = past_mean, past_sum
order = ["raceId", "driverId"]


def leaky_scores(df):
    assert df[order].reset_index(drop=True).equals(model_df[order].reset_index(drop=True)), "rows must align"
    Xtr = df.loc[df["year"] <= 2019, FEATURES]
    Xva = df.loc[df["year"].between(2020, 2021), FEATURES]
    prep = clone(prep_scaled).fit(Xtr)
    return lr_val_scores(prep.transform(Xtr), prep.transform(Xva))


# E3a: the rolling windows include the current race (no shift). WRONG on purpose.
def past_mean(s, k):
    return s.rolling(k, min_periods=1).mean()


def past_sum(s, k):
    return s.rolling(k, min_periods=1).sum()


try:
    leak_form = build_features(CLEAN["results"], CLEAN["driver_standings"], CLEAN["constructor_standings"])
finally:
    past_mean, past_sum = shifted_mean, shifted_sum          # restore the correct versions
assert past_mean is shifted_mean and past_sum is shifted_sum

# E3b: relabel the standings after race r as if they were published after round r - 1, so race r sees its own points.
r = CLEAN["races"][["raceId", "year", "round"]]
to_prev = r.merge(r.assign(round=r["round"] + 1), on=["year", "round"], suffixes=("", "_next"))
relabel = dict(zip(to_prev["raceId_next"], to_prev["raceId"]))      # race r -> the race before it


def standings_after_this_race(table):
    t = table[table["raceId"].isin(relabel)].copy()
    t["raceId"] = t["raceId"].map(relabel)
    return t


leak_standings = build_features(CLEAN["results"], standings_after_this_race(CLEAN["driver_standings"]),
                                standings_after_this_race(CLEAN["constructor_standings"]))

experiments["E3a leaky features: form includes this race"] = leaky_scores(leak_form)
experiments["E3b leaky features: standings after this race"] = leaky_scores(leak_standings)
e3_table = pd.DataFrame(experiments).T.loc[["E1 correct / E2 A: impute → scale, fit on train",
                                             "E3a leaky features: form includes this race",
                                             "E3b leaky features: standings after this race"]]
e3_table["Δ val PR-AUC vs correct"] = e3_table["val PR-AUC"] - e3_table.iloc[0]["val PR-AUC"]
moved = (leak_standings["drv_pts_share_before"] - model_df["drv_pts_share_before"]).abs().mean()
print(f"E3b: on average the leaked points share differs from the correct one by only {moved:.3f}")

fig, ax = plt.subplots(figsize=(10, 2.8))
ax.barh(e3_table.index[::-1], e3_table["val PR-AUC"][::-1], color=["#c0392b", "#c0392b", "#27ae60"])
for y, v in enumerate(e3_table["val PR-AUC"][::-1]):
    ax.text(v + 0.005, y, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlim(0, 1.05); ax.set_xlabel("Validation PR-AUC (same model, same preprocessing)")
ax.set_title("E3: a leak can inflate the score (E3a) or stay hidden in it (E3b)")
plt.tight_layout(); savefig("p4_leakage_demo"); plt.show()
e3_table.round(4)

**What E3 shows:**
- **E3a: the score jumps from 0.710 to 0.847 PR-AUC (+0.137).** Letting the form features include the race being
  predicted makes the model look far better than it is. In real use those values do not exist yet before the race,
  so this model would fail on race day. This is why `past_mean` and `past_sum` shift by one race before averaging.
- **E3b: the score barely moves (0.703, −0.007).** The leak is real, since each race now sees its own points, but it is
  hidden inside a season total. One race adds at most 25 points to a total of hundreds, so the points share changes
  by only 0.03 on average (printed above). A model cannot use such
  a small signal, but the feature is still wrong.
- **The lesson:** some leaks inflate the score (E3a) and some stay invisible in it (E3b). So a good score can never prove
  that a table is leakage-free. That is why Section 4.4 tests the **features themselves**: it damages a race's result
  and checks that the race's own features do not change.

### 4.9 Phase 4 gate
The run stops here if any check fails: duplicate driver-race rows, a season outside 2003–2024, a post-race feature, a
domain-filled feature with gaps, overlapping or incomplete splits, standings features in round 1 that are not empty or
zero, a podium rate per split far from 3 in 20, a failed leakage test, or any change to `CLEAN`.

In [ ]:
domain_filled = ["grid", "no_grid_slot", "field_size", "quali_pos", "gap_to_pole_pct", "grid_minus_quali",
                 "drv_prior_races", "drv_pts_share_before", "con_pts_share_before", "home_race", "round"]
assert not model_df.duplicated(["raceId", "driverId"]).any()
assert model_df["year"].between(2003, 2024).all()
assert not set(FEATURES) & set(POST_RACE)
assert model_df[domain_filled].notna().all().all(), "a domain-filled feature still has missing values"
assert set(X_train.index).isdisjoint(X_val.index) and set(X_val.index).isdisjoint(X_test.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_val) + len(X_test) == len(model_df)
assert X_train.index.equals(y_train.index) and X_val.index.equals(y_val.index) and X_test.index.equals(y_test.index)
first = model_df["round"] == 1
assert model_df.loc[first, ["drv_pts_share_before", "con_pts_share_before"]].eq(0).all().all()
assert model_df.loc[first, ["drv_pos_before", "con_pos_before"]].isna().all().all()
for y in (y_train, y_val, y_test):
    assert 0.12 < y.mean() < 0.18                           # podium base rate near 3 of 20
assert LEAKAGE_TESTS_PASSED                                  # perturbation test and hand checks in 4.4
assert {t: int(pd.util.hash_pandas_object(df).sum()) for t, df in CLEAN.items()} == CLEAN_FINGERPRINT, "CLEAN changed"
print(f"Phase 4 gate passed: {len(model_df):,} unique driver-race rows, {len(FEATURES)} pre-race features, "
      f"split {len(X_train):,} / {len(X_val):,} / {len(X_test):,}, no leakage, CLEAN unchanged.")

**Outputs for Phase 5:** `model_df` (with a `split` column), `FEATURES`, `GROUPS`, `POST_RACE`, `X_train`, `X_val`,
`X_test`, `y_train`, `y_val`, `y_test`, the pipeline templates `prep_scaled` and `prep_tree` (unfitted; Phase 5 fits them
inside each model's pipeline on train only), `build_features()`, and the figures in `figures/`.

## 5. Models

**What the brief asks:** at least two statistical models and one shallow neural network, trained on the same table
and the same features; ROC-AUC, PR-AUC and F1 (plus precision and recall) for every model on train, validation and
test; thresholds and hyperparameters chosen without the test set; and a justified primary and secondary metric.

| Step | What happens |
|---|---|
| 5.1 | Metrics: which one decides, and why |
| 5.2 | A reference rule with no learning: "started in the top 3" |
| 5.3 | How hyperparameters are tuned: season-by-season cross-validation inside the training seasons |
| 5.4 | Model 1, logistic regression (the baseline) |
| 5.5 | Model 2, random forest |
| 5.6 | Model 3, shallow neural network |
| 5.7 | Turning probabilities into yes/no: a tuned threshold vs the three most likely drivers per race |
| 5.8 | Comparison on train, validation and the unseen test set |
| 5.9 | Phase 5 gate |

**The three models** are logistic regression (statistical, and the official baseline), random forest (statistical)
and a shallow neural network. All three use exactly the rows, features and split built in Phase 4.

### 5.1 Metrics
| Metric | What it measures | Role |
|---|---|---|
| **PR-AUC** | How well the model ranks podium finishers above everyone else, focusing on the podium class | **Primary** |
| **ROC-AUC** | How well the model separates the two classes overall | **Secondary** |
| F1, precision, recall | Quality of the yes/no decisions at the chosen decision rule (5.7) | Reported for every model |
| Accuracy | Share of all yes/no decisions that are correct | Reported, never used to decide (see below) |

**Why PR-AUC is primary:** only about 15% of rows are podiums. ROC-AUC also rewards the model for correctly ranking
the many easy "no podium" rows (cars starting 18th), so it looks high for almost any sensible model. PR-AUC only
looks at how precise the model is among the drivers it ranks as likely podium finishers, which is exactly the
useful part. For reference: a model that guesses at random gets ROC-AUC 0.5 but PR-AUC ≈ 0.15 (the podium rate).

**Why accuracy is reported but never used to decide:** predicting "no podium" for every driver is already 85% accurate and completely useless. We still report it for every model, as our TA asked, so the comparison in 5.8 shows the problem on real numbers.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, accuracy_score,
                             precision_recall_curve, roc_curve, confusion_matrix)
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
import time

RACE_IDS = {s: model_df.loc[model_df["split"] == s, "raceId"] for s in SPLITS}
XS = {"train": X_train, "validation": X_val, "test": X_test}
YS = {"train": y_train, "validation": y_val, "test": y_test}


def top3_per_race(scores, race_ids):
    """1 for the three highest scores of each race, 0 for everyone else."""
    s = pd.Series(np.asarray(scores), index=race_ids.index)
    return (s.groupby(race_ids).rank(ascending=False, method="first") <= 3).astype(int).to_numpy()


def all_metrics(y, scores, decisions):
    """The six metrics: two ranking metrics from the scores, four from the yes/no decisions."""
    return {"ROC-AUC": roc_auc_score(y, scores), "PR-AUC": average_precision_score(y, scores),
            "F1": f1_score(y, decisions), "precision": precision_score(y, decisions, zero_division=0),
            "recall": recall_score(y, decisions), "accuracy": accuracy_score(y, decisions)}


print("Podium rate per split (= PR-AUC of random guessing):",
      {s: round(float(y.mean()), 3) for s, y in YS.items()})

### 5.2 Reference rule: "started in the top 3 → podium"
Before training anything we ask how far the grid alone gets us. The rule needs no learning: the score is the grid
position (lower is better) and the decision is "grid 1, 2 or 3 → podium". Any model must beat this to be worth its
complexity. It is a **reference point**, not the baseline model: the brief requires the baseline to be a statistical
model or a network, which is logistic regression (5.4).

In [ ]:
reference = {}
for s in SPLITS:
    g = XS[s]["grid"]
    reference[s] = all_metrics(YS[s], -g, (g <= 3).astype(int))
reference = pd.DataFrame(reference).T
print("Reference rule 'grid ≤ 3 → podium' (no learning):")
reference.round(3)

Starting position alone already reaches a validation PR-AUC of about 0.6, which confirms the Phase 1 and 4 finding
that the grid is the single strongest signal. The models below have to add something on top of it.

### 5.3 How hyperparameters are tuned
Each model has settings that are not learned from the data (for example how deep a tree may grow). We choose them
**without the validation or test seasons**, using season-by-season cross-validation inside 2003–2019:

| Fold | Train on | Score on |
|---|---|---|
| 1 | 2003–2014 | 2015 |
| 2 | 2003–2015 | 2016 |
| 3 | 2003–2016 | 2017 |
| 4 | 2003–2017 | 2018 |
| 5 | 2003–2018 | 2019 |

Every fold trains only on seasons *before* the one it scores, exactly like real use. The imputer and scaler are
inside the model pipeline, so they are refitted on each fold's training seasons only. The setting with the best
**mean PR-AUC over the five folds** wins. Validation (2020–2021) is not touched here, because it is too small to tune
on reliably (Section 4.6); it is used for the decision rule (5.7) and the final comparison.

In [ ]:
train_years = model_df.loc[model_df["split"] == "train", "year"]
CV_FOLDS = [(np.where(train_years < y)[0], np.where(train_years == y)[0]) for y in range(2015, 2020)]


def tune(make_pipeline, grid):
    """Season-based CV inside the training seasons. Returns one row per setting, best first."""
    rows = []
    for params in grid:
        scores = []
        for tr, va in CV_FOLDS:
            pipe = make_pipeline(**params).fit(X_train.iloc[tr], y_train.iloc[tr])
            scores.append(average_precision_score(y_train.iloc[va], pipe.predict_proba(X_train.iloc[va])[:, 1]))
        rows.append({**params, "mean CV PR-AUC": np.mean(scores), "std": np.std(scores)})
    return pd.DataFrame(rows).sort_values("mean CV PR-AUC", ascending=False).reset_index(drop=True)


print("Folds (training rows → scored rows):",
      [(len(tr), len(va)) for tr, va in CV_FOLDS])

### 5.4 Model 1: logistic regression (the baseline)
**How it works:** it gives every feature a weight, adds up *weight × value* for a driver, and squeezes the total
through the S-shaped logistic function into a probability between 0 and 1. Training finds the weights that make the
training podiums most likely. The penalty `C` keeps the weights small, so the model does not overreact to noise
(smaller `C` = stronger penalty).

**Why it is the baseline:** it is the simplest reasonable statistical model, fast, and fully transparent: each weight
says how a feature pushes the probability up or down.

**Limitations:** it can only draw a straight-line boundary in the (scaled) features. It cannot learn bends, such as
experience helping up to a point and then dropping (Section 4.5), or interactions, such as "a good grid matters more
for a reliable team", unless we build them by hand. It needs scaled inputs.

**Settings tried:** penalty `C` ∈ {0.01, 0.1, 1, 10}, and with or without balanced class weights (balanced weights make
the rare podium rows count more during training).

In [ ]:
def make_lr(C, class_weight):
    cw = None if class_weight == "none" else class_weight
    return Pipeline([("prep", clone(prep_scaled)),
                     ("model", LogisticRegression(C=C, class_weight=cw, max_iter=5000))])


t0 = time.time()
lr_grid = [{"C": C, "class_weight": cw} for C in [0.01, 0.1, 1, 10] for cw in ["none", "balanced"]]
lr_cv = tune(make_lr, lr_grid)
print(f"Logistic regression: {len(lr_grid)} settings × {len(CV_FOLDS)} folds in {time.time() - t0:.0f}s")
lr_cv.round(4)

### 5.5 Model 2: random forest
**How it works:** a decision tree asks a chain of yes/no questions ("grid ≤ 3?", then "team podium rate above 0.5?")
and ends in a leaf holding a podium probability. One deep tree memorises the training data. A random forest grows
many trees (here 300), each on a random sample of the rows and with a random subset of features at each question,
and **averages their answers**. The mistakes of individual trees cancel out.

**Why it is worth trying next to the baseline:** trees can follow bends and interactions that logistic regression
cannot, and they need no scaling because they only compare values.

**Limitations:** much larger and slower than logistic regression; there is no single formula to read; its
probabilities tend to be pulled towards the middle; and deep trees can still overfit, which shows up as a near-perfect
training score.

**Settings tried:** `max_depth` (how many questions in a row a tree may ask) ∈ {6, 10, unlimited} and
`min_samples_leaf` (the minimum number of training rows in a leaf; larger = smoother) ∈ {5, 20, 50}.

In [ ]:
def make_rf(max_depth, min_samples_leaf):
    depth = None if max_depth == "unlimited" else int(max_depth)
    return Pipeline([("prep", clone(prep_tree)),
                     ("model", RandomForestClassifier(n_estimators=300, max_depth=depth,
                                                      min_samples_leaf=int(min_samples_leaf), max_features="sqrt",
                                                      n_jobs=-1, random_state=SEED))])


t0 = time.time()
rf_grid = [{"max_depth": d, "min_samples_leaf": m} for d in [6, 10, "unlimited"] for m in [5, 20, 50]]
rf_cv = tune(make_rf, rf_grid)
print(f"Random forest: {len(rf_grid)} settings × {len(CV_FOLDS)} folds in {time.time() - t0:.0f}s")
rf_cv.round(4)

### 5.6 Model 3: shallow neural network
**How it works:** the scaled features enter a hidden layer of neurons. Each neuron computes its own weighted sum of the
inputs and passes it through ReLU (negative values become 0), which lets the network combine features in non-linear
ways. An optional second, smaller hidden layer combines those again, and one output neuron turns the result into a
probability. Training adjusts all weights in small steps with the Adam optimiser, one pass over the data at a time.

**Why "shallow":** one or two hidden layers, as the brief asks. With 18 features and about 7,000 rows, a deep network
would have far more weights than the data can support.

**Limitations:** a black box (weights cannot be read like logistic regression's); results depend on the random
starting weights; it needs scaled inputs; and on small tabular data it rarely beats simpler models.

**Attempt 1: scikit-learn's built-in early stopping.** Training stops when the score on a random 10% of the training
rows stops improving. Settings tried: layer sizes (32), (32, 16), (64, 32) and the L2 penalty `alpha` ∈ {0.0001,
0.01, 0.1} (larger = smaller weights = smoother model).

In [ ]:
def make_mlp_earlystop(hidden_layer_sizes, alpha):
    return Pipeline([("prep", clone(prep_scaled)),
                     ("model", MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, alpha=alpha,
                                             early_stopping=True, validation_fraction=0.1, n_iter_no_change=20,
                                             max_iter=1000, random_state=SEED))])


t0 = time.time()
mlp_es_grid = [{"hidden_layer_sizes": h, "alpha": a} for h in [(32,), (32, 16), (64, 32)] for a in [1e-4, 1e-2, 1e-1]]
mlp_es_cv = tune(make_mlp_earlystop, mlp_es_grid)
print(f"Neural network, attempt 1: {len(mlp_es_grid)} settings × {len(CV_FOLDS)} folds in {time.time() - t0:.0f}s")
mlp_es_cv.round(4)

**Why attempt 1 falls short:** the best setting reaches a mean CV PR-AUC of only about 0.72, below logistic regression.
The cause is the stopping signal. scikit-learn's early stopping watches **accuracy** on the held-out 10%, and with only
15% podiums, accuracy stops improving very early (predicting "no podium" is already 85% accurate). So training ends
before the network has learned to rank the podium candidates.

**Attempt 2: no accuracy-based early stopping, a stronger penalty instead.** We switch early stopping off. Training
then runs until the *training loss* stops improving (by less than 0.0001 for 10 passes in a row), with a cap of 200
passes over the data, and overfitting is controlled by the penalty `alpha`, now with larger values {0.1, 1, 3}. Same
layer sizes. If a setting reaches the 200-pass cap, scikit-learn warns that it has not fully converged; we accept
that (the cap limits how long the network can memorise), so this one warning is silenced.

In [ ]:
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # the 200-pass cap is deliberate (see above)


def make_mlp(hidden_layer_sizes, alpha):
    return Pipeline([("prep", clone(prep_scaled)),
                     ("model", MLPClassifier(hidden_layer_sizes=hidden_layer_sizes, alpha=alpha,
                                             early_stopping=False, max_iter=200, random_state=SEED))])


t0 = time.time()
mlp_grid = [{"hidden_layer_sizes": h, "alpha": a} for h in [(32,), (32, 16), (64, 32)] for a in [0.1, 1.0, 3.0]]
mlp_cv = tune(make_mlp, mlp_grid)
print(f"Neural network, attempt 2: {len(mlp_grid)} settings × {len(CV_FOLDS)} folds in {time.time() - t0:.0f}s")
print(f"Best mean CV PR-AUC: attempt 1 {mlp_es_cv['mean CV PR-AUC'].iloc[0]:.4f} → attempt 2 "
      f"{mlp_cv['mean CV PR-AUC'].iloc[0]:.4f}")
mlp_cv.round(4)

**Result of the two attempts:** switching off accuracy-based early stopping raised the best mean CV PR-AUC from about
0.72 to about 0.75. In attempt 2 the stronger penalty (`alpha` = 1) wins for every layer size, which confirms that the
network needs to be held back from memorising rather than trained for longer. We keep attempt 2.

**The tuning results side by side** (the neural network shown with attempt 2, which is the one we keep). Each bar is
one setting, sorted from best to worst mean CV PR-AUC, with the spread over the five folds as error bars. The winning
setting of each model (left-most bar) is refitted on all of 2003–2019 in this cell.

In [ ]:
CV_RESULTS = {"Logistic regression": lr_cv, "Random forest": rf_cv, "Neural network": mlp_cv}
SHORT = {"C": "C", "class_weight": "", "max_depth": "depth", "min_samples_leaf": "leaf",
         "hidden_layer_sizes": "", "alpha": "α"}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, (name, cv) in zip(axes, CV_RESULTS.items()):
    labels = [" ".join(f"{SHORT[k]}={v}" if SHORT[k] else str(v) for k, v in r.items()
                       if k not in ("mean CV PR-AUC", "std")) for _, r in cv.iterrows()]
    ax.bar(range(len(cv)), cv["mean CV PR-AUC"], yerr=cv["std"], capsize=2,
           color=["#c0392b"] + ["#95a5a6"] * (len(cv) - 1))
    ax.set_xticks(range(len(cv))); ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.set_title(f"{name}: best = {cv['mean CV PR-AUC'].iloc[0]:.3f}")
    ax.set_ylim(0.6, 0.85)
axes[0].set_ylabel("Mean PR-AUC over 5 season folds")
fig.suptitle("Hyperparameter tuning by season-based cross-validation (2015–2019 folds)", y=1.02)
plt.tight_layout(); savefig("p5_tuning"); plt.show()

BEST_PARAMS = {name: {k: (v.item() if hasattr(v, "item") else v) for k, v in cv.iloc[0].items()
                      if k not in ("mean CV PR-AUC", "std")}            # .item(): numpy number → plain Python
               for name, cv in CV_RESULTS.items()}
MAKERS = {"Logistic regression": make_lr, "Random forest": make_rf, "Neural network": make_mlp}
FINAL_MODELS = {name: MAKERS[name](**BEST_PARAMS[name]).fit(X_train, y_train) for name in MAKERS}
PROBA = {name: {s: m.predict_proba(XS[s])[:, 1] for s in SPLITS} for name, m in FINAL_MODELS.items()}
pd.Series({k: str(v) for k, v in BEST_PARAMS.items()}, name="chosen setting (fitted on 2003–2019)").to_frame()

**Performance during training (the network).** The checklist asks how the model performs *during* training and
validation. Logistic regression and the random forest are fitted in one step, so for them the comparison in 5.8 shows
train vs validation directly. The network learns in passes over the data, so here we follow it pass by pass: the
training loss, and PR-AUC on train and validation after 1, 2, 3, ... passes. Validation is only watched here; nothing
is chosen from this curve.

In [ ]:
# Performance DURING training: refit the chosen network for k passes (same seed, so each refit repeats the first
# k passes of the full run exactly) and score train and validation after each k. Validation is only watched here.
final_nn = FINAL_MODELS["Neural network"][-1]
print(f"The chosen network stopped after {final_nn.n_iter_} passes (cap {final_nn.max_iter}): its training loss "
      f"improved by less than {final_nn.tol} for {final_nn.n_iter_no_change} passes in a row.")
curve_rows = []
for k in [1, 2, 3, 5, 10, 20, 30, 50, 70, final_nn.n_iter_]:
    params = {**BEST_PARAMS["Neural network"]}
    pipe = Pipeline([("prep", clone(prep_scaled)),
                     ("model", MLPClassifier(**params, early_stopping=False, max_iter=k, random_state=SEED))])
    pipe.fit(X_train, y_train)
    curve_rows.append({"passes": k,
                       "train PR-AUC": average_precision_score(y_train, pipe.predict_proba(X_train)[:, 1]),
                       "validation PR-AUC": average_precision_score(y_val, pipe.predict_proba(X_val)[:, 1])})
training_curve = pd.DataFrame(curve_rows).set_index("passes")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(range(1, len(final_nn.loss_curve_) + 1), final_nn.loss_curve_, color="#8e44ad")
ax[0].set_xlabel("Pass over the training data (epoch)"); ax[0].set_ylabel("Training loss (log-loss + penalty)")
ax[0].set_title("Neural network: training loss per pass")
ax[1].plot(training_curve.index, training_curve["train PR-AUC"], marker="o", color="#2980b9", label="train 2003–2019")
ax[1].plot(training_curve.index, training_curve["validation PR-AUC"], marker="o", color="#e67e22",
           label="validation 2020–2021")
ax[1].set_xlabel("Passes over the training data"); ax[1].set_ylabel("PR-AUC"); ax[1].legend()
ax[1].set_title("PR-AUC on train and validation during training")
plt.tight_layout(); savefig("p5_training_curve"); plt.show()
training_curve.round(4)

**How to read the charts:** left, the training loss the optimiser minimises, pass by pass; right, PR-AUC on the
training seasons (blue) and the validation seasons (orange) after a given number of passes.

**What they show:** the network stopped by itself after 97 passes, well before the 200-pass cap, because its training
loss had stopped improving. Most of the learning happens in the first 10 passes: validation PR-AUC rises from 0.645
after one pass to 0.714 after ten, and reaches 0.732 by pass 50. After that both curves are flat. Validation never
turns down while training continues, so the network is not overfitting; the penalty `alpha = 1` holds it back.
Validation sits above train throughout, as in 5.8, because 2020–2021 were easier seasons to predict than 2003–2019.

### 5.7 From probabilities to yes/no: the decision rule
The models output a probability per driver, but F1, precision and recall need a yes/no answer. We compare two rules
**on the validation seasons only**:
- **Rule A, tuned threshold:** "probability above *t* → podium", with *t* chosen to maximise F1 on validation.
- **Rule B, top 3 per race:** in every race, the three drivers with the highest probabilities are "podium". Every race
  has exactly three podium places, so this rule always predicts exactly three, and it has no number to tune.

Whichever rule gives the higher mean validation F1 across the three models is used for all of them, so the models are
compared under the same rule. The test set plays no part in this choice.

In [ ]:
def best_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
    return float(thr[np.argmax(f1)])


THRESHOLDS = {name: best_threshold(y_val, PROBA[name]["validation"]) for name in FINAL_MODELS}
rule_rows = []
for name in FINAL_MODELS:
    p = PROBA[name]["validation"]
    dec_a = (p >= THRESHOLDS[name]).astype(int)
    dec_b = top3_per_race(p, RACE_IDS["validation"])
    per_race_a = pd.Series(dec_a, index=RACE_IDS["validation"].index).groupby(RACE_IDS["validation"]).sum()
    rule_rows.append({"model": name, "threshold (rule A)": THRESHOLDS[name],
                      "F1 rule A": f1_score(y_val, dec_a), "F1 rule B": f1_score(y_val, dec_b),
                      "rule A: predicted podiums per race (min–max)": f"{per_race_a.min()}–{per_race_a.max()}"})
rule_table = pd.DataFrame(rule_rows).set_index("model")
DECISION_RULE = "B" if rule_table["F1 rule B"].mean() >= rule_table["F1 rule A"].mean() else "A"
print(f"Mean validation F1: rule A {rule_table['F1 rule A'].mean():.3f}, rule B {rule_table['F1 rule B'].mean():.3f} "
      f"→ rule {DECISION_RULE} is used for every model from here on.")
rule_table.round(3)

**Result: rule A, the tuned threshold.** It gives a mean validation F1 of 0.706 against 0.684 for the top-3 rule, so
rule A is used for all three models. The thresholds are low (0.25–0.35): a lower cut-off lets each model name more
podium candidates, which gains more in recall than it loses in precision. Rule A predicts between 2 and 5 podiums
per race instead of always 3, and that flexibility pays off: in races with a dominant front-runner it can say "only
two drivers are likely", and in open races it can name four or five candidates.

**One caution:** rule A's threshold was chosen on the validation seasons, so its validation F1 is slightly flattered;
rule B has nothing to tune. The test set is where both the models and the rule get an unbiased check.

### 5.8 Comparison on train, validation and the unseen test set
Every model, with its tuned setting and the chosen decision rule, scored on all three splits. The test seasons
(2022–2024) are used here for the first and only time; nothing is changed after seeing them.

**How the final model is chosen, fixed before looking at the test set:** the highest **validation PR-AUC**. If two models
are within 0.01 of each other, which is well inside the noise of a validation set with 117 podiums, we keep the
**simpler** one (logistic regression < random forest < neural network), because a model must earn its extra
complexity with a clear gain.

In [ ]:
def decide(name, s):
    p = PROBA[name][s]
    return top3_per_race(p, RACE_IDS[s]) if DECISION_RULE == "B" else (p >= THRESHOLDS[name]).astype(int)


rows = []
for s in SPLITS:
    for name in FINAL_MODELS:
        rows.append({"model": name, "split": s, **all_metrics(YS[s], PROBA[name][s], decide(name, s))})
    rows.append({"model": "Reference: grid ≤ 3", "split": s, **reference.loc[s].to_dict()})
comparison = pd.DataFrame(rows).set_index(["model", "split"])
order = ["Reference: grid ≤ 3", "Logistic regression", "Random forest", "Neural network"]
comparison = comparison.reindex(pd.MultiIndex.from_product([order, list(SPLITS)], names=["model", "split"]))
comparison.round(3)

**How to read the table:** one row per model and split. ROC-AUC and PR-AUC judge how well the probabilities rank the
drivers; F1, precision, recall and accuracy judge the yes/no decisions of the rule chosen in 5.7. The reference rows use
"grid ≤ 3" for both. Higher is better in every column.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
colors = {"Reference: grid ≤ 3": "#7f8c8d", "Logistic regression": "#2980b9", "Random forest": "#27ae60",
          "Neural network": "#8e44ad"}
for ax, metric in zip(axes, ["PR-AUC", "ROC-AUC"]):
    tab = comparison[metric].unstack("split")[list(SPLITS)].loc[order]
    x = np.arange(len(SPLITS)); w = 0.2
    for i, name in enumerate(order):
        ax.bar(x + (i - 1.5) * w, tab.loc[name], w, label=name, color=colors[name])
    ax.set_xticks(x); ax.set_xticklabels(list(SPLITS)); ax.set_title(metric)
    ax.set_ylim(0.5 if metric == "PR-AUC" else 0.8, 1.0)
axes[0].axhline(y_test.mean(), color="black", ls=":", lw=0.8)
axes[0].legend(fontsize=8, loc="upper right")
fig.suptitle("Ranking metrics per model and split (test = 2022–2024, unseen until now)", y=1.02)
plt.tight_layout(); savefig("p5_comparison"); plt.show()

**How to read the chart:** left PR-AUC, right ROC-AUC. For each split there are four bars: grey = the grid rule, blue =
logistic regression, green = random forest, purple = neural network. The dotted line in the left panel is the test
podium rate (0.15), which is what random guessing would score. Every model clears the grid rule on every split, and on
test the three models are within 0.01 of each other.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for name in FINAL_MODELS:
    prec, rec, _ = precision_recall_curve(y_test, PROBA[name]["test"])
    axes[0].plot(rec, prec, color=colors[name], label=f"{name} ({comparison.loc[(name, 'test'), 'PR-AUC']:.3f})")
    fpr, tpr, _ = roc_curve(y_test, PROBA[name]["test"])
    axes[1].plot(fpr, tpr, color=colors[name], label=f"{name} ({comparison.loc[(name, 'test'), 'ROC-AUC']:.3f})")
ref_t = reference.loc["test"]
axes[0].scatter(ref_t["recall"], ref_t["precision"], color="black", zorder=3, label="grid ≤ 3 (one point)")
axes[0].axhline(y_test.mean(), ls=":", color="grey", label=f"random guessing ({y_test.mean():.2f})")
axes[0].set_xlabel("Recall (share of real podiums found)"); axes[0].set_ylabel("Precision (share of predictions correct)")
axes[0].set_title("Precision-recall curves, test 2022–2024"); axes[0].legend(fontsize=7)
axes[1].plot([0, 1], [0, 1], ls=":", color="grey")
axes[1].set_xlabel("False-positive rate"); axes[1].set_ylabel("True-positive rate")
axes[1].set_title("ROC curves, test 2022–2024"); axes[1].legend(fontsize=7)

SIMPLICITY = ["Logistic regression", "Random forest", "Neural network"]
val_pr = comparison.xs("validation", level="split")["PR-AUC"].reindex(SIMPLICITY)
FINAL_MODEL_NAME = next(m for m in SIMPLICITY if val_pr[m] >= val_pr.max() - 0.01)   # simplest within 0.01 of the best
cm = confusion_matrix(y_test, decide(FINAL_MODEL_NAME, "test"))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2],
            xticklabels=["predicted: no podium", "predicted: podium"], yticklabels=["real: no podium", "real: podium"])
axes[2].set_title(f"{FINAL_MODEL_NAME} on test (rule {DECISION_RULE})")
plt.tight_layout(); savefig("p5_curves_confusion"); plt.show()
print("Validation PR-AUC:", val_pr.round(4).to_dict())
print(f"Final model (simplest within 0.01 of the best validation PR-AUC): {FINAL_MODEL_NAME}")

**How to read the charts:**
- *Left, precision-recall curves on test:* each point of a curve is one possible threshold; further up and to the right
  is better. The black dot is the grid rule (one fixed yes/no rule, so one point), and the dotted line is random
  guessing.
- *Middle, ROC curves:* the closer a curve runs to the top-left corner, the better.
- *Right, confusion matrix* of the final model on test, with the decision rule from 5.7. Rows are what really
  happened, columns what the model said. Of the 204 real podiums the network found 173 (bottom right) and missed 31
  (bottom left). It also named 140 drivers who did not reach the podium (top right): that is the price of the low
  threshold, which favours finding podiums over being right every time. 1,015 drivers were correctly given no podium.

**Does the model cope with the 2022 rule change?** The test set spans the first three seasons of the new cars. Scoring
each test season on its own shows whether performance holds up or drops in a particular year.

In [ ]:
test_rows = model_df.loc[model_df["split"] == "test"]
season_rows = []
for yr in sorted(test_rows["year"].unique()):
    m = (test_rows["year"] == yr).to_numpy()
    for name in FINAL_MODELS:
        season_rows.append({"season": yr, "model": name,
                            "PR-AUC": average_precision_score(y_test[m], PROBA[name]["test"][m])})
    season_rows.append({"season": yr, "model": "Reference: grid ≤ 3",
                        "PR-AUC": average_precision_score(y_test[m], -X_test["grid"][m])})
by_season = pd.DataFrame(season_rows).pivot(index="season", columns="model", values="PR-AUC")[order]
ax = by_season.plot.bar(figsize=(10, 3.4), color=[colors[c] for c in order], rot=0)
ax.set_ylim(0.4, 1); ax.set_ylabel("PR-AUC"); ax.set_title("Test PR-AUC per season"); ax.legend(fontsize=8)
plt.tight_layout(); savefig("p5_test_by_season"); plt.show()
by_season.round(3)

**What the comparison shows:**
- **All three models beat the reference rule on every split.** On the unseen test seasons, PR-AUC is 0.700–0.708 for
  the models against 0.647 for "grid ≤ 3", and ROC-AUC 0.936–0.938 against 0.914. So the features beyond the grid
  (qualifying gap, form, standings, reliability) add real information.
- **The three models are practically tied.** Test PR-AUC: logistic regression 0.707, random forest 0.700, neural network
  0.708. Differences below 0.01 are well inside the noise of 204 test podiums. The honest conclusion: on this table the
  choice of model matters much less than the features; the simple baseline is almost as good as the network.
- **Final model: the neural network**, by the rule fixed before the test (validation PR-AUC 0.732 vs 0.720 and 0.719, a
  margin just above the 0.01 tie-break). The test confirms it performs as expected, not that it is clearly better.
  In cross-validation the order was different (random forest 0.762, network 0.748, logistic regression 0.733): the
  ranking of the three flips from one sample of seasons to another, which is one more sign that they are tied.
- **Overfitting check:** the random forest scores much higher on its own training seasons (PR-AUC 0.774) than on
  validation (0.719), so it memorises some training detail even at depth 6. Logistic regression and the network score
  about the same on train and validation, a sign that they generalise. (Their training score is even slightly lower:
  the 2003–2019 seasons are harder to predict on average than 2020–2021, when a few teams dominated.)
- **Decisions (rule A) on test:** the network finds 173 of the 204 real podiums (recall 0.85), and 55% of its podium
  predictions are correct (precision 0.55). The low threshold trades precision for recall.
- **Accuracy is reported, but it would pick the wrong model.** On test: neural network 0.874, logistic regression
  0.876, random forest 0.895, and the grid rule also 0.895, while always saying "no podium" already scores 0.850.
  The grid rule ties the best accuracy although it ranks podium finishers far worse (PR-AUC 0.647 vs 0.70 and
  above). The network gives up a little accuracy to find many more real podiums (recall 0.85 vs 0.65 for the grid
  rule). This is why PR-AUC, not accuracy, decides.
- **Per season:** all models beat the reference in 2022, the first season of the new rules (about 0.67–0.69 vs 0.62),
  so the rule change did not break them, and they do best in 2023 (about 0.78), when Red Bull won 21 of 22 races. 2024
  is the hardest: it was the most open of the three seasons, with four teams winning races (Red Bull 9, McLaren 6,
  Ferrari 5, Mercedes 4), and logistic regression is only level with the grid rule there. Form built on past races helps less when the order changes during
  a season.

**Limitations of the models:** the network's result depends on its random starting weights (fixed with `SEED`, so the
notebook is reproducible); validation has only 117 podiums, so small validation gaps are not meaningful; and the
models predict each driver separately, so nothing forces the three probabilities of a race to add up to three
podiums.

### 5.9 Phase 5 gate
The run stops here if any check fails: the three models do not use the same rows and features, a metric is outside
[0, 1], rule B does not predict exactly three podiums in every race, the thresholds or settings were not chosen from
training and validation data, or `CLEAN` changed.

In [ ]:
assert set(FINAL_MODELS) == {"Logistic regression", "Random forest", "Neural network"}
for name, m in FINAL_MODELS.items():
    assert list(m[0].feature_names_in_) == FEATURES, f"{name} was trained on different features"
assert comparison.dropna().apply(lambda s: s.between(0, 1)).all().all()
for s in SPLITS:
    picks = pd.Series(top3_per_race(PROBA[FINAL_MODEL_NAME][s], RACE_IDS[s]), index=RACE_IDS[s].index)
    assert (picks.groupby(RACE_IDS[s]).sum() == 3).all(), "rule B must pick exactly 3 per race"
assert all(train_years.iloc[tr].max() < train_years.iloc[va].min() <= train_years.iloc[va].max() <= 2019
           for tr, va in CV_FOLDS)                  # each fold trains on earlier seasons and scores a training season
assert {t: int(pd.util.hash_pandas_object(df).sum()) for t, df in CLEAN.items()} == CLEAN_FINGERPRINT, "CLEAN changed"
print(f"Phase 5 gate passed: 3 models on {len(FEATURES)} features, tuned on 2003–2019 folds, decision rule "
      f"{DECISION_RULE} chosen on validation, final model = {FINAL_MODEL_NAME}, test used once.")

**Outputs for Phase 6:** `FINAL_MODELS` (the three fitted pipelines), `FINAL_MODEL_NAME`, `BEST_PARAMS`, `MAKERS`
(to refit a model during the ablation), `PROBA`, `DECISION_RULE`, `THRESHOLDS`, `comparison`, `top3_per_race()` and
`all_metrics()`.

## 6. Feature-group ablation and explainability (XAI)

**What the brief asks:** a feature-group ablation with results per group; global explanations (permutation importance
and SHAP); a local explanation of single predictions (LIME and/or local SHAP); everything plotted and interpreted; and
a clear statement that explanations are not causal effects.

| Step | What happens |
|---|---|
| 6.1 | Feature-group ablation: remove one group at a time, retrain, measure the change |
| 6.2 | Permutation importance of the final model |
| 6.3 | SHAP, global: which features push predictions up or down, and how |
| 6.4 | Two single predictions explained with a SHAP waterfall and with LIME |
| 6.5 | What these explanations do **not** show |
| 6.6 | Phase 6 gate |

**Which model is explained:** the final model chosen in Phase 5 (`FINAL_MODEL_NAME`, the neural network). The
ablation repeats every step for logistic regression, the baseline, to see whether the pattern depends on the model.

### 6.1 Feature-group ablation
**How:** for each of the six feature groups of Section 4.2, we drop its columns, refit the model with **the same tuned
setting** (no re-tuning), and score it. One extra variant drops **grid and qualifying together**, to test the
prediction written down in 4.5: the two groups carry almost the same information, so removing one should cost little
and removing both a lot.

**Three scores per variant:** the mean PR-AUC over the five season folds of 5.3 (the steadiest number, about 2,000
scored rows), validation PR-AUC and test PR-AUC. The ablation is a measurement, not a choice: the feature set stays as
it is whatever the result, so looking at the test seasons here changes no decision.

In [ ]:
ABLATIONS = {"all features": []}
ABLATIONS |= {f"without {g}": cols for g, cols in GROUPS.items()}
ABLATIONS["without grid + qualifying"] = GROUPS["grid"] + GROUPS["qualifying"]
ABLATION_MODELS = list(dict.fromkeys([FINAL_MODEL_NAME, "Logistic regression"]))   # final model + the baseline


def ablation_scores(name, drop):
    """Refit `name` with its tuned setting on the remaining features; score CV folds, validation and test."""
    keep = [f for f in FEATURES if f not in drop]
    cv = []
    for tr, va in CV_FOLDS:
        pipe = MAKERS[name](**BEST_PARAMS[name]).fit(X_train.iloc[tr][keep], y_train.iloc[tr])
        cv.append(average_precision_score(y_train.iloc[va], pipe.predict_proba(X_train.iloc[va][keep])[:, 1]))
    pipe = MAKERS[name](**BEST_PARAMS[name]).fit(X_train[keep], y_train)
    return {"features": len(keep), "CV PR-AUC": np.mean(cv),
            "val PR-AUC": average_precision_score(y_val, pipe.predict_proba(X_val[keep])[:, 1]),
            "test PR-AUC": average_precision_score(y_test, pipe.predict_proba(X_test[keep])[:, 1])}


t0 = time.time()
rows = [{"model": name, "variant": v, **ablation_scores(name, drop)} for name in ABLATION_MODELS
        for v, drop in ABLATIONS.items()]
ablation_table = pd.DataFrame(rows).set_index(["model", "variant"])
for col in ["CV PR-AUC", "val PR-AUC", "test PR-AUC"]:
    full = ablation_table.xs("all features", level="variant")[col]
    ablation_table[f"Δ {col.split()[0]}"] = ablation_table[col] - ablation_table.index.get_level_values("model").map(full)
print(f"{len(rows)} refits ({len(ABLATION_MODELS)} models × {len(ABLATIONS)} variants, each with 5 CV folds) "
      f"in {time.time() - t0:.0f}s")
ablation_table.round(4)

**How to read the table:** one row per variant and model. The `Δ` columns are the change against "all features":
negative means the model got worse without that group, so the group was helping.

In [ ]:
fig, axes = plt.subplots(1, len(ABLATION_MODELS), figsize=(7.5 * len(ABLATION_MODELS), 4.6), sharex=True)
axes = np.atleast_1d(axes)
variants = [v for v in ABLATIONS if v != "all features"]
for ax, name in zip(axes, ABLATION_MODELS):
    d = ablation_table.loc[name].loc[variants]
    y = np.arange(len(variants)); h = 0.27
    for i, (col, color) in enumerate([("Δ CV", "#2980b9"), ("Δ val", "#e67e22"), ("Δ test", "#c0392b")]):
        ax.barh(y + (i - 1) * h, d[col], h, color=color, label=col.replace("Δ ", "") + " PR-AUC change")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(variants); ax.invert_yaxis()
    ax.set_title(f"{name}: PR-AUC change when a group is removed")
    ax.set_xlabel("Change in PR-AUC vs all features")
axes[0].legend(fontsize=8, loc="upper left")
fig.suptitle("Feature-group ablation: bars left of zero mean the model got worse without that group", y=1.02)
plt.tight_layout(); savefig("p6_ablation"); plt.show()

**How to read the chart:** one panel per model, one group of three bars per variant (blue = CV, orange = validation,
red = test). Bars to the left of zero mean the group was helping.

**What the ablation shows:**
- **Grid and qualifying can replace each other.** Removing either group alone changes PR-AUC by about 0.01 or less
  (network, CV: −0.002 without grid, −0.006 without qualifying). Removing **both** drops it by 0.104 in CV, 0.091 on
  validation and 0.137 on test for the network, and by a similar 0.081, 0.090 and 0.139 for logistic regression.
  This confirms the prediction from 4.5: the model replaces grid with qualifying position and vice versa (ρ = 0.97),
  so together they are by far the most important information, but neither is needed on its own.
- **Standings** is the most useful single group in cross-validation (−0.010 for the network, −0.011 for logistic
  regression).
- **Driver form, constructor form and context** change PR-AUC by less than 0.01 either way, and the sign flips between
  CV, validation and test. Their information is already covered by grid, qualifying and standings (form and standings
  correlate at 0.8–0.9, Section 4.5). The weak context features (home race, round), which 4.5 kept for this test,
  add nothing measurable, and we report that rather than hide it.
- **Both models show the same pattern,** so it is a property of the data, not of one model.
- **Noise level:** changes smaller than about 0.01 are within the spread we see between CV, validation and test, so
  they should not be read as real gains or losses.

### 6.2 Permutation importance (global)
**How it works:** we take the test seasons, shuffle one feature's column so its values no longer belong to the right
drivers, and measure how much test PR-AUC drops. This is repeated 20 times with different shuffles, giving a mean and
a spread. A big drop means the model relies on that feature. It runs on the whole pipeline, so importance is reported
per original feature, not per missing-indicator column.

**Why on the test seasons:** nothing is decided from this result. It only describes what the final model relies on for
races it has never seen.

In [ ]:
from sklearn.inspection import permutation_importance

final_pipe = FINAL_MODELS[FINAL_MODEL_NAME]
t0 = time.time()
pi = permutation_importance(final_pipe, X_test, y_test, scoring="average_precision", n_repeats=20,
                            random_state=SEED)
perm = (pd.DataFrame({"group": [next(g for g, fs in GROUPS.items() if f in fs) for f in FEATURES],
                      "mean PR-AUC drop": pi.importances_mean, "std": pi.importances_std}, index=FEATURES)
        .sort_values("mean PR-AUC drop", ascending=False))
print(f"{FINAL_MODEL_NAME}, test PR-AUC with nothing shuffled: {comparison.loc[(FINAL_MODEL_NAME, 'test'), 'PR-AUC']:.3f} "
      f"({len(FEATURES)} features × 20 shuffles in {time.time() - t0:.0f}s)")

group_colors = dict(zip(GROUPS, sns.color_palette("tab10")))
fig, ax = plt.subplots(figsize=(10, 6))
d = perm.iloc[::-1]
ax.barh(d.index, d["mean PR-AUC drop"], xerr=d["std"], color=[group_colors[g] for g in d["group"]], capsize=2)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("Drop in test PR-AUC when the feature is shuffled (mean ± std over 20 shuffles)")
ax.set_title(f"Permutation importance, {FINAL_MODEL_NAME}, test 2022–2024")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in group_colors.values()], labels=list(group_colors),
          fontsize=8, loc="lower right")
plt.tight_layout(); savefig("p6_permutation_importance"); plt.show()
perm.round(4)

**How to read the chart:** each bar is the mean drop in test PR-AUC (0.708 with nothing shuffled), the whisker is the
spread over the 20 shuffles, and the colour is the feature group. A bar near zero, or a whisker that crosses zero,
means the model barely notices when that feature is scrambled.

**What it shows:** two features dominate, grid (−0.124) and qualifying position (−0.122). Next come the driver's recent
podium rate (−0.028), the points share (−0.014) and the gap to pole (−0.013). Every other feature is below 0.004, and a
few are slightly negative: shuffling them changes nothing real, so those small values are noise.

**Why does shuffling grid hurt so much when the ablation says grid can be removed?** The two methods answer different
questions. Permutation keeps the trained model and breaks grid, so the model suddenly sees impossible combinations
such as grid 18 with qualifying position 1, after being trained to use both. Ablation retrains without grid, and the
new model simply leans on qualifying position instead. So correlated features look important under permutation and
replaceable under ablation, and both statements are true.

### 6.3 SHAP (global)
**What a SHAP value is:** for one prediction, each feature gets its share of the gap between that prediction and the
average prediction. The shares are Shapley values from game theory: a feature's average contribution over all the
orders in which the features could be added. The base value plus all SHAP values of a row equals the model's predicted
probability for that row exactly (the cell checks this). Our values are in probability units: +0.10 means the feature
raised that driver's podium probability by 10 percentage points.

**How we compute it:** the final model is a neural network, which has no fast exact SHAP method like trees do, so we
use SHAP's model-agnostic *permutation* explainer. 100 training rows serve as the "average driver" background, and
every one of the 1,359 test rows is explained (about 1–2 minutes).

In [ ]:
try:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")                      # shap's import can print a harmless progress-bar notice
        import shap
except ImportError:                                          # on Kaggle: Settings → Internet on
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap"])
    import shap


def predict_podium_proba(X):
    """The final model as a plain function: 18 feature values in, podium probability out."""
    return final_pipe.predict_proba(pd.DataFrame(np.asarray(X, dtype=float), columns=FEATURES))[:, 1]


X_test_f = X_test.astype(float)                             # SHAP needs plain floats (two columns are nullable ints)
background = shap.utils.sample(X_train.astype(float), 100, random_state=SEED)
explainer = shap.Explainer(predict_podium_proba, shap.maskers.Independent(background.to_numpy()),
                           algorithm="permutation", feature_names=FEATURES, seed=SEED)
t0 = time.time()
sv = explainer(X_test_f.to_numpy(), silent=True)
gap = np.abs(sv.values.sum(axis=1) + sv.base_values - predict_podium_proba(X_test_f)).max()
print(f"SHAP values for all {len(X_test_f):,} test rows in {time.time() - t0:.0f}s. "
      f"Base value (average prediction on the background) = {sv.base_values[0]:.3f}. "
      f"Check: base + sum of SHAP values = model output (largest gap {gap:.1e}).")

shap.plots.beeswarm(sv, max_display=len(FEATURES), show=False)
plt.title(f"SHAP values, {FINAL_MODEL_NAME}, every test row (2022–2024)")
plt.gcf().set_size_inches(10, 7); plt.tight_layout(); savefig("p6_shap_beeswarm"); plt.show()

**How to read the beeswarm:** one row per feature, one dot per test row. A dot's horizontal position is its SHAP value
(right = pushes the podium probability up, left = down), and its colour is the feature's value (red = high,
blue = low). Features are sorted by importance.

**What it shows:**
- **`grid` and `quali_pos`:** low values (blue: P1, P2, ...) sit far right, up to +0.20, while high values (red: the
  back of the grid) sit a little left of zero. Starting near the front is the strongest push towards a podium;
  starting at the back costs less, because those drivers had a low probability anyway.
- **`drv_podium_rate_last5`, `drv_pts_share_before`, `con_pts_share_before`:** high values (red) push up. Recent
  podiums and a strong championship position raise the predicted probability.
- **`gap_to_pole_pct`:** small gaps (blue) push up, large gaps push down.
- The other features rarely move a prediction by more than a few points. Grey dots are rows where the value is
  missing.

In [ ]:
shap_rank = pd.Series(np.abs(sv.values).mean(axis=0), index=FEATURES, name="mean |SHAP|").sort_values(ascending=False)
shap.plots.bar(sv, max_display=len(FEATURES), show=False)
plt.title("Mean absolute SHAP value per feature (average push on the predicted probability)")
plt.gcf().set_size_inches(10, 6.5); plt.tight_layout(); savefig("p6_shap_bar"); plt.show()

rank_table = pd.DataFrame({"SHAP rank": shap_rank.rank(ascending=False).astype(int),
                           "mean |SHAP|": shap_rank,
                           "permutation rank": perm["mean PR-AUC drop"].rank(ascending=False).astype(int),
                           "permutation PR-AUC drop": perm["mean PR-AUC drop"]}).sort_values("SHAP rank")
rank_table.round(4)

**How to read:** each bar is the average size of a feature's push on the predicted probability, ignoring the direction.
The table puts the SHAP ranking next to the permutation ranking from 6.2.

**Do SHAP and permutation importance agree?** On the top four, yes: grid, `quali_pos`, `drv_podium_rate_last5` and
`drv_pts_share_before` rank 1–4 in both. Further down they differ, for example `con_pts_share_before` (SHAP 5th,
permutation 7th) and `con_avg_finish_last5` (SHAP 7th, permutation 14th). The reason is that they measure different
things. SHAP measures how much a feature *moves* the predictions; permutation measures how much *ranking quality* is
lost when it is scrambled. A team-strength feature can move many predictions a little while barely changing the order
of the drivers, and therefore PR-AUC, because teammates and other features (the driver's points share) carry the same
information.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
shap.plots.scatter(sv[:, "grid"], color=sv[:, "con_podium_rate_last5"], ax=ax[0], show=False)
ax[0].set_title("Effect of grid, coloured by the team's recent podium rate")
shap.plots.scatter(sv[:, "drv_pts_share_before"], color=sv[:, "grid"], ax=ax[1], show=False)
ax[1].set_title("Effect of the driver's points share, coloured by grid")
plt.tight_layout(); savefig("p6_shap_dependence"); plt.show()

**How to read the charts:** each dot is one test row. *Left:* x = grid slot, y = the SHAP value of grid, colour = the
team's podium rate over its last five races. *Right:* x = the driver's points share before the race, y = its SHAP
value, colour = grid slot.

**What they show:**
- **The grid effect is steep at the front and flat at the back.** On average, P1 adds +0.18, P2 +0.15, P3 +0.10 and P5
  +0.03, while from about P8 onward the value stays near −0.03 to −0.05 whatever the slot. Among front-row starters,
  the red dots (strong recent team form) sit higher than the blue ones: the model gives a good grid slot more weight
  when the team has been on the podium lately. A straight-line model cannot learn an interaction like this.
- **The points share pushes up almost in a straight line,** and more for drivers starting near the front (blue) than
  for drivers starting at the back (red): leading the championship matters most when the driver also starts near
  the front.

### 6.4 Local explanations: SHAP waterfall and LIME on the same two rows
Two single predictions from the test seasons, each explained twice:
- **Row A, a confident hit:** the correct podium the model was surest about.
- **Row B, the biggest miss:** the real podium the model gave the lowest probability.

Both are rows where every feature is present, so no imputed values blur the explanation.

In [ ]:
test_info = model_df.loc[X_test.index, ["raceId", "driverId", "year", "podium"]].copy()
test_info["race"] = test_info["raceId"].map(CLEAN["races"].set_index("raceId")["name"])
test_info["driver"] = test_info["driverId"].map(CLEAN["drivers"].set_index("driverId")["surname"])
test_info["grid"] = X_test["grid"]
test_info["predicted probability"] = PROBA[FINAL_MODEL_NAME]["test"]
test_info["predicted podium"] = decide(FINAL_MODEL_NAME, "test")
complete = X_test.notna().all(axis=1).to_numpy()              # rows with every feature present (clearer explanations)

pos = np.arange(len(X_test))
hit = test_info["podium"].to_numpy() == 1
i_A = pos[complete & hit][np.argmax(test_info["predicted probability"].to_numpy()[complete & hit])]   # surest correct podium
i_B = pos[complete & hit][np.argmin(test_info["predicted probability"].to_numpy()[complete & hit])]   # biggest miss
LOCAL_ROWS = {"A: confident hit": i_A, "B: biggest miss": i_B}
test_info.iloc[[i_A, i_B]].assign(row=list(LOCAL_ROWS)).set_index("row")[
    ["year", "race", "driver", "grid", "predicted probability", "predicted podium", "podium"]].round(3)

- **Row A:** Verstappen at the 2023 Abu Dhabi Grand Prix, from pole. Predicted probability 0.81, and the podium
  happened.
- **Row B:** Gasly at the 2024 São Paulo Grand Prix, from 13th on the grid. Predicted probability 0.003, yet a podium
  (3rd). That race was run in heavy rain, with several crashes and a red flag, which no pre-race data can foresee.

In [ ]:
for label, i in LOCAL_ROWS.items():
    r = test_info.iloc[i]
    shap.plots.waterfall(sv[i], max_display=10, show=False)
    plt.title(f"SHAP, row {label}: {r['driver']}, {r['year']} {r['race']} "
              f"(grid {r['grid']:g}, predicted {r['predicted probability']:.3f}, podium = {r['podium']})", fontsize=10)
    plt.gcf().set_size_inches(11, 5); plt.tight_layout()
    savefig(f"p6_shap_waterfall_{label[0]}"); plt.show()

**How to read a waterfall:** start at the bottom, at E[f(X)] = 0.154, the average prediction over the background
drivers. Each bar adds (red) or subtracts (blue) one feature's SHAP value, and the top, f(x), is the model's prediction
for this row. The grey numbers are the row's feature values.

- **Row A:** pole position (+0.18), qualifying first (+0.17), a podium in each of the last five races (+0.16), leading
  the championship (+0.08) and no gap to pole (+0.04) lift the probability from 0.15 to 0.81.
- **Row B:** qualifying 15th (−0.04), grid 13 (−0.03), 7.2% slower than pole (−0.03), no podium in the last five
  races (−0.02) and only 2.5% of the leader's points (−0.01) pull it down to 0.003. Every feature agrees that this was
  not a podium car before the start; the race itself was the surprise.

**LIME, the second local method:** it makes 5,000 slightly changed copies of the row, asks the model about each, and
fits a small weighted linear model to those answers. That model's weights explain the prediction *near this row only*.
LIME shows features as ranges (for example "grid <= 6"), because it groups each feature into quartiles of the training
data. It needs complete rows, so it uses the training data with the same medians the pipeline uses.

In [ ]:
try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:                                          # on Kaggle: Settings → Internet on
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lime"])
    from lime.lime_tabular import LimeTabularExplainer

train_medians = final_pipe[0][0].statistics_                 # the training medians the pipeline itself uses
lime_train = X_train.astype(float).fillna(pd.Series(train_medians, index=FEATURES)).to_numpy()
lime_explainer = LimeTabularExplainer(lime_train, feature_names=FEATURES, class_names=["no podium", "podium"],
                                      categorical_features=[FEATURES.index("no_grid_slot"), FEATURES.index("home_race")],
                                      mode="classification", discretize_continuous=True, random_state=SEED)
predict_both = lambda A: final_pipe.predict_proba(pd.DataFrame(np.asarray(A, dtype=float), columns=FEATURES))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
lime_top = {}
for ax, (label, i) in zip(axes, LOCAL_ROWS.items()):
    exp = lime_explainer.explain_instance(X_test_f.iloc[i].to_numpy(), predict_both, num_features=8, num_samples=5000)
    weights = exp.as_list()
    lime_top[label] = weights
    names, vals = [w[0] for w in weights][::-1], [w[1] for w in weights][::-1]
    ax.barh(names, vals, color=["#27ae60" if v > 0 else "#c0392b" for v in vals])
    ax.axvline(0, color="black", lw=0.8)
    r = test_info.iloc[i]
    ax.set_title(f"LIME, row {label}: {r['driver']}, {r['year']} {r['race']}\n"
                 f"local model R² = {exp.score:.2f}", fontsize=10)
    ax.set_xlabel("LIME weight (green = towards podium, red = away)")
plt.tight_layout(); savefig("p6_lime"); plt.show()

compare_rows = []
for label, i in LOCAL_ROWS.items():
    shap_top = pd.Series(sv.values[i], index=FEATURES).abs().sort_values(ascending=False).head(5).index.tolist()
    lime_feats = [next(f for f in sorted(FEATURES, key=len, reverse=True) if f in rule) for rule, _ in lime_top[label][:5]]
    compare_rows.append({"row": label, "SHAP top 5": ", ".join(shap_top), "LIME top 5": ", ".join(lime_feats),
                         "shared in top 5": len(set(shap_top) & set(lime_feats))})
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame(compare_rows).set_index("row"))

**How to read the charts:** each bar is one weight in LIME's local linear model: green pushes towards a podium, red away
from it. R² says how well that simple model imitates the network around this row (1 = perfectly).

**What they show:**
- **Row A:** grid ≤ 6, qualifying ≤ 6, a recent podium rate above 0.2 and high points shares push up, the same story
  as SHAP.
- **Row B:** no recent podiums, qualifying 11–16, a gap above 3.18% and grid 11–16 push down, again matching SHAP.
- **Agreement:** SHAP and LIME share 4 of their top 5 features for both rows, so the explanation does not depend on the
  method.
- **Where they differ:** LIME lists `no_grid_slot = 0` high. In a few of LIME's random copies the flag flips to 1 (it
  is 1 for 0.4% of training rows), and the network punishes a car without a grid slot hard, so LIME credits the real
  value 0 with a positive weight. SHAP compares against realistic background drivers, where the flag is almost always
  0, so it gives the flag almost nothing.
- **Limits:** LIME is an approximation. Its local model imitates the network well for row A (R² = 0.43) but poorly for
  row B (R² = 0.16), because near a probability of 0.003 the network's output is almost flat. SHAP's waterfall adds up
  exactly to the prediction, so for precise numbers we trust SHAP and use LIME as a cross-check.

### 6.5 What these explanations do not show
> **These explanations describe what the model relies on, not what causes podiums.** A large SHAP value for `grid`
> means the model leans on starting position to make its prediction. It does not prove that starting position causes
> podium finishes: a fast car produces both a good grid slot and a good result. The same holds for every feature here,
> for example a high points share does not *cause* the next podium; it reflects a strong driver and car. Showing cause
> would need a controlled experiment, which observational race data cannot give us.

### 6.6 Phase 6 gate
The run stops here if any check fails: an ablation variant is missing or a score is outside [0, 1], permutation
importance does not cover every feature, the SHAP values do not have one value per test row and feature or do not add
up to the model's predictions, the LIME explanations are missing, or `CLEAN` changed.

In [ ]:
assert set(ablation_table.index.get_level_values("variant")) >= {"all features", "without grid + qualifying"} | {
    f"without {g}" for g in GROUPS}
assert ablation_table[["CV PR-AUC", "val PR-AUC", "test PR-AUC"]].apply(lambda s: s.between(0, 1)).all().all()
assert list(perm.index.sort_values()) == sorted(FEATURES) and perm["std"].ge(0).all()
assert sv.values.shape == (len(X_test), len(FEATURES))
assert np.allclose(sv.values.sum(axis=1) + sv.base_values, predict_podium_proba(X_test_f), atol=1e-6)   # SHAP adds up
assert set(lime_top) == set(LOCAL_ROWS) and all(len(w) == 8 for w in lime_top.values())
assert {t: int(pd.util.hash_pandas_object(df).sum()) for t, df in CLEAN.items()} == CLEAN_FINGERPRINT, "CLEAN changed"
print(f"Phase 6 gate passed: ablation of {len(GROUPS)} groups (+ grid & qualifying together) for "
      f"{' and '.join(ABLATION_MODELS)}, permutation importance and SHAP for all {len(X_test):,} test rows, "
      f"SHAP + LIME for 2 rows, CLEAN unchanged.")

**Outputs for Phase 7:** `final_pipe` (the final model as one pipeline), `predict_podium_proba()`, `THRESHOLDS` and
`DECISION_RULE` (the decision rule from 5.7), `ablation_table`, `perm`, `sv` (SHAP values of every test row),
`LOCAL_ROWS`, and the figures in `figures/`.

## 7. Inference and final checks

**What the brief asks:** an inference function, demonstrated on a complete record and on a partially missing one. We
also run the raw-file check promised at the top of the notebook (Section 0.2).

| Step | What happens |
|---|---|
| 7.1 | `predict_podium()`: one driver, one race, pre-race values in, probability and decision out |
| 7.2 | Two demos (complete and partially missing record) and a typo check |
| 7.3 | MD5 re-check: the raw CSVs are unchanged after the whole run |
| 7.4 | Phase 7 gate |

### 7.1 The prediction function
**What it does:** it takes one driver's pre-race information as a dictionary (feature name → value), runs it through
the final pipeline from Phase 5 (imputer, scaler and neural network, all fitted on 2003–2019), and returns the podium
probability, the yes/no decision and the list of fields it had to fill.

**Design choices:**
- **Missing fields are allowed.** A field that is left out, `None` or `NaN` is filled by the pipeline with the training
  median, and its missing-indicator column is switched on where one exists, exactly as during training.
- **Unknown field names raise an error.** A typo such as `quali_position` would otherwise be ignored silently, and the
  real value would be treated as missing.
- **The decision uses the threshold chosen on validation in 5.7** (rule A), never re-tuned.

In [ ]:
def predict_podium(record, model=final_pipe, threshold=THRESHOLDS[FINAL_MODEL_NAME]):
    """Predict whether one driver reaches the podium in one race, from pre-race information only.

    record: dict {feature name: value}. A feature that is left out, None or NaN counts as missing: the pipeline fills
    it with the training median and switches on its missing-indicator column, exactly as during training.
    Returns the podium probability, the yes/no decision at the validation threshold, and the fields that were filled.
    """
    unknown = sorted(set(record) - set(FEATURES))
    if unknown:
        raise ValueError(f"unknown field(s) {unknown}, probably a typo; valid names are in FEATURES")
    row = pd.DataFrame([{f: record.get(f, np.nan) for f in FEATURES}], dtype=float)
    imputed = [f for f in FEATURES if pd.isna(row.at[0, f])]
    proba = float(model.predict_proba(row)[0, 1])
    return {"podium_probability": round(proba, 3), "predicted_podium": bool(proba >= threshold),
            "threshold": round(threshold, 3), "imputed_fields": imputed}


print(f"predict_podium() uses the final model ({FINAL_MODEL_NAME}, fitted on 2003–2019) and the decision threshold "
      f"{THRESHOLDS[FINAL_MODEL_NAME]:.3f} chosen on validation in 5.7.")

The function is defined. It always uses the final model and the validation threshold printed above (0.260), so every
prediction below is made exactly as the model was evaluated in Phase 5.

### 7.2 Demo 1: a complete record
A real test row: the driver on pole for the last race of 2024. The record holds all 18 feature values, so nothing
needs to be filled.

In [ ]:
surname = CLEAN["drivers"].set_index("driverId")["surname"]
race_name = CLEAN["races"].set_index("raceId")["name"]
last_round = model_df.loc[model_df["year"] == 2024, "round"].max()
demo = model_df[(model_df["year"] == 2024) & (model_df["round"] == last_round) & (model_df["grid"] == 1)].iloc[0]
complete_record = {f: float(demo[f]) for f in FEATURES}

result_complete = predict_podium(complete_record)
assert np.isclose(result_complete["podium_probability"],
                  final_pipe.predict_proba(X_test.loc[[demo.name]])[0, 1], atol=5e-4)   # same as the pipeline itself
print(f"Record: {surname[demo['driverId']]}, {demo['year']} {race_name[demo['raceId']]} (round {demo['round']}), "
      f"grid {demo['grid']:g}. Real result: podium = {demo['podium']}")
print("Prediction:", result_complete)
pd.Series(complete_record, name="value in the record").round(3).to_frame()

**Result:** Norris started the 2024 Abu Dhabi Grand Prix from pole. The function gives a podium probability of 0.708,
well above the threshold of 0.26, so it predicts a podium, and Norris did finish on the podium. `imputed_fields` is
empty because every value was given. The cell also checks that the function returns the same probability as the
pipeline does for that test row, so the function adds no error of its own.

**How to read the table:** the record exactly as the function received it: pole position, 0% gap to pole, second in
the drivers' championship with 81% of the leader's points, and McLaren leading the constructors' championship.

### 7.2 Demo 2: a partially missing record
The same record with five fields removed: the three qualifying fields and two driver-form fields, as if the qualifying
times had not been published yet and the driver's history were unknown.

In [ ]:
REMOVED = ["quali_pos", "gap_to_pole_pct", "grid_minus_quali", "drv_avg_finish_last5", "drv_podium_rate_last5"]
partial_record = {f: v for f, v in complete_record.items() if f not in REMOVED}
result_partial = predict_podium(partial_record)
print("Prediction with five fields missing:", result_partial)

medians = pd.Series(final_pipe[0][0].statistics_, index=FEATURES)
has_indicator = set(final_pipe[0][0].feature_names_in_[final_pipe[0][0].indicator_.features_])
fill_table = pd.DataFrame({"real value": [complete_record[f] for f in REMOVED],
                           "value used (training median)": medians[REMOVED].to_numpy(),
                           "missing-indicator switched on": ["yes" if f in has_indicator else "no (never missing in training)"
                                                             for f in REMOVED]}, index=REMOVED)
print(f"\nPodium probability: complete record {result_complete['podium_probability']:.3f} → "
      f"partial record {result_partial['podium_probability']:.3f}")
fill_table.round(3)

**Result:** the function still returns a prediction, and `imputed_fields` lists exactly the five missing fields. The
probability moves from 0.708 to 0.508, still above the threshold, so the decision stays "podium".

**Why the probability moves:** the pipeline filled the gaps with values for an *average* training driver: qualifying
11th instead of 1st, 1.85% slower than pole instead of 0%, and median form. The model now sees a driver who starts from
pole but qualified 11th, so it is less sure. Grid position is still there, which is why the decision does not flip.

**How to read the table:** the real value, the value the pipeline used instead, and whether a missing-indicator
column told the model that the value was filled. The two form features have an indicator, because form was sometimes
missing in training (debuts). The three qualifying fields have none: they were never missing in training, so the model
cannot tell a filled median from a real value. That is a limitation of the model, and it is why the function reports
`imputed_fields`: the person using it can see which inputs were guessed.

### 7.2 Demo 3: a typo in a field name

In [ ]:
try:
    predict_podium({**complete_record, "quali_position": 1})            # typo: the real name is quali_pos
    typo_rejected = False
except ValueError as err:
    typo_rejected = True
    print("ValueError:", err)

**Result:** a record with `quali_position` instead of `quali_pos` is rejected with a clear message, instead of being
accepted with the qualifying position silently treated as missing.

### 7.3 Raw files unchanged
Section 0.2 stored an MD5 fingerprint of every raw CSV before anything ran. We compute them again now, after the whole
notebook.

In [ ]:
RAW_HASHES_END = {t: file_md5(os.path.join(DATA_DIR, f"{t}.csv")) for t in TABLES}
md5_check = pd.DataFrame({"md5 at the start (0.2)": pd.Series(RAW_HASHES), "md5 now": pd.Series(RAW_HASHES_END)})
md5_check["unchanged"] = md5_check["md5 at the start (0.2)"] == md5_check["md5 now"]
assert md5_check["unchanged"].all(), "A raw file changed during the run"
print(f"All {len(TABLES)} raw CSVs are byte-for-byte unchanged after the full run.")
md5_check

**Result:** all 14 fingerprints match. The raw files are byte-for-byte unchanged, so every cleaning and modelling step
worked on copies, as the brief requires. (`CLEAN` itself has been checked unchanged at the end of every phase since
Phase 3.)

### 7.4 Phase 7 gate
The run stops here if any check fails: a probability outside [0, 1], fields reported as filled that were given (or the
reverse), an unknown field name that is not rejected, a changed raw file, or a changed `CLEAN`.

In [ ]:
assert 0 <= result_complete["podium_probability"] <= 1 and 0 <= result_partial["podium_probability"] <= 1
assert result_complete["imputed_fields"] == []
assert sorted(result_partial["imputed_fields"]) == sorted(REMOVED)
assert typo_rejected, "a misspelled field name must raise an error"
assert md5_check["unchanged"].all()
assert {t: int(pd.util.hash_pandas_object(df).sum()) for t, df in CLEAN.items()} == CLEAN_FINGERPRINT, "CLEAN changed"
print("Phase 7 gate passed: the inference function works on a complete and a partially missing record, rejects "
      "unknown fields, and the raw files and CLEAN are unchanged.")

## Where each checklist item is answered

| Checklist item | Sections |
|---|---|
| Exploratory data analysis, before and after cleaning | 1.6, 2.8 |
| Sentinels and semantic typing | 1.2–1.3, 2.2–2.3 |
| One row per (raceId, driverId), duplicates documented | 1.4, 2.6 |
| Primary and foreign keys; raw files unchanged | 1.4–1.5, 2.7; 0.2 and 7.3 |
| The three data-engineering questions, plotted and answered | 3.1, 3.2, 3.3 |
| Reasons for the chosen features, with evidence and plots | 4.2, 4.5 |
| Leakage-safe form features (past races only) | 4.2, 4.4 |
| Effect of preprocessing on the data; preprocessing-order experiments | 4.7; 4.8 (E1, E2, plus E3) |
| Train / validation / test split | 4.6 |
| Each model: how it works and its limitations | 5.4, 5.5, 5.6 |
| Performance during training and validation | 5.3, 5.6 (training curve), 5.8 |
| Performance on unseen test data, with plots | 5.8 |
| Feature-group ablation | 6.1 |
| Global explanations: permutation importance and SHAP | 6.2, 6.3 |
| Local explanations: LIME (and SHAP) | 6.4 |
| Explanations are not causal | 6.5 |
| Inference on a complete and a partially missing record | 7.1, 7.2 |